In [2]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [3]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt


In [4]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'TQQQ'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = yf.download(TICKER, start=START_DATE, interval='1d')

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data


[*********************100%***********************]  1 of 1 completed

Successfully downloaded 2167 records for TQQQ from 2018-01-01
Data range: 2018-01-02 to 2026-08-17

First 5 rows:
Price          Close      High       Low      Open    Volume
Ticker          TQQQ      TQQQ      TQQQ      TQQQ      TQQQ
Date                                                        
2018-01-02  5.778714  5.790602  5.542953  5.582973  91735200
2018-01-03  5.947908  5.969702  5.801302  5.802094  85224000
2018-01-04  5.982775  6.039041  5.961775  6.017644  70024800
2018-01-05  6.163460  6.176536  6.035080  6.066383  82380000
2018-01-08  6.232009  6.244689  6.156725  6.162668  72252000


Price,Close,High,Low,Open,Volume
Ticker,TQQQ,TQQQ,TQQQ,TQQQ,TQQQ
Date,,,,,
2018-01-02,5.778714,5.790602,5.542953,5.582973,91735200
2018-01-03,5.947908,5.969702,5.801302,5.802094,85224000
2018-01-04,5.982775,6.039041,5.961775,6.017644,70024800
2018-01-05,6.163460,6.176536,6.035080,6.066383,82380000
2018-01-08,6.232009,6.244689,6.156725,6.162668,72252000
...,...,...,...,...,...
2026-08-11,73.059998,74.540001,72.239998,74.540001,43674100
2026-08-12,74.599998,75.690002,74.379997,75.620003,46867300


In [5]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data is available from the previous cell
if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

# Extract OHLCV data (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

print(f"Calculating technical indicators for {TICKER}...")

# Simple Moving Averages
sma_20 = talib.SMA(close, timeperiod=20)
sma_50 = talib.SMA(close, timeperiod=50)

# Exponential Moving Averages
ema_12 = talib.EMA(close, timeperiod=12)
ema_26 = talib.EMA(close, timeperiod=26)

# MACD
macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

# RSI
rsi = talib.RSI(close, timeperiod=14)

# Stochastic RSI
stochrsi_k, stochrsi_d = talib.STOCHRSI(close, timeperiod=14, fastk_period=3, fastd_period=3, fastd_matype=0)

# VWAP (manual calculation)
typical_price = (high + low + close) / 3
price_volume = typical_price * volume
cumulative_price_volume = np.cumsum(price_volume)
cumulative_volume = np.cumsum(volume)
vwap = cumulative_price_volume / cumulative_volume

# Schaff Trend Cycle (defaults: EMA 23/50, stochastic cycle 10)
STC_EMA_SHORT = 23
STC_EMA_LONG = 50
STC_CYCLE_PERIOD = 10

stc_ema_short = talib.EMA(close, timeperiod=STC_EMA_SHORT)
stc_ema_long = talib.EMA(close, timeperiod=STC_EMA_LONG)
stc_macd = stc_ema_short - stc_ema_long

stc_macd_highest = talib.MAX(stc_macd, timeperiod=STC_CYCLE_PERIOD)
stc_macd_lowest = talib.MIN(stc_macd, timeperiod=STC_CYCLE_PERIOD)
stc_macd_denom = stc_macd_highest - stc_macd_lowest
with np.errstate(divide="ignore", invalid="ignore"):
    stc_k_macd = np.where(np.abs(stc_macd_denom) > 0, 100.0 * (stc_macd - stc_macd_lowest) / stc_macd_denom, 0.0)
stc_d_macd = talib.EMA(stc_k_macd, timeperiod=STC_CYCLE_PERIOD)

stc_d_highest = talib.MAX(stc_d_macd, timeperiod=STC_CYCLE_PERIOD)
stc_d_lowest = talib.MIN(stc_d_macd, timeperiod=STC_CYCLE_PERIOD)
stc_d_denom = stc_d_highest - stc_d_lowest
with np.errstate(divide="ignore", invalid="ignore"):
    stc_k = np.where(np.abs(stc_d_denom) > 0, 100.0 * (stc_d_macd - stc_d_lowest) / stc_d_denom, 0.0)
stc_d = talib.EMA(stc_k, timeperiod=STC_CYCLE_PERIOD)

# AROON
AROON_PERIOD = 14
if len(high) != len(close) or len(low) != len(close):
    raise ValueError("High, Low, and Close arrays must have the same length for AROON")
if len(close) <= AROON_PERIOD:
    raise ValueError(f"AROON requires more than {AROON_PERIOD} rows of OHLC data")
aroon_down, aroon_up = talib.AROON(
    high.astype(float, copy=False),
    low.astype(float, copy=False),
    timeperiod=AROON_PERIOD,
)

# KAMA (Kaufman Adaptive Moving Average — custom implementation for full parameter control)
KAMA_PERIOD = 10
KAMA_FAST_EMA_CONSTANT = 2
KAMA_SLOW_EMA_CONSTANT = 30


def validate_kama_params(period, fast_ema_constant, slow_ema_constant):
    try:
        period = int(period)
        fast_ema_constant = int(fast_ema_constant)
        slow_ema_constant = int(slow_ema_constant)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KAMA parameters must be integers; "
            f"got period={period!r}, fast={fast_ema_constant!r}, slow={slow_ema_constant!r}"
        ) from exc
    if period <= 0 or fast_ema_constant <= 0 or slow_ema_constant <= 0:
        raise ValueError("KAMA parameters must be positive integers.")
    if fast_ema_constant >= slow_ema_constant:
        raise ValueError("KAMA fastest EMA constant must be less than slowest EMA constant.")
    return period, fast_ema_constant, slow_ema_constant


def _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant):
    if close.dropna().shape[0] <= period:
        raise ValueError(f"KAMA requires more than {period} rows of close data")

    change = (close - close.shift(period)).abs()
    volatility = close.diff().abs().rolling(window=period).sum()
    er = change / (volatility + 1e-9)
    sc = (
        (er * (2 / (fast_ema_constant + 1) - 2 / (slow_ema_constant + 1)))
        + 2 / (slow_ema_constant + 1)
    ) ** 2
    sc = sc.clip(0, 1).fillna(0)

    kama_values = np.full(len(close), np.nan)
    seed = close.iloc[period]
    if pd.isna(seed):
        return pd.Series(kama_values, index=close.index)

    kama_values[period] = seed
    sc_values = sc.to_numpy(dtype=float)
    close_values = close.to_numpy(dtype=float)
    for i in range(period + 1, len(close)):
        if np.isnan(close_values[i]):
            kama_values[i] = np.nan
            continue
        prev = kama_values[i - 1]
        if np.isnan(prev):
            kama_values[i] = np.nan
            continue
        kama_values[i] = prev + sc_values[i] * (close_values[i] - prev)

    kama_values[:period] = np.nan
    return pd.Series(kama_values, index=close.index)


def compute_kama_series(close, period, fast_ema_constant, slow_ema_constant, index=None):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    if not isinstance(close, pd.Series):
        if index is None:
            raise ValueError("index is required when close is not a Series")
        close = pd.Series(close, index=index)
    close = close.astype(float)
    return _compute_kama_core(close, period, fast_ema_constant, slow_ema_constant)


kama = compute_kama_series(
    close,
    KAMA_PERIOD,
    KAMA_FAST_EMA_CONSTANT,
    KAMA_SLOW_EMA_CONSTANT,
    index=stock_data.index,
).values

# SUPERTREND (ATR from TA-Lib + SuperTrend band/direction logic)
SUPERTREND_ATR_LENGTH = 10
SUPERTREND_FACTOR = 3


def validate_supertrend_params(atr_length, factor):
    try:
        atr_length_num = float(atr_length)
        factor_num = float(factor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "SUPERTREND parameters must be numeric; "
            f"got atr_length={atr_length!r}, factor={factor!r}"
        ) from exc
    if not np.isfinite(atr_length_num) or not np.isfinite(factor_num):
        raise ValueError("SUPERTREND ATR length and factor must be finite numbers.")
    if atr_length_num != int(atr_length_num):
        raise ValueError(
            f"SUPERTREND ATR length must be an integer; got atr_length={atr_length!r}"
        )
    atr_length = int(atr_length_num)
    # Reject silent float truncation (e.g. 2.7 -> 2) while still allowing 3.0
    if factor_num != int(factor_num):
        raise ValueError(
            "SUPERTREND factor must be a whole number for this grid "
            f"(sensitivity uses int grids); got factor={factor!r}"
        )
    factor = int(factor_num)
    if atr_length <= 0 or factor <= 0:
        raise ValueError("SUPERTREND ATR length and factor must be positive integers.")
    return atr_length, factor


def _compute_supertrend_core(high, low, close, atr_length, factor):
    if len(high) != len(close) or len(low) != len(close):
        raise ValueError("High, Low, and Close arrays must have the same length for SUPERTREND")
    if len(close) == 0:
        raise ValueError("SUPERTREND requires a non-empty OHLC series")
    if len(close) <= atr_length:
        raise ValueError(f"SUPERTREND requires more than {atr_length} rows of OHLC data")

    high_arr = np.asarray(high, dtype=float)
    low_arr = np.asarray(low, dtype=float)
    close_arr = np.asarray(close, dtype=float)

    finite_mask = np.isfinite(high_arr) & np.isfinite(low_arr) & np.isfinite(close_arr)
    if not finite_mask.any():
        raise ValueError("SUPERTREND requires at least one finite OHLC row")
    if np.any(high_arr[finite_mask] < low_arr[finite_mask]):
        raise ValueError("SUPERTREND requires High >= Low on all finite rows")

    atr = talib.ATR(high_arr, low_arr, close_arr, timeperiod=atr_length)
    hl2 = (high_arr + low_arr) / 2.0
    basic_upper = hl2 + factor * atr
    basic_lower = hl2 - factor * atr

    final_upper = np.full(len(close_arr), np.nan)
    final_lower = np.full(len(close_arr), np.nan)
    supertrend = np.full(len(close_arr), np.nan)
    direction = np.full(len(close_arr), np.nan)

    for i in range(len(close_arr)):
        # Gap / bad bar: carry prior state forward so trend memory is not reset
        if (
            np.isnan(atr[i])
            or not np.isfinite(close_arr[i])
            or not np.isfinite(high_arr[i])
            or not np.isfinite(low_arr[i])
        ):
            if i > 0:
                final_upper[i] = final_upper[i - 1]
                final_lower[i] = final_lower[i - 1]
                direction[i] = direction[i - 1]
                supertrend[i] = supertrend[i - 1]
            continue

        if i == 0 or np.isnan(final_lower[i - 1]):
            final_lower[i] = basic_lower[i]
        elif basic_lower[i] > final_lower[i - 1] or close_arr[i - 1] < final_lower[i - 1]:
            final_lower[i] = basic_lower[i]
        else:
            final_lower[i] = final_lower[i - 1]

        if i == 0 or np.isnan(final_upper[i - 1]):
            final_upper[i] = basic_upper[i]
        elif basic_upper[i] < final_upper[i - 1] or close_arr[i - 1] > final_upper[i - 1]:
            final_upper[i] = basic_upper[i]
        else:
            final_upper[i] = final_upper[i - 1]

        # Seed first direction from bands, then prior-close momentum (avoid hl2 tie -> always +1)
        if i == 0 or np.isnan(direction[i - 1]):
            if close_arr[i] > final_upper[i]:
                direction[i] = 1.0
            elif close_arr[i] < final_lower[i]:
                direction[i] = -1.0
            elif i > 0 and np.isfinite(close_arr[i - 1]):
                direction[i] = 1.0 if close_arr[i] >= close_arr[i - 1] else -1.0
            else:
                direction[i] = 1.0 if close_arr[i] > hl2[i] else -1.0
        elif direction[i - 1] == 1.0:
            direction[i] = -1.0 if close_arr[i] < final_lower[i] else 1.0
        else:
            direction[i] = 1.0 if close_arr[i] > final_upper[i] else -1.0

        supertrend[i] = final_lower[i] if direction[i] == 1.0 else final_upper[i]

    return supertrend, direction


def compute_supertrend_series(high, low, close, atr_length, factor, index=None):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("SUPERTREND index length must match close length")
    supertrend, direction = _compute_supertrend_core(high, low, close, atr_length, factor)
    return (
        pd.Series(supertrend, index=index),
        pd.Series(direction, index=index),
    )


supertrend, supertrend_direction = compute_supertrend_series(
    high,
    low,
    close,
    SUPERTREND_ATR_LENGTH,
    SUPERTREND_FACTOR,
    index=stock_data.index,
)
supertrend = supertrend.values
supertrend_direction = supertrend_direction.values

# KALMAN (1D Kalman filter — custom implementation; Q=process noise, R=measurement noise)
KALMAN_PROCESS_NOISE = 1
KALMAN_MEASUREMENT_NOISE = 10


def validate_kalman_params(process_noise, measurement_noise):
    try:
        process_noise_num = float(process_noise)
        measurement_noise_num = float(measurement_noise)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "KALMAN parameters must be numeric; "
            f"got process_noise={process_noise!r}, measurement_noise={measurement_noise!r}"
        ) from exc
    if not np.isfinite(process_noise_num) or not np.isfinite(measurement_noise_num):
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be finite numbers.")
    if process_noise_num != int(process_noise_num):
        raise ValueError(
            "KALMAN process noise (Q) must be a whole number for this grid "
            f"(sensitivity uses int grids); got process_noise={process_noise!r}"
        )
    if measurement_noise_num != int(measurement_noise_num):
        raise ValueError(
            "KALMAN measurement noise (R) must be a whole number for this grid "
            f"(sensitivity uses int grids); got measurement_noise={measurement_noise!r}"
        )
    process_noise = int(process_noise_num)
    measurement_noise = int(measurement_noise_num)
    if process_noise <= 0 or measurement_noise <= 0:
        raise ValueError("KALMAN process noise (Q) and measurement noise (R) must be positive integers.")
    return process_noise, measurement_noise


def _compute_kalman_core(close, process_noise, measurement_noise):
    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("KALMAN requires a non-empty close series")
    if not np.isfinite(close_arr).any():
        raise ValueError("KALMAN requires at least one finite close price")

    kalman = np.full(close_arr.size, np.nan)
    q = float(process_noise)
    r = float(measurement_noise)

    first_idx = int(np.flatnonzero(np.isfinite(close_arr))[0])
    x = close_arr[first_idx]
    # Seed uncertainty from R so the first updates are not artificially overconfident
    p = float(measurement_noise)
    kalman[first_idx] = x

    for i in range(first_idx + 1, close_arr.size):
        z = close_arr[i]
        # Predict every bar (including gaps) so multi-bar holes inflate uncertainty
        p = p + q
        if not np.isfinite(z):
            # Carry estimate forward across gaps; do not update with bad measurements
            kalman[i] = x
            continue
        # Update: higher R trusts the filter prediction over noisy measurements
        k = p / (p + r)
        x = x + k * (z - x)
        p = (1.0 - k) * p
        kalman[i] = x

    return kalman


def compute_kalman_series(close, process_noise, measurement_noise, index=None):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("KALMAN index length must match close length")
    kalman = _compute_kalman_core(close, process_noise, measurement_noise)
    return pd.Series(kalman, index=index)


kalman = compute_kalman_series(
    close,
    KALMAN_PROCESS_NOISE,
    KALMAN_MEASUREMENT_NOISE,
    index=stock_data.index,
).values

# ALMA (Arnaud Legoux Moving Average — custom implementation; TA-Lib has no ALMA)
# Offset is stored as hundredths (85 = 0.85) so sensitivity int-grids stay coherent.
ALMA_LENGTH = 9
ALMA_OFFSET = 85
ALMA_SIGMA = 6


def validate_alma_params(length, offset, sigma):
    try:
        length_num = float(length)
        offset_num = float(offset)
        sigma_num = float(sigma)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ALMA parameters must be numeric; "
            f"got length={length!r}, offset={offset!r}, sigma={sigma!r}"
        ) from exc
    if not np.isfinite(length_num) or not np.isfinite(offset_num) or not np.isfinite(sigma_num):
        raise ValueError("ALMA length, offset, and sigma must be finite numbers.")
    if length_num != int(length_num):
        raise ValueError(
            "ALMA length must be a whole number for this grid "
            f"(sensitivity uses int grids); got length={length!r}"
        )
    if offset_num != int(offset_num):
        raise ValueError(
            "ALMA offset must be a whole number for this grid "
            f"(sensitivity uses int grids; offset is hundredths, e.g. 85 = 0.85); "
            f"got offset={offset!r}"
        )
    if sigma_num != int(sigma_num):
        raise ValueError(
            "ALMA sigma must be a whole number for this grid "
            f"(sensitivity uses int grids); got sigma={sigma!r}"
        )
    length = int(length_num)
    offset = int(offset_num)
    sigma = int(sigma_num)
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    # 0..100 hundredths maps to classic ALMA distribution offset [0.0, 1.0]
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    return length, offset, sigma


def _compute_alma_core(close, length, offset, sigma):
    # Defensive guards so direct callers cannot hit ZeroDivisionError / bad shapes.
    if length <= 1:
        raise ValueError("ALMA length must be an integer greater than 1.")
    if sigma <= 0:
        raise ValueError("ALMA sigma must be a positive integer.")
    if offset < 0 or offset > 100:
        raise ValueError("ALMA offset must be an integer in 0..100 (hundredths of the distribution offset).")

    close_arr = np.asarray(close, dtype=float)
    if close_arr.size == 0:
        raise ValueError("ALMA requires a non-empty close series")
    if close_arr.size < length:
        raise ValueError(f"ALMA requires at least {length} rows of close data")
    if not np.isfinite(close_arr).any():
        raise ValueError("ALMA requires at least one finite close price")

    offset_f = offset / 100.0
    m = int(np.floor(offset_f * (length - 1)))
    s = length / float(sigma)
    idxs = np.arange(length, dtype=float)
    weights = np.exp(-((idxs - m) ** 2) / (2.0 * s * s))
    norm = weights.sum()
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError("ALMA weights are degenerate for the selected parameters")

    # Vectorized weighted window (avoids Python per-bar loop on large grids).
    alma = np.full(close_arr.size, np.nan)
    windows = np.lib.stride_tricks.sliding_window_view(close_arr, window_shape=length)
    finite_mask = np.isfinite(windows).all(axis=1)
    if finite_mask.any():
        alma_vals = np.full(windows.shape[0], np.nan)
        alma_vals[finite_mask] = windows[finite_mask] @ weights / norm
        alma[length - 1:] = alma_vals

    return alma


def compute_alma_series(close, length, offset, sigma, index=None):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if index is None:
        if isinstance(close, pd.Series):
            index = close.index
        else:
            raise ValueError("index is required when close is not a Series")
    if len(index) != len(np.asarray(close)):
        raise ValueError("ALMA index length must match close length")
    alma = _compute_alma_core(close, length, offset, sigma)
    return pd.Series(alma, index=index)


alma = compute_alma_series(
    close,
    ALMA_LENGTH,
    ALMA_OFFSET,
    ALMA_SIGMA,
    index=stock_data.index,
).values

# Create indicators dataframe
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close,
    "SMA_20": sma_20,
    "SMA_50": sma_50,
    "EMA_12": ema_12,
    "EMA_26": ema_26,
    "MACD": macd,
    "MACD_Signal": macdsignal,
    "MACD_Hist": macdhist,
    "RSI": rsi,
    "StochRSI_K": stochrsi_k,
    "StochRSI_D": stochrsi_d,
    "VWAP": vwap,
    "STC_K": stc_k,
    "STC_D": stc_d,
    "AROON_Down": aroon_down,
    "AROON_Up": aroon_up,
    "KAMA": kama,
    "SUPERTREND": supertrend,
    "SUPERTREND_Direction": supertrend_direction,
    "KALMAN": kalman,
    "ALMA": alma,
})

print("All technical indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)


Calculating technical indicators for TQQQ...
All technical indicators calculated!
Data shape: (2167, 22)


,Date,Close,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_Signal,MACD_Hist,RSI,...,VWAP,STC_K,STC_D,AROON_Down,AROON_Up,KAMA,SUPERTREND,SUPERTREND_Direction,KALMAN,ALMA
2162,2026-08-11,73.059998,68.5960,73.866122,70.731378,70.720027,0.011350,-1.244960,1.256311,53.487632,...,19.075846,100.0,75.465535,35.714286,71.428571,69.372624,62.194070,1.0,72.138205,73.378488
2163,2026-08-12,74.599998,68.6040,73.641275,71.326550,71.007432,0.319118,-0.932145,1.251262,55.566428,...,19.082335,100.0,79.926347,28.571429,64.285714,70.509471,63.811093,1.0,72.803274,73.638167
2164,2026-08-13,77.150002,68.9245,73.443882,72.222466,71.462438,0.760028,-0.593710,1.353738,58.846298,...,19.089436,100.0,83.576102,21.428571,100.000000,71.791211,65.187482,1.0,73.977569,74.158672
2165,2026-08-14,76.790001,69.3875,73.252459,72.925163,71.857072,1.068091,-0.261350,1.329441,58.193220,...,19.094683,100.0,86.562265,14.285714,100.000000,72.626282,66.374738,1.0,74.737365,74.957050
2166,2026-08-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,7.142857,92.857143,NaN,66.374738,1.0,74.737365,NaN


In [6]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | val={val_close.index[0].date()} → {val_close.index[-1].date()}")

# Auto-detect market calendar from actual bars/year in the data
_calendar_days = max((close.index[-1] - close.index[0]).days, 1)
_bars_per_year = len(close) / (_calendar_days / 365.25)
IS_24_7_MARKET = _bars_per_year > 330
TRADING_DAYS_PER_YEAR = 365 if IS_24_7_MARKET else 252
YEAR_FREQ = f"{TRADING_DAYS_PER_YEAR}D"
FREQ = "1D"


def sample_years(price_series):
    return max(len(price_series) / TRADING_DAYS_PER_YEAR, 1e-9)


market_label = 'crypto (365-day)' if IS_24_7_MARKET else 'stock (252-day)'
print(f"Bars/year: {_bars_per_year:.0f} → FREQ='{FREQ}', YEAR_FREQ='{YEAR_FREQ}' ({market_label})")

Data ready: train=2018-01-02 → 2023-03-02 | val=2023-03-03 → 2026-08-17
Bars/year: 251 → FREQ='1D', YEAR_FREQ='252D' (stock (252-day))


MULTI-INDICATOR GRID SEARCH - TRAINING SET
------------------------------------------

This section performs grid search optimization for multiple technical indicators using only the **training data**.

The existing **Triple EMA Crossover Strategy** remains active, and this notebook adds **MACD Crossover Strategy**, **AROON Crossover Strategy**, **STC Trend Strategy**, **KAMA Crossover Strategy**, **SUPERTREND Strategy**, **KALMAN Strategy**, **RSI Strategy**, **ADX Strategy**, **DONCHIAN Strategy**, **TRIX Strategy**, **VORTEX Strategy**, and **ALMA Crossover Strategy** as separate grids. Each indicator owns its own parameter grid and signal builder, so Triple EMA, MACD, AROON, STC, KAMA, SUPERTREND, KALMAN, RSI, ADX, DONCHIAN, TRIX, VORTEX, and ALMA parameters are never combined into one search grid.

**Triple EMA Strategy Logic**: Buy when any EMA crosses above another (EMA1 > EMA2, EMA1 > EMA3, or EMA2 > EMA3). Sell when any EMA crosses below another.

**MACD Strategy Logic**: Buy when the MACD line crosses above the MACD signal line. Sell when the MACD line crosses below the MACD signal line.

**AROON Strategy Logic**: Aroon via TA-Lib (`talib.AROON`, timeperiod). Buy when Aroon Up crosses above Aroon Down and Aroon Up is strictly above `AROON_ENTRY_LEVEL`. Sell when Aroon Down crosses above Aroon Up. Grid search optimizes the AROON timeperiod independently.

**STC Strategy Logic**: Schaff Trend Cycle uses a custom MACD (fast EMA minus slow EMA) with a double-smoothed stochastic over the cycle period. Buy when the STC line crosses above 50. Sell when the STC line crosses below 50. Grid search optimizes the fast EMA length, slow EMA length, and cycle period.

**KAMA Strategy Logic**: Kaufman Adaptive Moving Average via a custom implementation (period, fastest EMA constant, slowest EMA constant). Buy when price crosses from below to above the KAMA line (shift into a new uptrend). Sell when price crosses from above to below the KAMA line (shift into a new downtrend). Grid search optimizes all three parameters independently.

**SUPERTREND Strategy Logic**: SuperTrend built from TA-Lib ATR (ATR length and factor/multiplier). Buy when price closes above a red SuperTrend line (line flips below price and turns green). Sell when price closes below a green SuperTrend line (line flips above price and turns red). Grid search optimizes ATR length and factor independently.

**KALMAN Strategy Logic**: 1D Kalman filter (process noise Q, measurement noise R). Higher Q makes the filter more responsive to price changes; higher R trusts the filter prediction over noisy measurements. Buy when price crosses from below to above the Kalman filter line. Sell when price crosses from above to below the Kalman filter line. Grid search optimizes Q and R independently.

**RSI Strategy Logic**: Relative Strength Index via TA-Lib (`talib.RSI`, timeperiod). Buy when the RSI crosses above 50, confirming rising bullish momentum. Sell when the RSI crosses below 50, confirming increasing bearish momentum. Grid search optimizes the RSI timeperiod independently.

**ADX Strategy Logic**: Directional Movement via TA-Lib (`talib.PLUS_DI`, `talib.MINUS_DI` for DI length) with ADX as Wilder-smoothed DX over the ADX smoothing timeperiod (so both parameters stay coherent). Buy when +DI crosses above -DI while ADX is above `ADX_TREND_LEVEL` (25, confirming a strong uptrend). Sell when -DI crosses above +DI while ADX is above `ADX_TREND_LEVEL` (confirming a strong downtrend). Grid search optimizes DI length and ADX smoothing independently.

**DONCHIAN Strategy Logic**: Donchian Channel built from TA-Lib (`talib.MAX` / `talib.MIN` over timeperiod for upper/lower bands; middle = average of the two — TA-Lib has no dedicated Donchian function). Buy when close crosses above the prior upper band (breakout to ride an upward trend). Sell when close crosses below the prior middle band (trailing stop to lock in profits). Grid search optimizes the Donchian timeperiod independently.

**TRIX Strategy Logic**: Triple Exponential Average via TA-Lib (`talib.TRIX`, timeperiod). Buy when the TRIX line crosses above zero, indicating bullish momentum and a potential buy signal. Sell when the TRIX line crosses below zero, suggesting bearish momentum and a potential sell signal. Grid search optimizes the TRIX timeperiod independently.

**VORTEX Strategy Logic**: Vortex Indicator built from TA-Lib helpers (`talib.TRANGE` / `talib.SUM` over timeperiod for VI+ and VI- — TA-Lib has no dedicated Vortex function). Buy when the VI+ line crosses upward from below the VI- line. Sell when the VI- line crosses upward from below the VI+ line. Grid search optimizes the VORTEX timeperiod independently.

**ALMA Strategy Logic**: Arnaud Legoux Moving Average via a custom implementation (TA-Lib has no ALMA; fast length, slow length, offset, sigma). Offset is stored as hundredths of the distribution offset in 0..100 (e.g. 85 = 0.85) so sensitivity int-grids stay coherent. Buy when the fast ALMA crosses above the slow ALMA, indicating an emerging bullish trend. Sell when the fast ALMA crosses below the slow ALMA, indicating a shifting bearish trend. Grid search optimizes fast length, slow length, offset, and sigma independently.

---


In [7]:
# Define Indicator Grids

# Existing Triple EMA grid. MACD, AROON, STC, KAMA, SUPERTREND, KALMAN, RSI, ADX, DONCHIAN, TRIX, VORTEX, and ALMA are added below as separate grids; parameters are not mixed.
ema1_periods = list(range(4, 10, 1))    # Fast EMA (shortest-term)
ema2_periods = list(range(90, 200, 1))  # Medium EMA
ema3_periods = list(range(120, 250, 1)) # Slow EMA

# MACD periods for crossover strategy
macd_fast_periods = list(range(10, 50, 1))
macd_slow_periods = list(range(60, 100, 1))
macd_signal_periods = list(range(40, 120, 1))

# AROON timeperiods for Up/Down crossover strategy
aroon_timeperiods = list(range(4, 80, 1))
AROON_ENTRY_LEVEL = 49

# STC EMA and cycle periods for Schaff Trend Cycle strategy
stc_ema_short_periods = list(range(10, 50, 1))
stc_ema_long_periods = list(range(60, 100, 1))
stc_cycle_periods = list(range(20, 100, 1))
STC_BUY_LEVEL = 50
STC_SELL_LEVEL = 50

# KAMA periods and EMA constants for price crossover strategy
kama_periods = list(range(5, 30, 5))
kama_fast_ema_constants = list(range(2, 40, 1))
kama_slow_ema_constants = list(range(60, 100, 1))

# SUPERTREND ATR length and factor/multiplier for trend-flip strategy
supertrend_atr_lengths = list(range(4, 112, 1))
supertrend_factors = list(range(1, 60, 1))

# KALMAN process noise (Q) and measurement noise (R) for price vs filter strategy
kalman_process_noise_values = list(range(1, 5, 1))
kalman_measurement_noise_values = list(range(100, 160, 1))

# RSI timeperiods for midline crossover strategy
rsi_timeperiods = list(range(40, 80, 1))
RSI_CROSS_LEVEL = 50

# ADX DI length and smoothing timeperiods for +DI/-DI crossover with trend-strength filter
adx_di_lengths = list(range(4, 60, 1))
adx_smoothing_timeperiods = list(range(20, 80, 1))
ADX_TREND_LEVEL = 20

# DONCHIAN timeperiods for upper-band breakout / middle-band trailing-stop strategy
donchian_timeperiods = list(range(10, 60, 1))

# TRIX timeperiods for zero-line crossover strategy
trix_timeperiods = list(range(4, 80, 1))
TRIX_CROSS_LEVEL = 0

# VORTEX timeperiods for VI+/VI- crossover strategy
vortex_timeperiods = list(range(40, 140, 1))

# ALMA fast/slow lengths / offset (hundredths) / sigma for ALMA crossover strategy
alma_fast_lengths = list(range(2, 40, 1))
alma_slow_lengths = list(range(60, 100, 1))
alma_offsets = list(range(10, 20, 2))
alma_sigmas = list(range(1, 10, 2))


def validate_triple_ema_params(ema1_period, ema2_period, ema3_period):
    ema1_period = int(ema1_period)
    ema2_period = int(ema2_period)
    ema3_period = int(ema3_period)
    if ema1_period <= 0 or ema2_period <= 0 or ema3_period <= 0:
        raise ValueError("EMA periods must be positive integers.")
    if not (ema1_period <= ema2_period <= ema3_period):
        raise ValueError("EMA periods must be ordered as EMA1 <= EMA2 <= EMA3.")
    return ema1_period, ema2_period, ema3_period


_ema_series_cache = {}


def get_ema_series(close_series, period):
    cache_key = (id(close_series), int(period))
    if cache_key not in _ema_series_cache:
        ema = vbt.MA.run(close_series, int(period), ewm=True).ma
        _ema_series_cache[cache_key] = pd.Series(ema.values.flatten(), index=close_series.index)
    return _ema_series_cache[cache_key]


def build_triple_ema_signals(close_series, params, shift_signals=True):
    ema1_period, ema2_period, ema3_period = validate_triple_ema_params(
        params["ema1_period"],
        params["ema2_period"],
        params["ema3_period"]
    )
    ema1 = get_ema_series(close_series, ema1_period)
    ema2 = get_ema_series(close_series, ema2_period)
    ema3 = get_ema_series(close_series, ema3_period)

    entries = (
        ema1.vbt.crossed_above(ema2) |
        ema1.vbt.crossed_above(ema3) |
        ema2.vbt.crossed_above(ema3)
    ).reindex(close_series.index).fillna(False)
    exits = (
        ema1.vbt.crossed_below(ema2) |
        ema1.vbt.crossed_below(ema3) |
        ema2.vbt.crossed_below(ema3)
    ).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_triple_ema_params(params):
    return f"EMA({params['ema1_period']},{params['ema2_period']},{params['ema3_period']})"


def validate_macd_params(fast_period, slow_period, signal_period):
    fast_period = int(fast_period)
    slow_period = int(slow_period)
    signal_period = int(signal_period)
    if fast_period <= 0 or slow_period <= 0 or signal_period <= 0:
        raise ValueError("MACD periods must be positive integers.")
    if fast_period >= slow_period:
        raise ValueError("MACD fast period must be less than slow period.")
    return fast_period, slow_period, signal_period


def compute_macd(close_series, fast_period, slow_period, signal_period):
    fast_period, slow_period, signal_period = validate_macd_params(
        fast_period,
        slow_period,
        signal_period
    )
    close_series = pd.Series(close_series, index=close_series.index).astype(float)
    if close_series.dropna().shape[0] < slow_period + signal_period:
        raise ValueError("Not enough price bars to compute MACD for the selected periods.")

    macd_line_arr, macd_signal_arr, macd_hist_arr = talib.MACD(
        close_series.to_numpy(dtype=float),
        fastperiod=fast_period,
        slowperiod=slow_period,
        signalperiod=signal_period
    )
    return (
        pd.Series(macd_line_arr, index=close_series.index),
        pd.Series(macd_signal_arr, index=close_series.index),
        pd.Series(macd_hist_arr, index=close_series.index)
    )


def build_macd_signals(close_series, params, shift_signals=True):
    macd_line, macd_signal_line, macd_hist = compute_macd(
        close_series,
        params["macd_fast_period"],
        params["macd_slow_period"],
        params["macd_signal_period"]
    )
    entries = macd_line.vbt.crossed_above(macd_signal_line).reindex(close_series.index).fillna(False)
    exits = macd_line.vbt.crossed_below(macd_signal_line).reindex(close_series.index).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool)
    )


def format_macd_params(params):
    return (
        f"MACD({params['macd_fast_period']},"
        f"{params['macd_slow_period']},"
        f"{params['macd_signal_period']})"
    )


# AROON Up/Down crossover strategy (TA-Lib AROON timeperiod)
def validate_aroon_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("AROON timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "AROON timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("AROON timeperiod must be a positive integer.")
    return timeperiod


def validate_aroon_entry_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"AROON entry level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("AROON entry level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"AROON entry level must be between 0 and 100 (Aroon scale); got {level_num}."
        )
    return level_num


AROON_ENTRY_LEVEL = validate_aroon_entry_level(AROON_ENTRY_LEVEL)


_aroon_ohlc_cache = {}
_aroon_series_cache = {}


def get_aroon_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run the download and prepare cells first.")
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run the full Define Indicator Grids cell (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _aroon_ohlc_cache:
        return _aroon_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("AROON close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("AROON requires High >= Low on all valid close dates.")

    _aroon_ohlc_cache[cache_key] = (high, low, close)
    return _aroon_ohlc_cache[cache_key]


def compute_aroon(close_series, timeperiod):
    timeperiod = validate_aroon_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("AROON close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _aroon_series_cache:
        return _aroon_series_cache[cache_key]

    high, low, close = get_aroon_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute AROON for the selected timeperiod.")

    try:
        aroon_down_arr, aroon_up_arr = talib.AROON(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            timeperiod=timeperiod,
        )
    except Exception as exc:
        raise ValueError(
            f"talib.AROON failed for timeperiod={timeperiod}: {type(exc).__name__}: {exc}"
        ) from exc

    aroon_up = pd.Series(aroon_up_arr, index=close.index)
    aroon_down = pd.Series(aroon_down_arr, index=close.index)
    if (~(aroon_up.isna() | aroon_down.isna())).sum() == 0:
        raise ValueError("AROON could not be computed for the selected parameter combination.")

    _aroon_series_cache[cache_key] = (aroon_up, aroon_down)
    return _aroon_series_cache[cache_key]


def build_aroon_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for AROON, got {type(params).__name__}.")
    missing_keys = [key for key in ("aroon_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"AROON params missing keys: {missing_keys}")
    if "AROON_ENTRY_LEVEL" not in globals():
        raise ValueError("AROON_ENTRY_LEVEL is missing. Re-run the Define Indicator Grids cell.")
    entry_level = validate_aroon_entry_level(AROON_ENTRY_LEVEL)

    aroon_up, aroon_down = compute_aroon(close_series, params["aroon_timeperiod"])
    # Require both lines valid on this bar and the prior bar so TA-Lib warmup NaNs
    # do not register as false Up/Down crosses (same pattern as KAMA/RSI).
    valid = aroon_up.notna() & aroon_down.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Entry trigger: Up crosses above Down, and Up is strictly above the entry level.
    entries = (
        aroon_up.vbt.crossed_above(aroon_down)
        & (aroon_up > entry_level)
        & valid_cross
    ).fillna(False)
    exits = (aroon_down.vbt.crossed_above(aroon_up) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_aroon_params(params):
    return f"AROON({params['aroon_timeperiod']})"


def validate_aroon_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_aroon_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "AROON sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("AROON sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# STC Schaff Trend Cycle strategy (EMA short/long + cycle period)
def validate_stc_params(ema_short, ema_long, cycle_period):
    ema_short = int(ema_short)
    ema_long = int(ema_long)
    cycle_period = int(cycle_period)
    if ema_short <= 0 or ema_long <= 0 or cycle_period <= 0:
        raise ValueError("STC periods must be positive integers.")
    if ema_short >= ema_long:
        raise ValueError("STC short EMA period must be less than long EMA period.")
    return ema_short, ema_long, cycle_period


def compute_stc(close_series, ema_short, ema_long, cycle_period):
    ema_short, ema_long, cycle_period = validate_stc_params(ema_short, ema_long, cycle_period)
    close = np.asarray(close_series, dtype=float)
    stc_ema_short = talib.EMA(close, timeperiod=ema_short)
    stc_ema_long = talib.EMA(close, timeperiod=ema_long)
    stc_macd = stc_ema_short - stc_ema_long

    stc_macd_highest = talib.MAX(stc_macd, timeperiod=cycle_period)
    stc_macd_lowest = talib.MIN(stc_macd, timeperiod=cycle_period)
    stc_macd_denom = stc_macd_highest - stc_macd_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc_k_macd = np.where(
            np.abs(stc_macd_denom) > 0,
            100.0 * (stc_macd - stc_macd_lowest) / stc_macd_denom,
            0.0,
        )
    stc_d_macd = talib.EMA(stc_k_macd, timeperiod=cycle_period)

    stc_d_highest = talib.MAX(stc_d_macd, timeperiod=cycle_period)
    stc_d_lowest = talib.MIN(stc_d_macd, timeperiod=cycle_period)
    stc_d_denom = stc_d_highest - stc_d_lowest
    with np.errstate(divide="ignore", invalid="ignore"):
        stc = np.where(
            np.abs(stc_d_denom) > 0,
            100.0 * (stc_d_macd - stc_d_lowest) / stc_d_denom,
            0.0,
        )
    stc = pd.Series(stc, index=close_series.index).replace([np.inf, -np.inf], np.nan)
    return stc


def build_stc_signals_from_series(
    stc,
    buy_level=STC_BUY_LEVEL,
    sell_level=STC_SELL_LEVEL,
    shift_signals=True,
):
    entries = (stc > buy_level) & (stc.shift(1) <= buy_level)
    exits = (stc < sell_level) & (stc.shift(1) >= sell_level)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=stc.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=stc.index, dtype=bool),
    )


def build_stc_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for STC, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"STC params missing keys: {missing_keys}")

    ema_short, ema_long, cycle_period = validate_stc_params(
        params["stc_ema_short_period"],
        params["stc_ema_long_period"],
        params["stc_cycle_period"],
    )
    stc = compute_stc(close_series, ema_short, ema_long, cycle_period)
    if stc.notna().sum() == 0:
        raise ValueError("STC could not be computed for the selected parameter combination.")
    return build_stc_signals_from_series(stc, shift_signals=shift_signals)


def format_stc_params(params):
    return (
        f"STC({params['stc_ema_short_period']},"
        f"{params['stc_ema_long_period']},"
        f"{params['stc_cycle_period']})"
    )


# KAMA price crossover strategy (custom KAMA: period + fastest/slowest EMA constants)
if (
    "validate_kama_params" not in globals()
    or "_compute_kama_core" not in globals()
):
    raise ValueError("KAMA helpers are missing. Run the TA-Lib indicator cell first.")


_kama_series_cache = {}


def get_kama_series(close_series, period, fast_ema_constant, slow_ema_constant):
    period, fast_ema_constant, slow_ema_constant = validate_kama_params(
        period,
        fast_ema_constant,
        slow_ema_constant,
    )
    cache_key = (id(close_series), period, fast_ema_constant, slow_ema_constant)
    if cache_key not in _kama_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        kama_line = _compute_kama_core(
            close,
            period,
            fast_ema_constant,
            slow_ema_constant,
        )
        if kama_line.notna().sum() == 0:
            raise ValueError("KAMA could not be computed for the selected parameter combination.")
        _kama_series_cache[cache_key] = (close, kama_line)
    return _kama_series_cache[cache_key]


def build_kama_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KAMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KAMA params missing keys: {missing_keys}")

    close, kama_line = get_kama_series(
        close_series,
        params["kama_period"],
        params["kama_fast_ema_constant"],
        params["kama_slow_ema_constant"],
    )
    valid_kama = kama_line.notna()
    valid_cross = valid_kama & valid_kama.shift(1).fillna(False)
    entries = (close.vbt.crossed_above(kama_line) & valid_cross).fillna(False)
    exits = (close.vbt.crossed_below(kama_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kama_params(params):
    return (
        f"KAMA({params['kama_period']},"
        f"{params['kama_fast_ema_constant']},"
        f"{params['kama_slow_ema_constant']})"
    )


# SUPERTREND trend-flip strategy (ATR length + factor/multiplier via TA-Lib ATR)
if (
    "validate_supertrend_params" not in globals()
    or "_compute_supertrend_core" not in globals()
):
    raise ValueError("SUPERTREND helpers are missing. Run the TA-Lib indicator cell first.")


_supertrend_ohlc_cache = {}
_supertrend_series_cache = {}


def _select_ohlc_column(df, ticker, column_name):
    if isinstance(df.columns, pd.MultiIndex):
        key = (column_name, ticker)
        if key in df.columns:
            series = df[key]
        else:
            # Exact level match only — avoid substring false positives like "HigherHigh"
            cols = [
                c for c in df.columns
                if isinstance(c, tuple) and len(c) >= 1 and c[0] == column_name
            ]
            if not cols:
                raise KeyError(f"{column_name} not found for ticker {ticker!r}")
            series = df[cols[0]]
    else:
        if column_name not in df.columns:
            raise KeyError(f"{column_name} not found in stock_data columns")
        series = df[column_name]
    return pd.Series(series.astype(float).to_numpy(), index=series.index, name=column_name)


def get_supertrend_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run the download and prepare cells first.")
    if close_series is None or len(close_series) == 0:
        raise ValueError("SUPERTREND close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _supertrend_ohlc_cache:
        return _supertrend_ohlc_cache[cache_key]

    high = _select_ohlc_column(stock_data, TICKER, "High").reindex(close_series.index)
    low = _select_ohlc_column(stock_data, TICKER, "Low").reindex(close_series.index)
    close = pd.Series(close_series, index=close_series.index).astype(float)

    valid_close = close.notna()
    if not valid_close.any():
        raise ValueError("SUPERTREND close_series contains no finite prices.")
    if high.loc[valid_close].isna().any() or low.loc[valid_close].isna().any():
        raise ValueError(
            "High/Low data is missing for one or more dates in the selected price series."
        )
    if (high.loc[valid_close] < low.loc[valid_close]).any():
        raise ValueError("SUPERTREND requires High >= Low on all valid close dates.")

    _supertrend_ohlc_cache[cache_key] = (high, low, close)
    return _supertrend_ohlc_cache[cache_key]


def get_supertrend_series(close_series, atr_length, factor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    cache_key = (id(close_series), atr_length, factor)
    if cache_key not in _supertrend_series_cache:
        high, low, close = get_supertrend_ohlc(close_series)
        supertrend_line, direction = _compute_supertrend_core(
            high.to_numpy(dtype=float),
            low.to_numpy(dtype=float),
            close.to_numpy(dtype=float),
            atr_length,
            factor,
        )
        supertrend_line = pd.Series(supertrend_line, index=close.index)
        direction = pd.Series(direction, index=close.index)
        if direction.notna().sum() == 0:
            raise ValueError("SUPERTREND could not be computed for the selected parameter combination.")
        _supertrend_series_cache[cache_key] = (close, supertrend_line, direction)
    return _supertrend_series_cache[cache_key]


def build_supertrend_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for SUPERTREND, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("supertrend_atr_length", "supertrend_factor")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"SUPERTREND params missing keys: {missing_keys}")

    close, _supertrend_line, direction = get_supertrend_series(
        close_series,
        params["supertrend_atr_length"],
        params["supertrend_factor"],
    )
    valid_dir = direction.notna()
    valid_flip = valid_dir & valid_dir.shift(1).fillna(False)
    # Buy: close above red line → direction flips to green (+1)
    entries = ((direction == 1.0) & (direction.shift(1) == -1.0) & valid_flip).fillna(False)
    # Sell: close below green line → direction flips to red (-1)
    exits = ((direction == -1.0) & (direction.shift(1) == 1.0) & valid_flip).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_supertrend_params(params):
    return (
        f"SUPERTREND(ATR={params['supertrend_atr_length']},"
        f"Factor={params['supertrend_factor']})"
    )


def validate_supertrend_sensitivity_params(atr_length, factor, factor_anchor):
    atr_length, factor = validate_supertrend_params(atr_length, factor)
    if int(factor_anchor) != factor:
        raise ValueError("SUPERTREND sensitivity anchor must match the factor.")
    return atr_length, factor, factor


# KALMAN price vs filter strategy (Q=process noise, R=measurement noise)
if (
    "validate_kalman_params" not in globals()
    or "_compute_kalman_core" not in globals()
):
    raise ValueError("KALMAN helpers are missing. Run the TA-Lib indicator cell first.")


_kalman_series_cache = {}


def get_kalman_series(close_series, process_noise, measurement_noise):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if close_series is None or len(close_series) == 0:
        raise ValueError("KALMAN close_series is empty.")
    cache_key = (id(close_series), process_noise, measurement_noise)
    if cache_key not in _kalman_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("KALMAN close_series contains no finite prices.")
        kalman_line = pd.Series(
            _compute_kalman_core(close_values, process_noise, measurement_noise),
            index=close.index,
        )
        if kalman_line.notna().sum() == 0:
            raise ValueError("KALMAN could not be computed for the selected parameter combination.")
        _kalman_series_cache[cache_key] = (close, kalman_line)
    return _kalman_series_cache[cache_key]


def build_kalman_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for KALMAN, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("kalman_process_noise", "kalman_measurement_noise")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"KALMAN params missing keys: {missing_keys}")

    close, kalman_line = get_kalman_series(
        close_series,
        params["kalman_process_noise"],
        params["kalman_measurement_noise"],
    )
    valid = kalman_line.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: price closes above the Kalman filter line (cross into above)
    entries = (close.vbt.crossed_above(kalman_line) & valid_cross).fillna(False)
    # Sell: price closes below the Kalman filter line (cross into below)
    exits = (close.vbt.crossed_below(kalman_line) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close.index, dtype=bool),
    )


def format_kalman_params(params):
    return (
        f"KALMAN(Q={params['kalman_process_noise']},"
        f"R={params['kalman_measurement_noise']})"
    )


def validate_kalman_sensitivity_params(process_noise, measurement_noise, measurement_noise_anchor):
    process_noise, measurement_noise = validate_kalman_params(process_noise, measurement_noise)
    if int(measurement_noise_anchor) != measurement_noise:
        raise ValueError("KALMAN sensitivity anchor must match the measurement noise (R).")
    return process_noise, measurement_noise, measurement_noise


# RSI midline crossover strategy (TA-Lib RSI timeperiod; buy/sell on cross of 50)
def validate_rsi_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"RSI timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("RSI timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "RSI timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("RSI timeperiod must be a positive integer.")
    return timeperiod


_rsi_series_cache = {}


def compute_rsi(close_series, timeperiod):
    timeperiod = validate_rsi_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("RSI close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _rsi_series_cache:
        return _rsi_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("RSI close_series contains no finite prices.")
    if np.isfinite(close_values).sum() < timeperiod + 1:
        raise ValueError("Not enough price bars to compute RSI for the selected timeperiod.")

    rsi = pd.Series(
        talib.RSI(close_values, timeperiod=timeperiod),
        index=close.index,
    )
    if rsi.notna().sum() == 0:
        raise ValueError("RSI could not be computed for the selected parameter combination.")
    _rsi_series_cache[cache_key] = rsi
    return rsi


def build_rsi_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for RSI, got {type(params).__name__}.")
    missing_keys = [key for key in ("rsi_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"RSI params missing keys: {missing_keys}")

    rsi = compute_rsi(close_series, params["rsi_timeperiod"])
    valid = rsi.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: RSI crosses above 50 (rising bullish momentum)
    entries = (
        (rsi > RSI_CROSS_LEVEL) & (rsi.shift(1) <= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: RSI crosses below 50 (increasing bearish momentum)
    exits = (
        (rsi < RSI_CROSS_LEVEL) & (rsi.shift(1) >= RSI_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=rsi.index, dtype=bool),
    )


def format_rsi_params(params):
    return f"RSI({params['rsi_timeperiod']})"


def validate_rsi_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_rsi_params(timeperiod)
    if int(timeperiod_anchor_a) != timeperiod or int(timeperiod_anchor_b) != timeperiod:
        raise ValueError("RSI sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod



# ADX +DI/-DI crossover strategy (TA-Lib PLUS_DI / MINUS_DI; ADX = Wilder-smoothed DX)
def validate_adx_params(di_length, adx_smoothing):
    try:
        di_length_num = float(di_length)
        adx_smoothing_num = float(adx_smoothing)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"ADX params must be numeric; got di_length={di_length!r}, "
            f"adx_smoothing={adx_smoothing!r}"
        ) from exc
    if not np.isfinite(di_length_num) or not np.isfinite(adx_smoothing_num):
        raise ValueError("ADX DI length and ADX smoothing must be finite numbers.")
    if di_length_num != int(di_length_num):
        raise ValueError(
            "ADX DI length must be a whole number for this grid "
            f"(sensitivity uses int grids); got di_length={di_length!r}"
        )
    if adx_smoothing_num != int(adx_smoothing_num):
        raise ValueError(
            "ADX smoothing timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got adx_smoothing={adx_smoothing!r}"
        )
    di_length = int(di_length_num)
    adx_smoothing = int(adx_smoothing_num)
    if di_length <= 0 or adx_smoothing <= 0:
        raise ValueError("ADX DI length and ADX smoothing must be positive integers.")
    return di_length, adx_smoothing


def validate_adx_trend_level(level):
    try:
        level_num = float(level)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"ADX trend level must be numeric; got level={level!r}") from exc
    if not np.isfinite(level_num):
        raise ValueError("ADX trend level must be a finite number.")
    if not 0.0 <= level_num <= 100.0:
        raise ValueError(
            f"ADX trend level must be between 0 and 100 (ADX scale); got {level_num}."
        )
    return level_num


ADX_TREND_LEVEL = validate_adx_trend_level(ADX_TREND_LEVEL)


_adx_ohlc_cache = {}
_adx_di_cache = {}
_adx_series_cache = {}


def get_adx_ohlc(close_series):
    if "stock_data" not in globals() or "TICKER" not in globals():
        raise ValueError("stock_data/TICKER are missing. Run the download and prepare cells first.")
    if "_select_ohlc_column" not in globals():
        raise ValueError(
            "Shared OHLC helper `_select_ohlc_column` is missing. "
            "Run the full Define Indicator Grids cell (SUPERTREND section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")

    cache_key = id(close_series)
    if cache_key in _adx_ohlc_cache:
        return _adx_ohlc_cache[cache_key]

    price_index = pd.DatetimeIndex(close_series.index)
    if price_index.tz is not None:
        price_index = price_index.tz_localize(None)

    high = _select_ohlc_column(stock_data, TICKER, "High")
    low = _select_ohlc_column(stock_data, TICKER, "Low")
    if isinstance(high.index, pd.DatetimeIndex) and high.index.tz is not None:
        high = high.copy()
        high.index = high.index.tz_localize(None)
    if isinstance(low.index, pd.DatetimeIndex) and low.index.tz is not None:
        low = low.copy()
        low.index = low.index.tz_localize(None)

    high = high.reindex(price_index)
    low = low.reindex(price_index)
    close = pd.Series(
        pd.Series(close_series).astype(float).to_numpy(),
        index=price_index,
        name=getattr(close_series, "name", None),
    )

    finite_close = np.isfinite(close.to_numpy(dtype=float))
    if not finite_close.any():
        raise ValueError("ADX close_series contains no finite prices.")
    valid_mask = pd.Series(finite_close, index=close.index)
    high_vals = high.loc[valid_mask].to_numpy(dtype=float)
    low_vals = low.loc[valid_mask].to_numpy(dtype=float)
    if (not np.isfinite(high_vals).all()) or (not np.isfinite(low_vals).all()):
        raise ValueError(
            "High/Low data is missing or non-finite for one or more dates in the selected price series."
        )
    if (high_vals < low_vals).any():
        raise ValueError("ADX requires High >= Low on all valid close dates.")

    _adx_ohlc_cache[cache_key] = (high, low, close)
    return _adx_ohlc_cache[cache_key]


def _compute_adx_di_lines(close_series, di_length):
    """TA-Lib +DI/-DI for a DI length; cached so each length is computed once per series."""
    di_length = validate_adx_params(di_length, di_length)[0]
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length)
    if cache_key in _adx_di_cache:
        return _adx_di_cache[cache_key]

    high, low, close = get_adx_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= di_length:
        raise ValueError("Not enough price bars to compute ADX DI lines for the selected DI length.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    try:
        plus_di_arr = talib.PLUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
        minus_di_arr = talib.MINUS_DI(high_vals, low_vals, close_vals, timeperiod=di_length)
    except Exception as exc:
        raise ValueError(
            f"talib PLUS_DI/MINUS_DI failed for di_length={di_length}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    plus_di = pd.Series(plus_di_arr, index=close.index)
    minus_di = pd.Series(minus_di_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna())).sum() == 0:
        raise ValueError("ADX DI lines could not be computed for the selected DI length.")

    _adx_di_cache[cache_key] = (plus_di, minus_di, close)
    return _adx_di_cache[cache_key]


def _wilder_smooth_dx(dx_values, adx_smoothing):
    """Wilder/RMA smooth of DX -> ADX (same DI lines for any DI length / smoothing pair)."""
    dx = np.asarray(dx_values, dtype=float)
    adx = np.full(dx.shape, np.nan, dtype=float)
    if adx_smoothing <= 0:
        raise ValueError("ADX smoothing must be a positive integer.")
    n = int(dx.size)
    if n == 0:
        return adx

    # Seed with SMA of the first contiguous `adx_smoothing` finite DX values,
    # then Wilder-smooth — common ADX / TA-Lib-style initialization.
    run = 0
    seed_end = None
    for i in range(n):
        if np.isfinite(dx[i]):
            run += 1
            if run == adx_smoothing:
                seed_end = i
                break
        else:
            run = 0
    if seed_end is None:
        return adx

    seed_start = seed_end - adx_smoothing + 1
    prev = float(np.mean(dx[seed_start : seed_end + 1]))
    adx[seed_end] = prev
    for i in range(seed_end + 1, n):
        if not np.isfinite(dx[i]):
            adx[i] = np.nan
            continue
        prev = ((prev * (adx_smoothing - 1)) + float(dx[i])) / adx_smoothing
        adx[i] = prev
    return adx


def compute_adx(close_series, di_length, adx_smoothing):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ADX close_series is empty.")
    cache_key = (id(close_series), di_length, adx_smoothing)
    if cache_key in _adx_series_cache:
        return _adx_series_cache[cache_key]

    plus_di, minus_di, close = _compute_adx_di_lines(close_series, di_length)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # DI warmup + ADX seed window
    min_bars = di_length + adx_smoothing
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute ADX for the selected parameter combination "
            f"(need at least di_length + adx_smoothing = {min_bars} finite bars)."
        )

    plus_vals = plus_di.to_numpy(dtype=float)
    minus_vals = minus_di.to_numpy(dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        denom = plus_vals + minus_vals
        dx_vals = np.where(
            np.isfinite(denom) & (denom != 0.0),
            100.0 * np.abs(plus_vals - minus_vals) / denom,
            np.nan,
        )
    if not np.isfinite(dx_vals).any():
        raise ValueError("ADX DX could not be computed for the selected parameter combination.")

    try:
        adx_arr = _wilder_smooth_dx(dx_vals, adx_smoothing)
    except Exception as exc:
        raise ValueError(
            f"ADX smoothing failed for di_length={di_length}, "
            f"adx_smoothing={adx_smoothing}: {type(exc).__name__}: {exc}"
        ) from exc

    adx = pd.Series(adx_arr, index=close.index)
    if (~(plus_di.isna() | minus_di.isna() | adx.isna())).sum() == 0:
        raise ValueError("ADX could not be computed for the selected parameter combination.")

    _adx_series_cache[cache_key] = (plus_di, minus_di, adx)
    return _adx_series_cache[cache_key]


def build_adx_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    if "ADX_TREND_LEVEL" not in globals():
        raise ValueError("ADX_TREND_LEVEL is missing. Re-run the Define Indicator Grids cell.")
    trend_level = validate_adx_trend_level(ADX_TREND_LEVEL)

    plus_di, minus_di, adx = compute_adx(
        close_series,
        params["adx_di_length"],
        params["adx_smoothing_timeperiod"],
    )
    if len(plus_di) != len(close_series):
        raise ValueError(
            "ADX output length does not match close_series; check OHLC alignment."
        )
    # Require DI lines and ADX valid on this bar and the prior bar so warmup NaNs
    # do not register as false +DI/-DI crosses (same pattern as AROON/KAMA/RSI).
    valid = plus_di.notna() & minus_di.notna() & adx.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: +DI crosses above -DI while ADX confirms a strong uptrend
    entries = (
        plus_di.vbt.crossed_above(minus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)
    # Sell: -DI crosses above +DI while ADX confirms a strong downtrend
    exits = (
        minus_di.vbt.crossed_above(plus_di)
        & (adx > trend_level)
        & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_adx_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ADX, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("adx_di_length", "adx_smoothing_timeperiod")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ADX params missing keys: {missing_keys}")
    return (
        f"ADX(DI={params['adx_di_length']},"
        f"Smooth={params['adx_smoothing_timeperiod']})"
    )


def validate_adx_sensitivity_params(di_length, adx_smoothing, adx_smoothing_anchor):
    di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
    try:
        anchor = int(adx_smoothing_anchor)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "ADX sensitivity anchor must be an integer matching the ADX smoothing timeperiod."
        ) from exc
    if anchor != adx_smoothing:
        raise ValueError("ADX sensitivity anchor must match the ADX smoothing timeperiod.")
    return di_length, adx_smoothing, adx_smoothing


# DONCHIAN upper-band breakout strategy (TA-Lib MAX/MIN timeperiod; sell on middle-band cross)
def validate_donchian_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"DONCHIAN timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("DONCHIAN timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "DONCHIAN timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("DONCHIAN timeperiod must be a positive integer.")
    return timeperiod


_donchian_series_cache = {}


def get_donchian_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run the full Define Indicator Grids cell (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("DONCHIAN " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "DONCHIAN requires", 1)) from exc
        raise


def compute_donchian(close_series, timeperiod):
    timeperiod = validate_donchian_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("DONCHIAN close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _donchian_series_cache:
        return _donchian_series_cache[cache_key]

    high, low, close = get_donchian_ohlc(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    if finite_bars <= timeperiod:
        raise ValueError("Not enough price bars to compute DONCHIAN for the selected timeperiod.")

    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    try:
        # TA-Lib has no DONCHIAN; upper/lower from MAX/MIN over High/Low.
        upper_arr = talib.MAX(high_vals, timeperiod=timeperiod)
        lower_arr = talib.MIN(low_vals, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib MAX/MIN failed for DONCHIAN timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    upper = pd.Series(upper_arr, index=close.index)
    lower = pd.Series(lower_arr, index=close.index)
    middle = (upper + lower) / 2.0
    if (~(upper.isna() | lower.isna() | middle.isna())).sum() == 0:
        raise ValueError("DONCHIAN could not be computed for the selected parameter combination.")

    _donchian_series_cache[cache_key] = (close, upper, middle, lower)
    return _donchian_series_cache[cache_key]


def build_donchian_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")

    close, upper, middle, _lower = compute_donchian(
        close_series, params["donchian_timeperiod"]
    )
    if len(close) != len(close_series):
        raise ValueError(
            "DONCHIAN output length does not match close_series; check OHLC alignment."
        )
    # Use prior bar's channel so close can break above upper (same-bar MAX(High)
    # embeds today's high and makes close > upper impossible).
    prior_upper = upper.shift(1)
    prior_middle = middle.shift(1)
    valid = prior_upper.notna() & prior_middle.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: close crosses above the prior upper band (uptrend breakout)
    entries = (close.vbt.crossed_above(prior_upper) & valid_cross).fillna(False)
    # Sell: close crosses below the prior middle band (trailing stop)
    exits = (close.vbt.crossed_below(prior_middle) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX pattern). close.index may be
    # tz-stripped inside get_aroon_ohlc and would mis-align the backtest.
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_donchian_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for DONCHIAN, got {type(params).__name__}.")
    missing_keys = [key for key in ("donchian_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"DONCHIAN params missing keys: {missing_keys}")
    return f"DONCHIAN({params['donchian_timeperiod']})"


def validate_donchian_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_donchian_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "DONCHIAN sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("DONCHIAN sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# TRIX zero-line crossover strategy (TA-Lib TRIX timeperiod; buy/sell on cross of 0)
def validate_trix_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"TRIX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("TRIX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "TRIX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("TRIX timeperiod must be a positive integer.")
    return timeperiod


_trix_series_cache = {}


def compute_trix(close_series, timeperiod):
    timeperiod = validate_trix_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("TRIX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _trix_series_cache:
        return _trix_series_cache[cache_key]

    close = pd.Series(close_series, index=close_series.index).astype(float)
    close_values = close.to_numpy(dtype=float)
    if not np.isfinite(close_values).any():
        raise ValueError("TRIX close_series contains no finite prices.")

    # TRIX is ROC of a triple EMA; needs far more warmup than timeperiod+1.
    # TA-Lib TRIX lookback is 3*(timeperiod-1)+1; need one more bar for a usable value.
    min_bars = 3 * (timeperiod - 1) + 2
    if np.isfinite(close_values).sum() < min_bars:
        raise ValueError(
            "Not enough price bars to compute TRIX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        trix_arr = talib.TRIX(close_values, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.TRIX failed for timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    trix = pd.Series(trix_arr, index=close.index)
    if trix.notna().sum() == 0:
        raise ValueError("TRIX could not be computed for the selected parameter combination.")
    _trix_series_cache[cache_key] = trix
    return trix


def build_trix_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    if "TRIX_CROSS_LEVEL" not in globals():
        raise ValueError("TRIX_CROSS_LEVEL is missing. Re-run the Define Indicator Grids cell.")

    trix = compute_trix(close_series, params["trix_timeperiod"])
    valid = trix.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: TRIX crosses above 0 (bullish momentum)
    entries = (
        (trix > TRIX_CROSS_LEVEL) & (trix.shift(1) <= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)
    # Sell: TRIX crosses below 0 (bearish momentum)
    exits = (
        (trix < TRIX_CROSS_LEVEL) & (trix.shift(1) >= TRIX_CROSS_LEVEL) & valid_cross
    ).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_trix_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for TRIX, got {type(params).__name__}.")
    missing_keys = [key for key in ("trix_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"TRIX params missing keys: {missing_keys}")
    return f"TRIX({params['trix_timeperiod']})"


def validate_trix_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_trix_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "TRIX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("TRIX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# VORTEX VI+/VI- crossover strategy (TA-Lib has no VORTEX; TRANGE/SUM over timeperiod)
def validate_vortex_params(timeperiod):
    try:
        timeperiod_num = float(timeperiod)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"VORTEX timeperiod must be numeric; got timeperiod={timeperiod!r}") from exc
    if not np.isfinite(timeperiod_num):
        raise ValueError("VORTEX timeperiod must be a finite number.")
    if timeperiod_num != int(timeperiod_num):
        raise ValueError(
            "VORTEX timeperiod must be a whole number for this grid "
            f"(sensitivity uses int grids); got timeperiod={timeperiod!r}"
        )
    timeperiod = int(timeperiod_num)
    if timeperiod <= 0:
        raise ValueError("VORTEX timeperiod must be a positive integer.")
    return timeperiod


_vortex_series_cache = {}
_vortex_base_cache = {}


def get_vortex_ohlc(close_series):
    """Reuse AROON's High/Low alignment cache — same extract, no second OHLC pass."""
    if "get_aroon_ohlc" not in globals():
        raise ValueError(
            "Shared OHLC helper get_aroon_ohlc is missing. "
            "Run the full Define Indicator Grids cell (AROON section included)."
        )
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    try:
        return get_aroon_ohlc(close_series)
    except ValueError as exc:
        msg = str(exc)
        if msg.startswith("AROON "):
            raise ValueError("VORTEX " + msg[len("AROON "):]) from exc
        if "AROON requires" in msg:
            raise ValueError(msg.replace("AROON requires", "VORTEX requires", 1)) from exc
        raise


def _get_vortex_base_arrays(close_series):
    """Cache TRANGE and VM+/- once per price series (independent of timeperiod)."""
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = id(close_series)
    if cache_key in _vortex_base_cache:
        return _vortex_base_cache[cache_key]

    high, low, close = get_vortex_ohlc(close_series)
    high_vals = high.to_numpy(dtype=float)
    low_vals = low.to_numpy(dtype=float)
    close_vals = close.to_numpy(dtype=float)
    if not (len(high_vals) == len(low_vals) == len(close_vals)):
        raise ValueError("VORTEX High/Low/Close lengths do not match after OHLC alignment.")

    try:
        # TA-Lib has no VORTEX; VI+/VI- from vortex movement vs true range.
        tr_arr = talib.TRANGE(high_vals, low_vals, close_vals)
    except Exception as exc:
        raise ValueError(
            f"talib.TRANGE failed for VORTEX: {type(exc).__name__}: {exc}"
        ) from exc

    # First bar has no prior High/Low; leave VM undefined (NaN), not 0.
    vm_plus = np.full(len(high_vals), np.nan, dtype=float)
    vm_minus = np.full(len(high_vals), np.nan, dtype=float)
    vm_plus[1:] = np.abs(high_vals[1:] - low_vals[:-1])
    vm_minus[1:] = np.abs(low_vals[1:] - high_vals[:-1])

    _vortex_base_cache[cache_key] = (tr_arr, vm_plus, vm_minus, close)
    return _vortex_base_cache[cache_key]


def compute_vortex(close_series, timeperiod):
    timeperiod = validate_vortex_params(timeperiod)
    if close_series is None or len(close_series) == 0:
        raise ValueError("VORTEX close_series is empty.")
    cache_key = (id(close_series), timeperiod)
    if cache_key in _vortex_series_cache:
        return _vortex_series_cache[cache_key]

    tr_arr, vm_plus, vm_minus, close = _get_vortex_base_arrays(close_series)
    finite_bars = int(np.isfinite(close.to_numpy(dtype=float)).sum())
    # First VI needs a prior bar for VM+/- and `timeperiod` finite TR/VM values.
    min_bars = timeperiod + 1
    if finite_bars < min_bars:
        raise ValueError(
            "Not enough price bars to compute VORTEX for the selected timeperiod "
            f"(need >= {min_bars} finite bars for timeperiod={timeperiod})."
        )

    try:
        sum_vm_plus = talib.SUM(vm_plus, timeperiod=timeperiod)
        sum_vm_minus = talib.SUM(vm_minus, timeperiod=timeperiod)
        sum_tr = talib.SUM(tr_arr, timeperiod=timeperiod)
    except Exception as exc:
        raise ValueError(
            f"talib.SUM failed for VORTEX timeperiod={timeperiod}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    vi_plus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    vi_minus_arr = np.full(len(sum_tr), np.nan, dtype=float)
    valid_tr = np.isfinite(sum_tr) & (sum_tr != 0)
    # Only divide where the TR sum is finite and non-zero.
    vi_plus_arr[valid_tr] = sum_vm_plus[valid_tr] / sum_tr[valid_tr]
    vi_minus_arr[valid_tr] = sum_vm_minus[valid_tr] / sum_tr[valid_tr]

    vi_plus = pd.Series(vi_plus_arr, index=close.index)
    vi_minus = pd.Series(vi_minus_arr, index=close.index)
    if (~(vi_plus.isna() | vi_minus.isna())).sum() == 0:
        raise ValueError("VORTEX could not be computed for the selected parameter combination.")

    _vortex_series_cache[cache_key] = (vi_plus, vi_minus)
    return _vortex_series_cache[cache_key]


def build_vortex_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")

    vi_plus, vi_minus = compute_vortex(close_series, params["vortex_timeperiod"])
    if len(vi_plus) != len(close_series) or len(vi_minus) != len(close_series):
        raise ValueError(
            "VORTEX output length does not match close_series; check OHLC alignment."
        )
    # Require both lines valid on this bar and the prior bar so warmup NaNs
    # do not register as false VI+/VI- crosses (same pattern as AROON/KAMA/RSI).
    valid = vi_plus.notna() & vi_minus.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: VI+ crosses upward from below VI-
    entries = (vi_plus.vbt.crossed_above(vi_minus) & valid_cross).fillna(False)
    # Sell: VI- crosses upward from below VI+
    exits = (vi_minus.vbt.crossed_above(vi_plus) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )

def format_vortex_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for VORTEX, got {type(params).__name__}.")
    missing_keys = [key for key in ("vortex_timeperiod",) if key not in params]
    if missing_keys:
        raise KeyError(f"VORTEX params missing keys: {missing_keys}")
    return f"VORTEX({params['vortex_timeperiod']})"


def validate_vortex_sensitivity_params(timeperiod, timeperiod_anchor_a, timeperiod_anchor_b):
    timeperiod = validate_vortex_params(timeperiod)
    try:
        anchor_a = int(timeperiod_anchor_a)
        anchor_b = int(timeperiod_anchor_b)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "VORTEX sensitivity anchors must be integers matching the timeperiod."
        ) from exc
    if anchor_a != timeperiod or anchor_b != timeperiod:
        raise ValueError("VORTEX sensitivity anchors must match the timeperiod.")
    return timeperiod, timeperiod, timeperiod


# ALMA crossover strategy (custom ALMA: fast/slow length + offset hundredths + sigma)
if (
    "validate_alma_params" not in globals()
    or "_compute_alma_core" not in globals()
):
    raise ValueError("ALMA helpers are missing. Run the TA-Lib indicator cell first.")


_alma_series_cache = {}


def validate_alma_crossover_params(fast_length, slow_length, offset, sigma):
    fast_length, offset, sigma = validate_alma_params(fast_length, offset, sigma)
    slow_length, offset, sigma = validate_alma_params(slow_length, offset, sigma)
    if fast_length >= slow_length:
        raise ValueError("ALMA fast length must be less than slow length.")
    return fast_length, slow_length, offset, sigma


def get_alma_series(close_series, length, offset, sigma):
    length, offset, sigma = validate_alma_params(length, offset, sigma)
    if close_series is None or len(close_series) == 0:
        raise ValueError("ALMA close_series is empty.")
    if not hasattr(close_series, "index"):
        raise TypeError(
            f"ALMA close_series must be a pandas Series; got {type(close_series).__name__}."
        )
    cache_key = (id(close_series), length, offset, sigma)
    if cache_key not in _alma_series_cache:
        close = pd.Series(close_series, index=close_series.index).astype(float)
        close_values = close.to_numpy(dtype=float)
        if not np.isfinite(close_values).any():
            raise ValueError("ALMA close_series contains no finite prices.")
        try:
            alma_line = pd.Series(
                _compute_alma_core(close_values, length, offset, sigma),
                index=close.index,
            )
        except ValueError as exc:
            raise ValueError(
                f"ALMA could not be computed for length={length}, offset={offset}, "
                f"sigma={sigma}: {exc}"
            ) from exc
        if alma_line.notna().sum() == 0:
            raise ValueError("ALMA could not be computed for the selected parameter combination.")
        _alma_series_cache[cache_key] = (close, alma_line)
    return _alma_series_cache[cache_key]


def build_alma_signals(close_series, params, shift_signals=True):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")

    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )

    close, alma_fast = get_alma_series(close_series, fast_length, offset, sigma)
    _, alma_slow = get_alma_series(close_series, slow_length, offset, sigma)
    if len(alma_fast) != len(close_series) or len(alma_slow) != len(close_series):
        raise ValueError(
            "ALMA output length does not match close_series; check close alignment."
        )
    # Require both ALMAs and close valid on this bar and the prior bar so warmup NaNs
    # do not register as false fast/slow crosses (same pattern as AROON/KAMA/RSI).
    valid = alma_fast.notna() & alma_slow.notna() & close.notna()
    valid_cross = valid & valid.shift(1).fillna(False)
    # Buy: fast ALMA crosses above slow ALMA
    entries = (alma_fast.vbt.crossed_above(alma_slow) & valid_cross).fillna(False)
    # Sell: fast ALMA crosses below slow ALMA
    exits = (alma_fast.vbt.crossed_below(alma_slow) & valid_cross).fillna(False)

    if shift_signals:
        entries = entries.shift(1).fillna(False)
        exits = exits.shift(1).fillna(False)

    # Return on the caller's index (AROON/ADX/DONCHIAN pattern).
    return (
        pd.Series(entries.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
        pd.Series(exits.to_numpy(dtype=bool), index=close_series.index, dtype=bool),
    )


def format_alma_params(params):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for ALMA, got {type(params).__name__}.")
    missing_keys = [
        key
        for key in ("alma_fast_length", "alma_slow_length", "alma_offset", "alma_sigma")
        if key not in params
    ]
    if missing_keys:
        raise KeyError(f"ALMA params missing keys: {missing_keys}")
    fast_length, slow_length, offset, sigma = validate_alma_crossover_params(
        params["alma_fast_length"],
        params["alma_slow_length"],
        params["alma_offset"],
        params["alma_sigma"],
    )
    return f"ALMA({fast_length},{slow_length},{offset / 100:.2f},{sigma})"


ema_combinations = []
for ema1 in ema1_periods:
    for ema2 in ema2_periods:
        for ema3 in ema3_periods:
            try:
                ema1, ema2, ema3 = validate_triple_ema_params(ema1, ema2, ema3)
                ema_combinations.append({
                    "ema1_period": ema1,
                    "ema2_period": ema2,
                    "ema3_period": ema3
                })
            except ValueError:
                pass

if not ema_combinations:
    raise ValueError("No valid Triple EMA parameter combinations were generated.")

macd_combinations = []
for fast_period in macd_fast_periods:
    for slow_period in macd_slow_periods:
        for signal_period in macd_signal_periods:
            try:
                fast_period, slow_period, signal_period = validate_macd_params(
                    fast_period,
                    slow_period,
                    signal_period
                )
                macd_combinations.append({
                    "macd_fast_period": fast_period,
                    "macd_slow_period": slow_period,
                    "macd_signal_period": signal_period
                })
            except ValueError:
                pass

if not macd_combinations:
    raise ValueError("No valid MACD parameter combinations were generated.")

aroon_combinations = []
for timeperiod in aroon_timeperiods:
    try:
        timeperiod = validate_aroon_params(timeperiod)
        aroon_combinations.append({
            "aroon_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not aroon_combinations:
    raise ValueError("No valid AROON parameter combinations were generated.")

stc_combinations = []
for ema_short in stc_ema_short_periods:
    for ema_long in stc_ema_long_periods:
        for cycle_period in stc_cycle_periods:
            try:
                ema_short, ema_long, cycle_period = validate_stc_params(
                    ema_short,
                    ema_long,
                    cycle_period,
                )
                stc_combinations.append({
                    "stc_ema_short_period": ema_short,
                    "stc_ema_long_period": ema_long,
                    "stc_cycle_period": cycle_period,
                })
            except ValueError:
                pass

if not stc_combinations:
    raise ValueError("No valid STC parameter combinations were generated.")

kama_combinations = []
for period in kama_periods:
    for fast_ema_constant in kama_fast_ema_constants:
        for slow_ema_constant in kama_slow_ema_constants:
            try:
                period, fast_ema_constant, slow_ema_constant = validate_kama_params(
                    period,
                    fast_ema_constant,
                    slow_ema_constant,
                )
                kama_combinations.append({
                    "kama_period": period,
                    "kama_fast_ema_constant": fast_ema_constant,
                    "kama_slow_ema_constant": slow_ema_constant,
                })
            except ValueError:
                pass

if not kama_combinations:
    raise ValueError("No valid KAMA parameter combinations were generated.")

supertrend_combinations = []
for atr_length in supertrend_atr_lengths:
    for factor in supertrend_factors:
        try:
            atr_length, factor = validate_supertrend_params(atr_length, factor)
            supertrend_combinations.append({
                "supertrend_atr_length": atr_length,
                "supertrend_factor": factor,
            })
        except ValueError:
            pass

if not supertrend_combinations:
    raise ValueError("No valid SUPERTREND parameter combinations were generated.")

kalman_combinations = []
for process_noise in kalman_process_noise_values:
    for measurement_noise in kalman_measurement_noise_values:
        try:
            process_noise, measurement_noise = validate_kalman_params(
                process_noise,
                measurement_noise,
            )
            kalman_combinations.append({
                "kalman_process_noise": process_noise,
                "kalman_measurement_noise": measurement_noise,
            })
        except ValueError:
            pass

if not kalman_combinations:
    raise ValueError("No valid KALMAN parameter combinations were generated.")

rsi_combinations = []
for timeperiod in rsi_timeperiods:
    try:
        timeperiod = validate_rsi_params(timeperiod)
        rsi_combinations.append({
            "rsi_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not rsi_combinations:
    raise ValueError("No valid RSI parameter combinations were generated.")

adx_combinations = []
for di_length in adx_di_lengths:
    for adx_smoothing in adx_smoothing_timeperiods:
        try:
            di_length, adx_smoothing = validate_adx_params(di_length, adx_smoothing)
            adx_combinations.append({
                "adx_di_length": di_length,
                "adx_smoothing_timeperiod": adx_smoothing,
            })
        except ValueError:
            pass

if not adx_combinations:
    raise ValueError("No valid ADX parameter combinations were generated.")

donchian_combinations = []
for timeperiod in donchian_timeperiods:
    try:
        timeperiod = validate_donchian_params(timeperiod)
        donchian_combinations.append({
            "donchian_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not donchian_combinations:
    raise ValueError("No valid DONCHIAN parameter combinations were generated.")

trix_combinations = []
for timeperiod in trix_timeperiods:
    try:
        timeperiod = validate_trix_params(timeperiod)
        trix_combinations.append({
            "trix_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not trix_combinations:
    raise ValueError("No valid TRIX parameter combinations were generated.")

vortex_combinations = []
for timeperiod in vortex_timeperiods:
    try:
        timeperiod = validate_vortex_params(timeperiod)
        vortex_combinations.append({
            "vortex_timeperiod": timeperiod,
        })
    except ValueError:
        pass

if not vortex_combinations:
    raise ValueError("No valid VORTEX parameter combinations were generated.")

alma_combinations = []
for fast_length in alma_fast_lengths:
    for slow_length in alma_slow_lengths:
        for offset in alma_offsets:
            for sigma in alma_sigmas:
                try:
                    fast_v, slow_v, offset_v, sigma_v = validate_alma_crossover_params(
                        fast_length, slow_length, offset, sigma
                    )
                    alma_combinations.append({
                        "alma_fast_length": fast_v,
                        "alma_slow_length": slow_v,
                        "alma_offset": offset_v,
                        "alma_sigma": sigma_v,
                    })
                except ValueError:
                    pass

if not alma_combinations:
    raise ValueError("No valid ALMA parameter combinations were generated.")

indicator_grids = {
    "Triple EMA": {
        "description": "Triple EMA crossover strategy",
        "params": ema_combinations,
        "build_signals": build_triple_ema_signals,
        "format_params": format_triple_ema_params
    },
    "MACD": {
        "description": "MACD line crossing its signal line",
        "params": macd_combinations,
        "build_signals": build_macd_signals,
        "format_params": format_macd_params
    },
    "AROON": {
        "description": "AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up",
        "params": aroon_combinations,
        "build_signals": build_aroon_signals,
        "format_params": format_aroon_params
    },
    "STC": {
        "description": "Schaff Trend Cycle: buy on cross above 50, sell on cross below 50",
        "params": stc_combinations,
        "build_signals": build_stc_signals,
        "format_params": format_stc_params
    },
    "KAMA": {
        "description": "Price crossing KAMA line: buy on cross above, sell on cross below",
        "params": kama_combinations,
        "build_signals": build_kama_signals,
        "format_params": format_kama_params
    },
    "SUPERTREND": {
        "description": "SUPERTREND flip: buy when close above red line, sell when close below green line",
        "params": supertrend_combinations,
        "build_signals": build_supertrend_signals,
        "format_params": format_supertrend_params
    },
    "KALMAN": {
        "description": "Price crossing Kalman filter: buy on cross above, sell on cross below",
        "params": kalman_combinations,
        "build_signals": build_kalman_signals,
        "format_params": format_kalman_params
    },
    "RSI": {
        "description": "RSI crossing 50: buy on cross above, sell on cross below",
        "params": rsi_combinations,
        "build_signals": build_rsi_signals,
        "format_params": format_rsi_params
    },
    "ADX": {
        "description": "ADX +DI/-DI crossover: buy when +DI crosses above -DI with ADX > trend level, sell when -DI crosses above +DI with ADX > trend level",
        "params": adx_combinations,
        "build_signals": build_adx_signals,
        "format_params": format_adx_params
    },
    "DONCHIAN": {
        "description": "DONCHIAN breakout: buy when close crosses above prior upper band, sell when close crosses below prior middle band",
        "params": donchian_combinations,
        "build_signals": build_donchian_signals,
        "format_params": format_donchian_params
    },
    "TRIX": {
        "description": "TRIX crossing 0: buy on cross above, sell on cross below",
        "params": trix_combinations,
        "build_signals": build_trix_signals,
        "format_params": format_trix_params
    },
    "VORTEX": {
        "description": "VORTEX VI+/VI- crossover: buy when VI+ crosses above VI-, sell when VI- crosses above VI+",
        "params": vortex_combinations,
        "build_signals": build_vortex_signals,
        "format_params": format_vortex_params
    },
    "ALMA": {
        "description": "ALMA fast/slow crossover: buy when fast crosses above slow, sell when fast crosses below slow",
        "params": alma_combinations,
        "build_signals": build_alma_signals,
        "format_params": format_alma_params
    },
}

active_indicator_names = ["Triple EMA", "MACD", "AROON", "STC", "KAMA", "SUPERTREND", "KALMAN", "RSI", "ADX", "DONCHIAN", "TRIX", "VORTEX", "ALMA"]
missing_indicator_names = [
    name for name in active_indicator_names
    if name not in indicator_grids
]
if missing_indicator_names:
    raise KeyError(f"Active indicators are not defined: {missing_indicator_names}")

active_indicator_grids = {
    name: indicator_grids[name]
    for name in active_indicator_names
}


def get_active_indicator_spec(indicator_name):
    if indicator_name not in active_indicator_grids:
        available = ", ".join(active_indicator_grids.keys())
        raise KeyError(f"Unknown indicator '{indicator_name}'. Available indicators: {available}")
    return active_indicator_grids[indicator_name]


def require_params_dict(params, indicator_name):
    if not isinstance(params, dict):
        raise TypeError(f"Expected params dict for {indicator_name}, got {type(params).__name__}.")
    return params




INIT_CASH = 100_000
FEE_RATE = 0.0005
SLIPPAGE_RATE = 0.0005


def trade_stats_from_portfolio(pf):
    """Extract trade-level stats; safe when there are zero trades."""
    trades = pf.trades
    total_trades = len(trades)
    stats = {
        "total_trades": total_trades,
        "win_rate_pct": np.nan,
        "profit_factor": np.nan,
        "expectancy": np.nan,
        "avg_win": np.nan,
        "avg_loss": np.nan,
    }
    if total_trades <= 0:
        return stats

    tr = trades.returns.values if hasattr(trades.returns, "values") else np.asarray(trades.returns)
    if tr.size == 0:
        return stats

    pos = tr[tr > 0]
    neg = tr[tr < 0]
    stats["win_rate_pct"] = (len(pos) / len(tr)) * 100.0
    gains = float(pos.sum()) if len(pos) else 0.0
    losses = float(abs(neg.sum())) if len(neg) else 0.0
    stats["profit_factor"] = (gains / losses) if losses > 0 else (np.inf if gains > 0 else np.nan)
    stats["expectancy"] = float(tr.mean())
    stats["avg_win"] = float(pos.mean()) if len(pos) else np.nan
    stats["avg_loss"] = float(abs(neg.mean())) if len(neg) else np.nan
    return stats


def run_signal_backtest(price, entries, exits):
    """Shared Portfolio.from_signals wrapper used by OOS / full-sample cells."""
    close_arg = price.to_numpy(dtype=float) if hasattr(price, "to_numpy") else np.asarray(price, dtype=float)
    if hasattr(entries, "to_numpy"):
        entries = entries.to_numpy(dtype=bool)
    if hasattr(exits, "to_numpy"):
        exits = exits.to_numpy(dtype=bool)
    return vbt.Portfolio.from_signals(
        close=close_arg,
        entries=entries,
        exits=exits,
        init_cash=INIT_CASH,
        fees=FEE_RATE,
        slippage=SLIPPAGE_RATE,
        freq=FREQ,
    )


def fmt_metric(value, spec=".3f", suffix=""):
    """Format metrics without crashing on NaN/None/inf."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "n/a"
    try:
        return format(float(value), spec) + suffix
    except (TypeError, ValueError):
        return "n/a"


def safe_float(value, default=np.nan):
    """Convert to float; return default for missing/invalid/non-finite values."""
    try:
        out = float(value)
        return out if np.isfinite(out) else default
    except (TypeError, ValueError):
        return default


_tied_is_sharpe_notes_printed = set()


def displayed_is_top_n(indicator_results, top_n=5):
    """Top-N rows shown in the in-sample results table."""
    if "sharpe_ratio" not in indicator_results.columns:
        raise KeyError("indicator_results is missing required column: sharpe_ratio")
    return indicator_results.dropna(subset=["sharpe_ratio"]).nlargest(top_n, "sharpe_ratio")


def is_sharpe_tied_at_best(indicator_results, rtol=1e-9, atol=1e-12):
    valid = indicator_results.dropna(subset=["sharpe_ratio"])
    if valid.empty:
        return False
    best_sharpe = valid["sharpe_ratio"].max()
    tied_count = np.isclose(valid["sharpe_ratio"], best_sharpe, rtol=rtol, atol=atol).sum()
    return int(tied_count) > 1


def maybe_print_tied_is_sharpe_note(indicator_name, indicator_results, used_param_displays, context, top_n=5):
    if indicator_name in _tied_is_sharpe_notes_printed:
        return

    required_columns = {"sharpe_ratio", "param_display"}
    missing_columns = required_columns.difference(indicator_results.columns)
    if missing_columns:
        return

    used_labels = {
        str(label).strip()
        for label in used_param_displays
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    if not used_labels:
        return

    display_top = displayed_is_top_n(indicator_results, top_n)
    if display_top.empty:
        return

    display_labels = {
        str(label).strip()
        for label in display_top["param_display"].tolist()
        if label is not None and pd.notna(label) and str(label).strip() and str(label).strip().lower() != "nan"
    }
    outside_display = sorted(used_labels - display_labels)
    if not outside_display or not is_sharpe_tied_at_best(indicator_results):
        return

    best_sharpe = float(display_top["sharpe_ratio"].iloc[0])
    print(
        f"Note ({context}): {indicator_name} has multiple parameter sets tied at the "
        f"best in-sample Sharpe ({best_sharpe:.3f}). The in-sample top-{top_n} table shows one "
        f"valid tied set; this step used different tied equivalent(s): "
        f"{', '.join(outside_display)}"
    )
    _tied_is_sharpe_notes_printed.add(indicator_name)


def best_is_row(indicator_results):
    """Pick the single best in-sample row by Sharpe (matches the IS results table)."""
    ranked = displayed_is_top_n(indicator_results, 1)
    if ranked.empty:
        return None
    return ranked.iloc[0]


def best_oos_row(oos_results_df):
    """Pick the single best OOS row by Sharpe among validated IS top-N candidates."""
    valid = oos_results_df.dropna(subset=["OOS_Sharpe"])
    if valid.empty:
        return None
    return valid.loc[valid["OOS_Sharpe"].idxmax()]


total_grid_combinations = sum(len(spec["params"]) for spec in active_indicator_grids.values())

print("Active indicator grids:")
for indicator_name, spec in active_indicator_grids.items():
    combinations = spec["params"]
    print(f"\n{indicator_name}: {spec['description']}")
    print(f"Generated {len(combinations)} valid {indicator_name} combinations")
    print("First 10 combinations preview:")
    for i, params in enumerate(combinations[:10], 1):
        print(f"  {i:2d}. {spec['format_params'](params)}")
    if len(combinations) > 10:
        print(f"   ... and {len(combinations) - 10} more combinations")

print(f"\nTotal combinations across active grids: {total_grid_combinations}")
print("Ready to test each indicator grid separately on training data!")


Active indicator grids:

Triple EMA: Triple EMA crossover strategy
Generated 66840 valid Triple EMA combinations
First 10 combinations preview:
   1. EMA(4,90,120)
   2. EMA(4,90,121)
   3. EMA(4,90,122)
   4. EMA(4,90,123)
   5. EMA(4,90,124)
   6. EMA(4,90,125)
   7. EMA(4,90,126)
   8. EMA(4,90,127)
   9. EMA(4,90,128)
  10. EMA(4,90,129)
   ... and 66830 more combinations

MACD: MACD line crossing its signal line
Generated 128000 valid MACD combinations
First 10 combinations preview:
   1. MACD(10,60,40)
   2. MACD(10,60,41)
   3. MACD(10,60,42)
   4. MACD(10,60,43)
   5. MACD(10,60,44)
   6. MACD(10,60,45)
   7. MACD(10,60,46)
   8. MACD(10,60,47)
   9. MACD(10,60,48)
  10. MACD(10,60,49)
   ... and 127990 more combinations

AROON: AROON Up/Down crossover: buy when Up crosses above Down and Up > entry level, sell when Down crosses above Up
Generated 76 valid AROON combinations
First 10 combinations preview:
   1. AROON(4)
   2. AROON(5)
   3. AROON(6)
   4. AROON(7)
   5. AROON(8)

In [8]:
# Initialize Multi-Indicator Results Collection System

# Create empty list to store all backtest results from every active indicator grid.
grid_search_results = []

print("Multi-Indicator Results Collection System Initialized")
print(f"   - Active indicators: {', '.join(active_indicator_grids.keys())}")
print(f"   - Will test {total_grid_combinations} total combinations")
print("   - Results will be stored in 'grid_search_results' list")

# Define what metrics we will collect (All TradingView-style metrics)
metrics_to_collect = [
    # Strategy Metadata
    "indicator",
    "strategy",
    "params",
    "param_display",
    "ema1_period",
    "ema2_period",
    "ema3_period",
    "macd_fast_period",
    "macd_slow_period",
    "macd_signal_period",
    "aroon_timeperiod",
    "stc_ema_short_period",
    "stc_ema_long_period",
    "stc_cycle_period",
    "supertrend_atr_length",
    "supertrend_factor",
    "kalman_process_noise",
    "kalman_measurement_noise",
    "rsi_timeperiod",
    "trix_timeperiod",
    "vortex_timeperiod",
    "alma_fast_length",
    "alma_slow_length",
    "alma_offset",
    "alma_sigma",
    
    # Return Metrics
    "total_return",
    "annualized_return",
    "total_profit",
    
    # Risk-Adjusted Return Metrics
    "sharpe_ratio",
    "sortino_ratio",
    "calmar_ratio",
    "omega_ratio",
    "information_ratio",
    "tail_ratio",
    "deflated_sharpe_ratio",
    
    # Risk Metrics
    "max_drawdown",
    "volatility",
    "ulcer_index",
    
    # Trade Performance Metrics
    "win_rate",
    "total_trades",
    "avg_trade_duration",
    "expectancy",
    "profit_factor", 
    "sqn",
    
    # Win/Loss Analysis
    "payoff_ratio",
    "largest_win",
    "largest_loss",
    "avg_win_amount",
    "avg_loss_amount",
    "winning_streak",
    "losing_streak",
    
    # Additional Ratios
    "recovery_factor",
    "gain_to_pain_ratio",
    "serenity_index"
]

print("Metrics to collect for each indicator combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the multi-indicator grid search!")


Multi-Indicator Results Collection System Initialized
   - Active indicators: Triple EMA, MACD, AROON, STC, KAMA, SUPERTREND, KALMAN, RSI, ADX, DONCHIAN, TRIX, VORTEX, ALMA
   - Will test 378754 total combinations
   - Results will be stored in 'grid_search_results' list
Metrics to collect for each indicator combination:
  1. Indicator
  2. Strategy
  3. Params
  4. Param Display
  5. Ema1 Period
  6. Ema2 Period
  7. Ema3 Period
  8. Macd Fast Period
  9. Macd Slow Period
  10. Macd Signal Period
  11. Aroon Timeperiod
  12. Stc Ema Short Period
  13. Stc Ema Long Period
  14. Stc Cycle Period
  15. Supertrend Atr Length
  16. Supertrend Factor
  17. Kalman Process Noise
  18. Kalman Measurement Noise
  19. Rsi Timeperiod
  20. Trix Timeperiod
  21. Vortex Timeperiod
  22. Alma Fast Length
  23. Alma Slow Length
  24. Alma Offset
  25. Alma Sigma
  26. Total Return
  27. Annualized Return
  28. Total Profit
  29. Sharpe Ratio
  30. Sortino Ratio
  31. Calmar Ratio
  32. Omega Ratio
  

In [9]:
# VISUALIZE SIGNALS FOR ONE EXAMPLE

example_params = {
    "macd_fast_period": 12,
    "macd_slow_period": 26,
    "macd_signal_period": 9
}
entries, exits = build_macd_signals(train_close, example_params, shift_signals=False)
macd_line, macd_signal_line, macd_hist = compute_macd(
    train_close,
    example_params["macd_fast_period"],
    example_params["macd_slow_period"],
    example_params["macd_signal_period"]
)

signals_df = pd.DataFrame({
    'Close': train_close.values,
    'MACD': macd_line.values,
    'MACD_Signal': macd_signal_line.values,
    'MACD_Hist': macd_hist.values,
    'Buy': entries.values.astype(bool),
    'Sell': exits.values.astype(bool)
}, index=train_close.index)
signals_df.index.name = 'Date'

pos = 0
pos_list = []
for buy, sell in zip(signals_df['Buy'], signals_df['Sell']):
    if buy:
        pos = 1
    elif sell:
        pos = 0
    pos_list.append(pos)
signals_df['Position'] = pos_list

print(f"Example {format_macd_params(example_params)} signals:")
print(signals_df.head(500).to_string())


Example MACD(12,26,9) signals:
                Close      MACD  MACD_Signal  MACD_Hist    Buy   Sell  Position
Date                                                                           
2018-01-02   5.778714       NaN          NaN        NaN  False  False         0
2018-01-03   5.947908       NaN          NaN        NaN  False  False         0
2018-01-04   5.982775       NaN          NaN        NaN  False  False         0
2018-01-05   6.163460       NaN          NaN        NaN  False  False         0
2018-01-08   6.232009       NaN          NaN        NaN  False  False         0
2018-01-09   6.236763       NaN          NaN        NaN  False  False         0
2018-01-10   6.192385       NaN          NaN        NaN  False  False         0
2018-01-11   6.316407       NaN          NaN        NaN  False  False         0
2018-01-12   6.450337       NaN          NaN        NaN  False  False         0
2018-01-16   6.400410       NaN          NaN        NaN  False  False         0
2018-01-1

In [10]:
# MULTI-INDICATOR GRID SEARCH ON TRAINING DATA - BATCHED

print("INITIATING MULTI-INDICATOR GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Active Indicators: {', '.join(active_indicator_grids.keys())}")
print(f"Training Period: {train_close.index[0].date()} -> {train_close.index[-1].date()}")
print(f"Market Regime: {market_label}")
print(f"Annualization Basis: {TRADING_DAYS_PER_YEAR} bars/year ({YEAR_FREQ})")
print("Initial Capital: $100,000")
print("Transaction Costs: 0.05% per trade (fees + slippage)")
print("Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

# Configuration
BATCH_SIZE = 1000  # Process combinations in batches to manage memory
MIN_TRADES_PER_YEAR = 2
train_years = sample_years(train_close)
total_combinations = total_grid_combinations

print(f"Total combinations to test: {total_combinations}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Minimum trade frequency filter: {MIN_TRADES_PER_YEAR}/year")
print("Each indicator grid is run separately.\n")

def metric_scalar(values, idx):
    """Return a scalar metric from vectorbt outputs that may be scalar, Series, or array-like."""
    if isinstance(values, pd.Series):
        return float(values.iloc[idx])
    if isinstance(values, pd.DataFrame):
        return float(values.iloc[:, idx].iloc[0])

    arr = np.asarray(values)
    if arr.ndim == 0:
        return float(arr)
    return float(arr.ravel()[idx])


# Process each active indicator grid independently.
grid_search_results = []
successful_tests = 0
filtered_tests = 0
failed_tests = 0
failure_details = []

print("Starting batched grid search...\n")

for indicator_name, spec in active_indicator_grids.items():
    indicator_params = spec["params"]
    indicator_total = len(indicator_params)
    indicator_successful = 0
    indicator_filtered = 0
    indicator_failed = 0
    batch_count = (indicator_total + BATCH_SIZE - 1) // BATCH_SIZE

    print("=" * 70)
    print(f"Running {indicator_name} grid: {indicator_total} combinations")
    print("=" * 70)

    for batch_start in range(0, indicator_total, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, indicator_total)
        batch_params = indicator_params[batch_start:batch_end]
        batch_size = len(batch_params)

        print(f"Processing {indicator_name} batch {batch_start // BATCH_SIZE + 1}/{batch_count}")
        print(f"  Combinations {batch_start + 1} to {batch_end}...")

        batch_entries = []
        batch_exits = []
        signal_ok = []

        for params in batch_params:
            try:
                entries, exits = spec["build_signals"](train_close, params, shift_signals=True)
                batch_entries.append(entries)
                batch_exits.append(exits)
                signal_ok.append(True)
            except Exception as exc:
                batch_entries.append(pd.Series(False, index=train_close.index, dtype=bool))
                batch_exits.append(pd.Series(False, index=train_close.index, dtype=bool))
                signal_ok.append(False)
                indicator_failed += 1
                failed_tests += 1
                if len(failure_details) < 10:
                    failure_details.append(f"{indicator_name} signal build failed for {spec['format_params'](params)}: {exc}")

        entries_df = pd.DataFrame(batch_entries).T
        exits_df = pd.DataFrame(batch_exits).T

        print(f"  Running vectorized backtest on {batch_size} {indicator_name} combinations...")
        try:
            portfolios = vbt.Portfolio.from_signals(
                close=train_close,
                entries=entries_df,
                exits=exits_df,
                init_cash=100_000,
                fees=0.0005,
                slippage=0.0005,
                freq=FREQ
            )

            total_returns = portfolios.total_return()
            annualized_returns = portfolios.annualized_return(freq=FREQ, year_freq=YEAR_FREQ)
            max_drawdowns = portfolios.max_drawdown()
            volatilities = portfolios.annualized_volatility(freq=FREQ, year_freq=YEAR_FREQ)
            sharpe_ratios = portfolios.sharpe_ratio(freq=FREQ, year_freq=YEAR_FREQ)
            sortino_ratios = portfolios.sortino_ratio(freq=FREQ, year_freq=YEAR_FREQ)
            portfolio_returns = portfolios.returns()

            for idx, params in enumerate(batch_params):
                if not signal_ok[idx]:
                    continue

                try:
                    total_return = metric_scalar(total_returns, idx)
                    annualized_return = metric_scalar(annualized_returns, idx)
                    max_drawdown = metric_scalar(max_drawdowns, idx)
                    volatility = metric_scalar(volatilities, idx)
                    sharpe_ratio = metric_scalar(sharpe_ratios, idx)
                    sortino_ratio = metric_scalar(sortino_ratios, idx)

                    trades = portfolios.trades if batch_size == 1 else portfolios[idx].trades
                    total_trades = len(trades)
                    trades_per_year = total_trades / train_years

                    if trades_per_year < MIN_TRADES_PER_YEAR:
                        filtered_tests += 1
                        indicator_filtered += 1
                        continue

                    win_rate_pct = np.nan
                    profit_factor = np.nan
                    expectancy = 0.0
                    avg_win_amount = 0.0
                    avg_loss_amount = 0.0
                    winning_streak = np.nan
                    losing_streak = np.nan

                    if total_trades > 0:
                        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
                        if tr.size > 0:
                            pos = tr[tr > 0]
                            neg = tr[tr < 0]
                            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                            gains = pos.sum() if len(pos) else 0.0
                            losses = abs(neg.sum()) if len(neg) else 0.0
                            profit_factor = gains / losses if losses > 0 else np.inf
                            expectancy = float(tr.mean())
                            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
                            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0

                            try:
                                winning_streak = int(trades.winning_streak[idx] if batch_size > 1 else trades.winning_streak())
                                losing_streak = int(trades.losing_streak[idx] if batch_size > 1 else trades.losing_streak())
                            except Exception:
                                pass

                    returns = portfolio_returns.iloc[:, idx] if batch_size > 1 else portfolio_returns
                    cum = (1 + returns).cumprod()
                    peak = cum.cummax()
                    dd = (cum - peak) / peak
                    ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
                    payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount != 0.0 else np.inf

                    result = {
                        "indicator": indicator_name,
                        "strategy": spec["description"],
                        "params": params.copy(),
                        "param_display": spec["format_params"](params),
                        "total_return": total_return,
                        "annualized_return": annualized_return,
                        "total_profit": total_return * 100_000,
                        "max_drawdown": max_drawdown,
                        "volatility": volatility,
                        "sharpe_ratio": sharpe_ratio,
                        "sortino_ratio": sortino_ratio,
                        "calmar_ratio": np.nan,
                        "omega_ratio": np.nan,
                        "information_ratio": np.nan,
                        "tail_ratio": np.nan,
                        "deflated_sharpe_ratio": np.nan,
                        "ulcer_index": ulcer_index,
                        "total_trades": total_trades,
                        "win_rate": win_rate_pct,
                        "profit_factor": profit_factor,
                        "expectancy": expectancy,
                        "avg_win_amount": avg_win_amount,
                        "avg_loss_amount": avg_loss_amount,
                        "payoff_ratio": payoff_ratio,
                        "winning_streak": winning_streak,
                        "losing_streak": losing_streak,
                        "trades_per_year": trades_per_year
                    }
                    result.update(params)
                    grid_search_results.append(result)
                    successful_tests += 1
                    indicator_successful += 1

                except Exception as exc:
                    failed_tests += 1
                    indicator_failed += 1
                    if len(failure_details) < 10:
                        failure_details.append(f"{indicator_name} metric extraction failed for {spec['format_params'](params)}: {exc}")

        except Exception as exc:
            print(f"  Batch failed: {str(exc)[:100]}")
            backtestable_count = sum(signal_ok)
            failed_tests += backtestable_count
            indicator_failed += backtestable_count
            if len(failure_details) < 10:
                failure_details.append(f"{indicator_name} batch backtest failed: {exc}")

        progress_pct = (batch_end / indicator_total) * 100 if indicator_total else 100.0
        print(
            f"  {indicator_name} batch complete: "
            f"{indicator_successful} stored, {indicator_filtered} filtered, {indicator_failed} failed"
        )
        print(f"  {indicator_name} progress: {batch_end}/{indicator_total} ({progress_pct:.1f}%)\n")

# SUMMARY
print("=" * 70)
print("MULTI-INDICATOR GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Stored after filters: {successful_tests}")
print(f"Filtered by trade frequency: {filtered_tests}")
print(f"Failed: {failed_tests}")
if total_combinations > 0:
    print(f"Stored-result rate: {(successful_tests / total_combinations) * 100:.1f}%")
print(f"\nResults stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if failure_details:
    print("\nFirst failure details:")
    for detail in failure_details[:5]:
        print(f"  - {detail}")

def print_top_in_sample_by_indicator(results, top_n=5, include_summary=False):
    required_columns = {
        'indicator',
        'param_display',
        'sharpe_ratio',
        'total_return',
        'annualized_return',
        'max_drawdown',
        'win_rate',
        'profit_factor',
        'total_trades',
        'trades_per_year'
    }
    missing_columns = sorted(required_columns - set(results.columns))
    if missing_columns:
        raise KeyError(f"Missing required result columns: {missing_columns}")

    if results.empty:
        print("No in-sample results to display.")
        return

    print("\n" + "=" * 70)
    print("TOP 5 IN-SAMPLE RESULTS BY INDICATOR")
    print("=" * 70)

    for indicator_name, indicator_results in results.groupby('indicator', sort=False):
        ranked = displayed_is_top_n(indicator_results, top_n)
        print(f"\n{indicator_name} - Top {top_n} by In-Sample Sharpe Ratio")
        print("-" * 70)

        if include_summary:
            print("In-sample summary:")
            print(f"   Stored combinations: {len(indicator_results)}")
            print(f"   Best Sharpe Ratio: {indicator_results['sharpe_ratio'].max():.3f}")
            print(f"   Best Total Return: {indicator_results['total_return'].max():.2%}")
            print(f"   Average Total Return: {indicator_results['total_return'].mean():.2%}")
            print(f"   Smallest Max Drawdown: {indicator_results['max_drawdown'].max():.2%}")
            print(f"   Best Profit Factor: {indicator_results['profit_factor'].max():.2f}")

        if ranked.empty:
            print("No valid Sharpe ratios for this indicator.")
            continue

        for rank, (_, row) in enumerate(ranked.iterrows(), 1):
            print(f"\n#{rank} - {row['param_display']}")
            print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
            print(f"   Total Return:      {row['total_return']:.2%}")
            print(f"   Annualized Return: {row['annualized_return']:.2%}")
            print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
            print(f"   Win Rate:          {row['win_rate']:.1f}%")
            print(f"   Profit Factor:     {row['profit_factor']:.2f}")
            print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")

    print("\n" + "=" * 70)

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    print_top_in_sample_by_indicator(results_df_preview, top_n=5, include_summary=False)


INITIATING MULTI-INDICATOR GRID SEARCH OPTIMIZATION
Active Indicators: Triple EMA, MACD, AROON, STC, KAMA, SUPERTREND, KALMAN, RSI, ADX, DONCHIAN, TRIX, VORTEX, ALMA
Training Period: 2018-01-02 -> 2023-03-02
Market Regime: stock (252-day)
Annualization Basis: 252 bars/year (252D)
Initial Capital: $100,000
Transaction Costs: 0.05% per trade (fees + slippage)
Optimization Metric: Sharpe Ratio (risk-adjusted returns)
Total combinations to test: 378754
Batch size: 1000
Minimum trade frequency filter: 2/year
Each indicator grid is run separately.

Starting batched grid search...

Running Triple EMA grid: 66840 combinations
Processing Triple EMA batch 1/67
  Combinations 1 to 1000...


  Running vectorized backtest on 1000 Triple EMA combinations...


  Triple EMA batch complete: 1000 stored, 0 filtered, 0 failed
  Triple EMA progress: 1000/66840 (1.5%)

Processing Triple EMA batch 2/67
  Combinations 1001 to 2000...


  Running vectorized backtest on 1000 Triple EMA combinations...


  Triple EMA batch complete: 2000 stored, 0 filtered, 0 failed
  Triple EMA progress: 2000/66840 (3.0%)

Processing Triple EMA batch 3/67
  Combinations 2001 to 3000...


...

  Running vectorized backtest on 1000 ALMA combinations...


  ALMA batch complete: 31469 stored, 5531 filtered, 0 failed
  ALMA progress: 37000/38000 (97.4%)

Processing ALMA batch 38/38
  Combinations 37001 to 38000...


  Running vectorized backtest on 1000 ALMA combinations...


  ALMA batch complete: 32077 stored, 5923 filtered, 0 failed
  ALMA progress: 38000/38000 (100.0%)

MULTI-INDICATOR GRID SEARCH COMPLETED!
Total combinations attempted: 378754
Stored after filters: 232314
Filtered by trade frequency: 146440
Failed: 0
Stored-result rate: 61.3%

Results stored in 'grid_search_results' (232314 entries)



TOP 5 IN-SAMPLE RESULTS BY INDICATOR

Triple EMA - Top 5 by In-Sample Sharpe Ratio
----------------------------------------------------------------------

#1 - EMA(5,112,150)
   Sharpe Ratio:      0.934
   Total Return:      371.37%
   Annualized Return: 35.06%
   Max Drawdown:      -35.78%
   Win Rate:          62.5%
   Profit Factor:     10.88
   Total Trades:      16 (3.1/year)

#2 - EMA(5,112,148)
   Sharpe Ratio:      0.930
   Total Return:      365.64%
   Annualized Return: 34.74%
   Max Drawdown:      -35.78%
   Win Rate:          58.8%
   Profit Factor:     10.25
   Total Trades:      17 (3.3/year)

#3 - EMA(5,112,149)
   Sharpe Ratio:      0.930
   Total Return:      365.64%
   Annualized Return: 34.74%
   Max Drawdown:      -35.78%
   Win Rate:          58.8%
   Profit Factor:     10.25
   Total Trades:      17 (3.3/year)

#4 - EMA(4,117,125)
   Sharpe Ratio:      0.928
   Total Return:      367.01%
   Annualized Return: 34.82%
   Max Drawdown:      -35.78%
   Win Rate:     

In [11]:
# Analyze Multi-Indicator Grid Search Results

# Convert results to DataFrame for analysis
results_df = pd.DataFrame(grid_search_results)

print("Grid Search Results Analysis")
print("=" * 50)
print(f"Total combinations tested after filters: {len(results_df)}")
print(f"Results shape: {results_df.shape}")

if results_df.empty:
    raise ValueError("No grid search results were produced. Run the grid search cell first.")

print("Results by indicator:")
print(results_df['indicator'].value_counts().to_string())

print_top_in_sample_by_indicator(results_df, top_n=5, include_summary=True)

print("\nAnalysis complete! Top in-sample results listed separately for each indicator.")

Grid Search Results Analysis
Total combinations tested after filters: 232314
Results shape: (232314, 54)
Results by indicator:
indicator
MACD          110789
STC            58293
ALMA           32077
Triple EMA     21455
KAMA            7600
ADX             1122
SUPERTREND       485
KALMAN           240
VORTEX            88
AROON             59
DONCHIAN          47
RSI               40
TRIX              19

TOP 5 IN-SAMPLE RESULTS BY INDICATOR

Triple EMA - Top 5 by In-Sample Sharpe Ratio
----------------------------------------------------------------------
In-sample summary:
   Stored combinations: 21455
   Best Sharpe Ratio: 0.934
   Best Total Return: 371.37%
   Average Total Return: 193.82%
   Smallest Max Drawdown: -35.78%
   Best Profit Factor: 13.15

#1 - EMA(5,112,150)
   Sharpe Ratio:      0.934
   Total Return:      371.37%
   Annualized Return: 35.06%
   Max Drawdown:      -35.78%
   Win Rate:          62.5%
   Profit Factor:     10.88
   Total Trades:      16 (3.1/year)

#

In [12]:
# TOP 5 OUT-OF-SAMPLE VALIDATION BY INDICATOR

oos_results_by_indicator = {}
oos_df_by_indicator = {}
oos_df_sorted = pd.DataFrame()  # compatibility: left empty; use oos_df_by_indicator

if "results_df" not in globals():
    raise ValueError("results_df is missing. Run the grid-search analysis cell first.")
if results_df.empty:
    print("No results to validate.")
elif "val_close" not in globals() or val_close is None or len(val_close) < 2:
    raise ValueError("val_close is missing or too short for OOS validation.")
else:
    print("=" * 90)
    print("TOP 5 OUT-OF-SAMPLE VALIDATION BY INDICATOR")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} -> {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} -> {val_close.index[-1].date()}")
    print("Each indicator is validated separately (no cross-indicator ranking).")
    print("Candidates: top 5 in-sample Sharpe per indicator (same metric as the IS table; tie notes below if applicable).")
    print("=" * 90)

    for indicator_name, indicator_results in results_df.groupby("indicator", sort=False):
        top_5_strategies = (
            indicator_results
            .dropna(subset=["sharpe_ratio"])
            .sort_values("sharpe_ratio", ascending=False)
            .head(5)
        )
        if top_5_strategies.empty:
            print(f"No valid in-sample results for {indicator_name}.")
            continue

        maybe_print_tied_is_sharpe_note(
            indicator_name,
            indicator_results,
            top_5_strategies["param_display"].tolist(),
            "OOS validation",
        )

        print("\n" + "=" * 90)
        print(f"{indicator_name} - TOP 5 IN-SAMPLE CANDIDATES ON OOS DATA")
        print("=" * 90)

        try:
            indicator_spec = get_active_indicator_spec(indicator_name)
        except KeyError as exc:
            print(f"Skipping {indicator_name}: {exc}")
            continue

        oos_results = []

        for is_rank, (_, strategy) in enumerate(top_5_strategies.iterrows(), 1):
            try:
                params = require_params_dict(strategy["params"], indicator_name)
                entries, exits = indicator_spec["build_signals"](val_close, params, shift_signals=True)
                pf_val = run_signal_backtest(val_close, entries, exits)

                oos_total_return = float(pf_val.total_return())
                oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ, year_freq=YEAR_FREQ))
                oos_max_drawdown = float(pf_val.max_drawdown())
                tstats = trade_stats_from_portfolio(pf_val)

                is_sharpe = float(strategy["sharpe_ratio"])
                is_return = float(strategy["total_return"])

                oos_results.append({
                    "IS_Rank": is_rank,
                    "Indicator": indicator_name,
                    "Strategy": strategy["param_display"],
                    "Params": params,
                    "IS_Sharpe": is_sharpe,
                    "IS_Return": is_return,
                    "IS_MaxDD": float(strategy["max_drawdown"]),
                    "IS_WinRate": float(strategy["win_rate"]),
                    "OOS_Sharpe": oos_sharpe,
                    "OOS_Return": oos_total_return,
                    "OOS_MaxDD": oos_max_drawdown,
                    "OOS_WinRate": tstats["win_rate_pct"],
                    "OOS_Trades": tstats["total_trades"],
                    "OOS_ProfitFactor": tstats["profit_factor"],
                    "Sharpe_Degradation": (
                        ((oos_sharpe - is_sharpe) / abs(is_sharpe) * 100)
                        if is_sharpe != 0 else np.nan
                    ),
                    "Return_Degradation": (
                        ((oos_total_return - is_return) / abs(is_return) * 100)
                        if is_return != 0 else np.nan
                    ),
                })
            except Exception as exc:
                print(f"  Skipping IS#{is_rank} {strategy.get('param_display', '?')}: {type(exc).__name__}: {exc}")
                continue

        if not oos_results:
            print(f"No successful OOS backtests for {indicator_name}.")
            continue

        oos_df_indicator = pd.DataFrame(oos_results)
        oos_df_indicator["OOS_Rank"] = (
            oos_df_indicator["OOS_Sharpe"]
            .rank(method="first", ascending=False)
            .astype(int)
        )
        oos_df_is_order = oos_df_indicator.sort_values("IS_Rank").reset_index(drop=True)

        oos_results_by_indicator[indicator_name] = oos_results
        oos_df_by_indicator[indicator_name] = oos_df_is_order

        print(f"\n{indicator_name} - OOS RESULTS (in-sample rank order)")
        print("-" * 90)
        for _, row in oos_df_is_order.iterrows():
            print(f"#{int(row['IS_Rank'])} - {row['Strategy']}")
            print(f"   In-Sample Sharpe:   {fmt_metric(row['IS_Sharpe'], '.3f')}")
            print(f"   OOS Sharpe Ratio:   {fmt_metric(row['OOS_Sharpe'], '.3f')}  (OOS rank #{int(row['OOS_Rank'])})")
            print(f"   OOS Return:         {fmt_metric(row['OOS_Return'], '.2%')}")
            print(f"   OOS Max Drawdown:   {fmt_metric(row['OOS_MaxDD'], '.2%')}")
            print(f"   OOS Win Rate:       {fmt_metric(row['OOS_WinRate'], '.1f', '%')}")
            print(f"   OOS Profit Factor:  {fmt_metric(row['OOS_ProfitFactor'], '.2f')}")
            print(f"   OOS Total Trades:   {int(row['OOS_Trades'])}")
            print(f"   Sharpe Degradation: {fmt_metric(row['Sharpe_Degradation'], '+.1f', '%')}\n")

        display_df = pd.DataFrame({
            "IS Rank": oos_df_is_order["IS_Rank"].astype(int),
            "OOS Rank": oos_df_is_order["OOS_Rank"].astype(int),
            "Strategy": oos_df_is_order["Strategy"],
            "IS Sharpe": oos_df_is_order["IS_Sharpe"].map(lambda x: fmt_metric(x, ".3f")),
            "OOS Sharpe": oos_df_is_order["OOS_Sharpe"].map(lambda x: fmt_metric(x, ".3f")),
            "Sharpe Delta %": oos_df_is_order["Sharpe_Degradation"].map(lambda x: fmt_metric(x, "+.1f", "%")),
            "IS Return": oos_df_is_order["IS_Return"].map(lambda x: fmt_metric(x, ".1%")),
            "OOS Return": oos_df_is_order["OOS_Return"].map(lambda x: fmt_metric(x, ".1%")),
            "Return Delta %": oos_df_is_order["Return_Degradation"].map(lambda x: fmt_metric(x, "+.1f", "%")),
            "OOS Trades": oos_df_is_order["OOS_Trades"].astype(int),
            "OOS WinRate": oos_df_is_order["OOS_WinRate"].map(lambda x: fmt_metric(x, ".1f", "%")),
            "OOS PF": oos_df_is_order["OOS_ProfitFactor"].map(lambda x: fmt_metric(x, ".2f")),
        })
        print(f"{indicator_name} - OOS COMPARISON TABLE (in-sample rank order)")
        print(display_df.to_string(index=False))

        best_oos = best_oos_row(oos_df_indicator)
        if best_oos is None:
            print(f"\nNo valid OOS Sharpe values for {indicator_name}.")
            continue

        print("\n" + "-" * 90)
        print(f"BEST OOS AMONG TOP-5 IS CANDIDATES: {indicator_name}")
        print("-" * 90)
        print(f"Strategy:              {best_oos['Strategy']}")
        print(f"In-Sample Rank:        #{int(best_oos['IS_Rank'])}")
        print(f"Out-of-Sample Rank:    #{int(best_oos['OOS_Rank'])}")
        print(f"OOS Sharpe Ratio:      {fmt_metric(best_oos['OOS_Sharpe'], '.3f')}")
        print(f"OOS Return:            {fmt_metric(best_oos['OOS_Return'], '.2%')}")
        print(f"OOS Max Drawdown:      {fmt_metric(best_oos['OOS_MaxDD'], '.2%')}")
        print(f"OOS Win Rate:          {fmt_metric(best_oos['OOS_WinRate'], '.1f', '%')}")
        print(f"OOS Profit Factor:     {fmt_metric(best_oos['OOS_ProfitFactor'], '.2f')}")
        print(f"OOS Total Trades:      {int(best_oos['OOS_Trades'])}")
        print(f"Sharpe Degradation:    {fmt_metric(best_oos['Sharpe_Degradation'], '+.1f', '%')}")

    print("\n" + "=" * 90)
    print("OOS validation complete for each indicator separately.")
    print("=" * 90)


TOP 5 OUT-OF-SAMPLE VALIDATION BY INDICATOR
Training Period: 2018-01-02 -> 2023-03-02
Validation Period: 2023-03-03 -> 2026-08-17
Each indicator is validated separately (no cross-indicator ranking).
Candidates: top 5 in-sample Sharpe per indicator (same metric as the IS table; tie notes below if applicable).

Triple EMA - TOP 5 IN-SAMPLE CANDIDATES ON OOS DATA



Triple EMA - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - EMA(5,112,150)
   In-Sample Sharpe:   0.934
   OOS Sharpe Ratio:   0.562  (OOS rank #4)
   OOS Return:         65.84%
   OOS Max Drawdown:   -44.64%
   OOS Win Rate:       40.0%
   OOS Profit Factor:  2.68
   OOS Total Trades:   10
   Sharpe Degradation: -39.8%

#2 - EMA(5,112,148)
   In-Sample Sharpe:   0.930
   OOS Sharpe Ratio:   0.585  (OOS rank #3)
   OOS Return:         71.35%
   OOS Max Drawdown:   -44.64%
   OOS Win Rate:       40.0%
   OOS Profit Factor:  2.93
   OOS Total Trades:   10
   Sharpe Degradation: -37.1%

#3 - EMA(5,112,149)
   In-Sample Sharpe:   0.930
   OOS Sharpe Ratio:   0.562  (OOS rank #5)
   OOS Return:         65.84%
   OOS Max Drawdown:   -44.64%
   OOS Win Rate:       40.0%
   OOS Profit Factor:  2.68
   OOS Total Trades:   10
   Sharpe Degradation: -39.6%

#4 - EMA(4,117,125)
   In-Sample Sharpe:   0.928
   OOS 


AROON - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - AROON(22)
   In-Sample Sharpe:   0.796
   OOS Sharpe Ratio:   0.868  (OOS rank #3)
   OOS Return:         130.46%
   OOS Max Drawdown:   -34.68%
   OOS Win Rate:       50.0%
   OOS Profit Factor:  3.31
   OOS Total Trades:   16
   Sharpe Degradation: +9.1%

#2 - AROON(18)
   In-Sample Sharpe:   0.790
   OOS Sharpe Ratio:   0.816  (OOS rank #4)
   OOS Return:         125.79%
   OOS Max Drawdown:   -35.83%
   OOS Win Rate:       43.5%
   OOS Profit Factor:  2.10
   OOS Total Trades:   23
   Sharpe Degradation: +3.3%

#3 - AROON(21)
   In-Sample Sharpe:   0.722
   OOS Sharpe Ratio:   0.986  (OOS rank #1)
   OOS Return:         181.09%
   OOS Max Drawdown:   -32.31%
   OOS Win Rate:       47.1%
   OOS Profit Factor:  3.45
   OOS Total Trades:   17
   Sharpe Degradation: +36.4%

#4 - AROON(23)
   In-Sample Sharpe:   0.706
   OOS Sharpe Ratio:   0.922  (


SUPERTREND - TOP 5 IN-SAMPLE CANDIDATES ON OOS DATA

SUPERTREND - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - SUPERTREND(ATR=88,Factor=3)
   In-Sample Sharpe:   0.852
   OOS Sharpe Ratio:   1.059  (OOS rank #3)
   OOS Return:         191.21%
   OOS Max Drawdown:   -26.28%
   OOS Win Rate:       42.9%
   OOS Profit Factor:  3.25
   OOS Total Trades:   14
   Sharpe Degradation: +24.3%

#2 - SUPERTREND(ATR=87,Factor=3)
   In-Sample Sharpe:   0.814
   OOS Sharpe Ratio:   1.059  (OOS rank #4)
   OOS Return:         191.21%
   OOS Max Drawdown:   -26.28%
   OOS Win Rate:       42.9%
   OOS Profit Factor:  3.25
   OOS Total Trades:   14
   Sharpe Degradation: +30.1%

#3 - SUPERTREND(ATR=89,Factor=3)
   In-Sample Sharpe:   0.808
   OOS Sharpe Ratio:   1.064  (OOS rank #1)
   OOS Return:         193.00%
   OOS Max Drawdown:   -26.28%
   OOS Win Rate:       42.9%
   OOS Profit Factor:  3.27
   OOS Total Trade


RSI - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - RSI(71)
   In-Sample Sharpe:   0.781
   OOS Sharpe Ratio:   0.781  (OOS rank #5)
   OOS Return:         129.34%
   OOS Max Drawdown:   -39.73%
   OOS Win Rate:       46.2%
   OOS Profit Factor:  4.05
   OOS Total Trades:   13
   Sharpe Degradation: -0.1%

#2 - RSI(72)
   In-Sample Sharpe:   0.779
   OOS Sharpe Ratio:   0.853  (OOS rank #2)
   OOS Return:         155.94%
   OOS Max Drawdown:   -39.73%
   OOS Win Rate:       50.0%
   OOS Profit Factor:  5.57
   OOS Total Trades:   12
   Sharpe Degradation: +9.5%

#3 - RSI(73)
   In-Sample Sharpe:   0.779
   OOS Sharpe Ratio:   0.855  (OOS rank #1)
   OOS Return:         156.86%
   OOS Max Drawdown:   -39.73%
   OOS Win Rate:       53.8%
   OOS Profit Factor:  5.63
   OOS Total Trades:   13
   Sharpe Degradation: +9.8%

#4 - RSI(74)
   In-Sample Sharpe:   0.769
   OOS Sharpe Ratio:   0.808  (OOS rank #3


TRIX - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - TRIX(6)
   In-Sample Sharpe:   0.769
   OOS Sharpe Ratio:   1.066  (OOS rank #2)
   OOS Return:         212.24%
   OOS Max Drawdown:   -32.21%
   OOS Win Rate:       44.0%
   OOS Profit Factor:  2.48
   OOS Total Trades:   25
   Sharpe Degradation: +38.7%

#2 - TRIX(5)
   In-Sample Sharpe:   0.724
   OOS Sharpe Ratio:   0.950  (OOS rank #4)
   OOS Return:         166.31%
   OOS Max Drawdown:   -38.96%
   OOS Win Rate:       37.5%
   OOS Profit Factor:  2.19
   OOS Total Trades:   32
   Sharpe Degradation: +31.2%

#3 - TRIX(7)
   In-Sample Sharpe:   0.711
   OOS Sharpe Ratio:   1.169  (OOS rank #1)
   OOS Return:         262.23%
   OOS Max Drawdown:   -35.98%
   OOS Win Rate:       47.6%
   OOS Profit Factor:  3.01
   OOS Total Trades:   21
   Sharpe Degradation: +64.4%

#4 - TRIX(8)
   In-Sample Sharpe:   0.679
   OOS Sharpe Ratio:   1.037  (OOS ran


ALMA - OOS RESULTS (in-sample rank order)
------------------------------------------------------------------------------------------
#1 - ALMA(2,99,0.10,3)
   In-Sample Sharpe:   0.786
   OOS Sharpe Ratio:   0.665  (OOS rank #1)
   OOS Return:         92.38%
   OOS Max Drawdown:   -43.11%
   OOS Win Rate:       50.0%
   OOS Profit Factor:  2.54
   OOS Total Trades:   10
   Sharpe Degradation: -15.3%

#2 - ALMA(2,98,0.18,7)
   In-Sample Sharpe:   0.767
   OOS Sharpe Ratio:   0.589  (OOS rank #4)
   OOS Return:         72.72%
   OOS Max Drawdown:   -46.71%
   OOS Win Rate:       58.3%
   OOS Profit Factor:  2.08
   OOS Total Trades:   12
   Sharpe Degradation: -23.2%

#3 - ALMA(2,97,0.18,7)
   In-Sample Sharpe:   0.754
   OOS Sharpe Ratio:   0.558  (OOS rank #5)
   OOS Return:         65.16%
   OOS Max Drawdown:   -47.76%
   OOS Win Rate:       46.7%
   OOS Profit Factor:  1.78
   OOS Total Trades:   15
   Sharpe Degradation: -25.9%

#4 - ALMA(2,99,0.12,3)
   In-Sample Sharpe:   0.752
 

In [13]:
# FULL-SAMPLE EVALUATION BY INDICATOR

if "results_df" not in globals() or results_df.empty:
    raise ValueError("results_df is empty or missing. Run the grid search first.")

# Reuse the already-prepared full Close series (avoid re-parsing MultiIndex columns).
if "close" in globals() and close is not None and len(close) >= 2:
    full_close = close.astype(float).copy()
    full_close.name = TICKER
else:
    full_close = select_close_series(stock_data, TICKER).astype(float)
    full_close.name = TICKER

if len(full_close) < 2:
    raise ValueError("full_close is too short for full-sample evaluation.")

years = sample_years(full_close)
bh_total_return = (full_close.iloc[-1] / full_close.iloc[0]) - 1.0
bh_annualized = (1.0 + bh_total_return) ** (1.0 / years) - 1.0 if years > 0 else np.nan

full_sample_results_by_indicator = {}

print("=" * 90)
print("FULL-SAMPLE EVALUATION BY INDICATOR")
print("=" * 90)
print(f"Period: {full_close.index[0].date()} -> {full_close.index[-1].date()}")
print("Selection rule: OOS-best among top-5 IS when OOS results exist; else best IS Sharpe.")
print("=" * 90)

for indicator_name, indicator_results in results_df.groupby("indicator", sort=False):
    selection_source = "is"
    selected_params = None
    param_display = None
    is_sharpe_ref = np.nan
    oos_sharpe_ref = np.nan

    oos_df_ind = None
    if "oos_df_by_indicator" in globals():
        oos_df_ind = oos_df_by_indicator.get(indicator_name)

    if oos_df_ind is not None and not oos_df_ind.empty:
        best_oos = best_oos_row(oos_df_ind)
        if best_oos is None:
            print(f"\n{indicator_name}: no valid OOS Sharpe among top-5 IS candidates; falling back to best IS.")
        else:
            selected_params = require_params_dict(best_oos["Params"], indicator_name)
            param_display = best_oos["Strategy"]
            is_sharpe_ref = float(best_oos["IS_Sharpe"])
            oos_sharpe_ref = float(best_oos["OOS_Sharpe"])
            selection_source = "oos"
            maybe_print_tied_is_sharpe_note(
                indicator_name,
                indicator_results,
                [param_display],
                "full-sample selection",
            )

    if selected_params is None:
        best = best_is_row(indicator_results)
        if best is None:
            print(f"\n{indicator_name}: no valid in-sample results. Skipping.")
            continue
        selected_params = require_params_dict(best["params"], indicator_name)
        param_display = best["param_display"]
        is_sharpe_ref = float(best["sharpe_ratio"])
        selection_source = "is"

    try:
        best_spec = get_active_indicator_spec(indicator_name)
    except KeyError as exc:
        print(f"\n{indicator_name}: skipping full-sample ({exc})")
        continue

    try:
        entries_series, exits_series = best_spec["build_signals"](
            full_close, selected_params, shift_signals=True
        )
        pf_full = run_signal_backtest(full_close, entries_series, exits_series)
    except Exception as exc:
        print(f"\n{indicator_name}: full-sample backtest failed ({type(exc).__name__}: {exc})")
        continue

    full_total_return = float(pf_full.total_return())
    full_annualized_return = float(pf_full.annualized_return(freq=FREQ, year_freq=YEAR_FREQ))
    full_sharpe = float(pf_full.sharpe_ratio(freq=FREQ, year_freq=YEAR_FREQ))
    full_sortino = float(pf_full.sortino_ratio(freq=FREQ, year_freq=YEAR_FREQ))
    full_max_dd = float(pf_full.max_drawdown())
    full_vol = float(pf_full.annualized_volatility(freq=FREQ, year_freq=YEAR_FREQ))

    tstats = trade_stats_from_portfolio(pf_full)
    full_total_trades = tstats["total_trades"]
    full_trades_per_year = full_total_trades / years if years > 0 else np.nan

    full_sample_results_by_indicator[indicator_name] = {
        "params": selected_params,
        "param_display": param_display,
        "selection_source": selection_source,
        "is_sharpe": is_sharpe_ref,
        "oos_sharpe": oos_sharpe_ref,
        "total_return": full_total_return,
        "annualized_return": full_annualized_return,
        "sharpe": full_sharpe,
        "sortino": full_sortino,
        "max_drawdown": full_max_dd,
        "volatility": full_vol,
        "total_trades": full_total_trades,
        "trades_per_year": full_trades_per_year,
        "win_rate": tstats["win_rate_pct"],
        "profit_factor": tstats["profit_factor"],
        "expectancy": tstats["expectancy"],
        "avg_win_amount": tstats["avg_win"],
        "avg_loss_amount": tstats["avg_loss"],
        "portfolio": pf_full,
    }

    print("\n" + "=" * 90)
    print(f"FULL-SAMPLE: {indicator_name} {param_display}")
    print("=" * 90)
    print(f"Selection source: {selection_source}")
    print(f"In-Sample Sharpe (reference): {fmt_metric(is_sharpe_ref, '.3f')}")
    if selection_source == "oos":
        print(f"OOS Sharpe (selection): {fmt_metric(oos_sharpe_ref, '.3f')}")
    print("RETURN METRICS:")
    print(f"{'Total Return':<25} {fmt_metric(full_total_return, '.2%'):>10}")
    print(f"{'Annualized Return':<25} {fmt_metric(full_annualized_return, '.2%'):>10}")
    print("RISK-ADJUSTED:")
    print(f"{'Sharpe Ratio':<25} {fmt_metric(full_sharpe, '.3f'):>10}")
    print(f"{'Sortino Ratio':<25} {fmt_metric(full_sortino, '.3f'):>10}")
    print("RISK:")
    print(f"{'Max Drawdown':<25} {fmt_metric(full_max_dd, '.2%'):>10}")
    print(f"{'Volatility':<25} {fmt_metric(full_vol, '.2%'):>10}")
    print("TRADES:")
    print(f"{'Total Trades':<25} {full_total_trades:>10.0f}")
    print(f"{'Trades per Year':<25} {fmt_metric(full_trades_per_year, '.1f'):>10}")
    print(f"{'Win Rate %':<25} {fmt_metric(tstats['win_rate_pct'], '.1f'):>10}")
    print(f"{'Profit Factor':<25} {fmt_metric(tstats['profit_factor'], '.2f'):>10}")
    print(f"{'Expectancy':<25} {fmt_metric(tstats['expectancy'], '.4f'):>10}")
    print(f"{'Avg Win Amount':<25} {fmt_metric(tstats['avg_win'], '.4f'):>10}")
    print(f"{'Avg Loss Amount':<25} {fmt_metric(tstats['avg_loss'], '.4f'):>10}")
    print("BENCHMARK (BUY & HOLD):")
    print(f"{'Total Return':<25} {fmt_metric(bh_total_return, '.2%'):>10}")
    print(f"{'Annualized Return':<25} {fmt_metric(bh_annualized, '.2%'):>10}")

print("\n" + "=" * 90)
print("Full-sample evaluation complete for each indicator separately.")
print("=" * 90)


FULL-SAMPLE EVALUATION BY INDICATOR
Period: 2018-01-02 -> 2026-08-17
Selection rule: OOS-best among top-5 IS when OOS results exist; else best IS Sharpe.

FULL-SAMPLE: Triple EMA EMA(4,117,125)
Selection source: oos
In-Sample Sharpe (reference): 0.928
OOS Sharpe (selection): 0.653
RETURN METRICS:
Total Return                1156.09%
Annualized Return             34.22%
RISK-ADJUSTED:
Sharpe Ratio                   0.895
Sortino Ratio                  1.254
RISK:
Max Drawdown                 -44.64%
Volatility                    43.63%
TRADES:
Total Trades                      26
Trades per Year                  3.0
Win Rate %                      57.7
Profit Factor                   6.27
Expectancy                       n/a
Avg Win Amount                0.2584
Avg Loss Amount               0.0618
BENCHMARK (BUY & HOLD):
Total Return                     n/a
Annualized Return                n/a

FULL-SAMPLE: MACD MACD(12,69,78)
Selection source: oos
In-Sample Sharpe (reference): 0.724
OO


FULL-SAMPLE: SUPERTREND SUPERTREND(ATR=89,Factor=3)
Selection source: oos
In-Sample Sharpe (reference): 0.808
OOS Sharpe (selection): 1.064
RETURN METRICS:
Total Return                2227.68%
Annualized Return             44.20%
RISK-ADJUSTED:
Sharpe Ratio                   1.083
Sortino Ratio                  1.560
RISK:
Max Drawdown                 -51.22%
Volatility                    42.05%
TRADES:
Total Trades                      36
Trades per Year                  4.2
Win Rate %                      44.4
Profit Factor                   3.52
Expectancy                       n/a
Avg Win Amount                0.3922
Avg Loss Amount               0.0938
BENCHMARK (BUY & HOLD):
Total Return                     n/a
Annualized Return                n/a

FULL-SAMPLE: KALMAN KALMAN(Q=1,R=159)
Selection source: oos
In-Sample Sharpe (reference): 0.851
OOS Sharpe (selection): 1.315
RETURN METRICS:
Total Return                1578.21%
Annualized Return             38.82%
RISK-ADJUSTED:
Sha

In [14]:
# SAVE TOP IS / OOS / FULL-SAMPLE RESULTS BY INDICATOR

import os
import re


def _to_builtin(value):
    """Convert numpy scalars / nested containers to plain Python builtins."""
    if isinstance(value, dict):
        return {str(key): _to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_builtin(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def _stringify_mapping_cell(value):
    """Serialize dict-like param cells for json/parquet export (literal_eval-safe)."""
    if isinstance(value, dict):
        return str(_to_builtin(value))
    return value


def _require_positive_int(name, value):
    try:
        number = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be an integer >= 1; got {value!r}") from exc
    if number < 1:
        raise ValueError(f"{name} must be an integer >= 1; got {value!r}")
    return number


def _write_results_dataframe(export_df, filepath):
    """Write a results DataFrame to parquet or json based on filepath suffix."""
    if filepath is None or not str(filepath).strip():
        raise ValueError("filepath must be a non-empty path ending in .json or .parquet")

    path = str(filepath).strip()
    suffix = path.lower().rsplit(".", 1)[-1] if "." in path else ""
    if suffix not in {"parquet", "json"}:
        raise ValueError(f"Unsupported file format (use .parquet or .json): {filepath!r}")

    parent = os.path.dirname(path)
    if parent and not os.path.isdir(parent):
        raise FileNotFoundError(
            f"Parent directory does not exist for filepath={path!r}: {parent}"
        )

    try:
        if suffix == "parquet":
            export_df.to_parquet(path, index=False)
        else:
            # Ensure strict JSON: map +/-inf to null via NaN (version-safe).
            json_df = export_df.replace([np.inf, -np.inf], np.nan)
            json_df.to_json(path, orient="records", indent=2)
    except ImportError as exc:
        hint = "pip install pyarrow" if suffix == "parquet" else "pip install pandas"
        raise ImportError(
            f"Writing {suffix!r} requires a dependency that is not installed "
            f"({hint}). Original error: {exc}"
        ) from exc
    except ValueError as exc:
        raise ValueError(f"Failed to serialize results to {path!r}: {exc}") from exc
    except PermissionError as exc:
        raise PermissionError(
            f"Permission denied writing {path!r}. Close the file if it is open elsewhere "
            f"(e.g. Excel) and retry. Original error: {exc}"
        ) from exc
    except OSError as exc:
        raise OSError(f"Failed to write results to {path!r}: {exc}") from exc
    return path


def _collect_top_in_sample_by_indicator(results, top_n=5):
    """Top-N in-sample rows per indicator with normalized identity columns."""
    if not isinstance(results, pd.DataFrame):
        raise TypeError(f"results must be a pandas DataFrame; got {type(results).__name__}")
    if "displayed_is_top_n" not in globals():
        raise NameError("displayed_is_top_n is missing. Run the indicator-grid helper cell first.")

    top_n = _require_positive_int("top_n", top_n)
    required_columns = {
        "indicator",
        "params",
        "param_display",
        "sharpe_ratio",
        "total_return",
        "annualized_return",
        "max_drawdown",
        "win_rate",
        "profit_factor",
        "total_trades",
        "trades_per_year",
    }
    missing_columns = sorted(required_columns - set(results.columns))
    if missing_columns:
        raise KeyError(f"Missing required result columns: {missing_columns}")

    if results.empty:
        return None

    top_frames = []
    for indicator_name, indicator_results in results.groupby("indicator", sort=False):
        ranked = displayed_is_top_n(indicator_results, top_n)
        if ranked.empty:
            continue
        ranked = ranked.copy()
        ranked["rank"] = range(1, len(ranked) + 1)
        ranked["indicator"] = indicator_name
        ranked["params"] = ranked["params"].map(_stringify_mapping_cell)
        top_frames.append(ranked)

    if not top_frames:
        raise ValueError(
            "No in-sample rows to export: every indicator lacked a finite sharpe_ratio "
            "for displayed_is_top_n. Re-run the grid search cell."
        )

    return pd.concat(top_frames, ignore_index=True)


def _collect_top_oos_by_indicator(oos_df_by_indicator):
    """Top OOS rows per indicator with normalized identity columns."""
    if oos_df_by_indicator is None:
        return None
    if not isinstance(oos_df_by_indicator, dict):
        raise TypeError(
            "oos_df_by_indicator must be a dict of DataFrames; "
            f"got {type(oos_df_by_indicator).__name__}"
        )
    if not oos_df_by_indicator:
        return None

    required_after_rename = {"param_display", "params", "OOS_Sharpe"}
    frames = []
    for indicator_name, oos_df in oos_df_by_indicator.items():
        if oos_df is None:
            continue
        if not isinstance(oos_df, pd.DataFrame):
            raise TypeError(
                f"oos_df_by_indicator[{indicator_name!r}] must be a DataFrame; "
                f"got {type(oos_df).__name__}"
            )
        if oos_df.empty:
            continue

        frame = oos_df.copy()
        rename_map = {
            "Indicator": "indicator",
            "Strategy": "param_display",
            "Params": "params",
            "IS_Rank": "rank",
        }
        frame = frame.rename(columns={k: v for k, v in rename_map.items() if k in frame.columns})
        missing = sorted(required_after_rename - set(frame.columns))
        if missing:
            raise KeyError(
                f"oos_df_by_indicator[{indicator_name!r}] missing required columns "
                f"after normalize: {missing}"
            )
        frame["indicator"] = indicator_name
        frame["params"] = frame["params"].map(_stringify_mapping_cell)
        frames.append(frame)

    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


def _collect_full_sample_by_indicator(full_sample_results_by_indicator):
    """Full-sample rows per indicator with normalized identity columns."""
    if full_sample_results_by_indicator is None:
        return None
    if not isinstance(full_sample_results_by_indicator, dict):
        raise TypeError(
            "full_sample_results_by_indicator must be a dict; "
            f"got {type(full_sample_results_by_indicator).__name__}"
        )
    if not full_sample_results_by_indicator:
        return None

    required_keys = {"params", "param_display", "sharpe"}
    rows = []
    for indicator_name, info in full_sample_results_by_indicator.items():
        if not isinstance(info, dict):
            raise TypeError(
                f"full_sample_results_by_indicator[{indicator_name!r}] must be a dict; "
                f"got {type(info).__name__}"
            )
        missing_keys = sorted(required_keys - set(info))
        if missing_keys:
            raise KeyError(
                f"full_sample_results_by_indicator[{indicator_name!r}] "
                f"missing required keys: {missing_keys}"
            )
        row = {"indicator": indicator_name}
        for key, value in info.items():
            if key == "portfolio":
                continue
            row[key] = _stringify_mapping_cell(value)
        rows.append(row)

    if not rows:
        return None
    return pd.DataFrame(rows)


def save_notebook_results_by_indicator(
    results,
    oos_df_by_indicator,
    full_sample_results_by_indicator,
    filepath,
    top_n=5,
    require_all_stages=True,
):
    """Save top IS, top OOS, and full-sample results per indicator in one file.

    Rows are stacked with a leading `stage` column: 'is', 'oos', or 'full'.
    Identity columns are normalized to: indicator, rank, param_display, params.
    """
    top_n = _require_positive_int("top_n", top_n)
    stage_frames = {}

    is_df = _collect_top_in_sample_by_indicator(results, top_n=top_n)
    if is_df is not None and not is_df.empty:
        stage_frames["is"] = is_df.assign(stage="is")

    oos_df = _collect_top_oos_by_indicator(oos_df_by_indicator)
    if oos_df is not None and not oos_df.empty:
        stage_frames["oos"] = oos_df.assign(stage="oos")

    full_df = _collect_full_sample_by_indicator(full_sample_results_by_indicator)
    if full_df is not None and not full_df.empty:
        stage_frames["full"] = full_df.assign(stage="full")

    missing_stages = [name for name in ("is", "oos", "full") if name not in stage_frames]
    if missing_stages:
        message = f"Missing result stages for export: {missing_stages}"
        if require_all_stages or not stage_frames:
            raise ValueError(message)
        print(f"Warning: {message}")

    if not stage_frames:
        print("No notebook results to save.")
        return None

    # Put stage first, then shared identity columns when present.
    frames = []
    for stage_name in ("is", "oos", "full"):
        if stage_name not in stage_frames:
            continue
        frame = stage_frames[stage_name]
        leading = [
            col for col in ("stage", "indicator", "rank", "param_display", "params")
            if col in frame.columns
        ]
        ordered = leading + [col for col in frame.columns if col not in leading]
        frames.append(frame.loc[:, ordered])

    export_df = pd.concat(frames, ignore_index=True, sort=False)
    path = _write_results_dataframe(export_df, filepath)
    stage_counts = export_df["stage"].value_counts().to_dict()
    print(
        f"Saved notebook results by indicator to {path} "
        f"({len(export_df)} rows; stages={stage_counts})"
    )
    return export_df


_WINDOWS_RESERVED_NAMES = {
    "con", "prn", "aux", "nul",
    "com1", "com2", "com3", "com4", "com5", "com6", "com7", "com8", "com9",
    "lpt1", "lpt2", "lpt3", "lpt4", "lpt5", "lpt6", "lpt7", "lpt8", "lpt9",
}


def _ticker_results_slug(ticker):
    """Normalize TICKER into a safe filesystem slug for results filenames."""
    if ticker is None:
        raise ValueError("TICKER is missing or empty. Run the download cell first.")
    if not isinstance(ticker, str):
        ticker = str(ticker)
    text = ticker.strip()
    if not text:
        raise ValueError("TICKER is missing or empty. Run the download cell first.")

    slug = re.sub(r"[^a-z0-9]+", "_", text.lower())
    slug = re.sub(r"_+", "_", slug).strip("_")
    if not slug:
        raise ValueError(f"Could not build results filename from TICKER={ticker!r}")
    if slug in _WINDOWS_RESERVED_NAMES:
        slug = f"asset_{slug}"
    return slug


def _indicator_sweep_results_dir():
    """Return Indicator_sweep_results, creating it if it does not exist yet."""
    search = os.path.abspath(os.getcwd())
    for _ in range(6):
        candidate = os.path.join(search, "Indicator_sweep_results")
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(search)
        if parent == search:
            break
        search = parent

    cwd = os.path.abspath(os.getcwd())
    if os.path.basename(cwd).lower() == "indicator_sweeps":
        results_dir = os.path.join(os.path.dirname(cwd), "Indicator_sweep_results")
    else:
        results_dir = os.path.join(cwd, "Indicator_sweep_results")
    try:
        os.makedirs(results_dir, exist_ok=True)
    except OSError as exc:
        raise OSError(
            f"Could not create results directory {results_dir!r}: {exc}"
        ) from exc
    return results_dir


def _results_filepath_for_ticker(ticker, file_format="json"):
    """Build the results path for the asset under test.

    Examples:
      BTC-USD -> Indicator_sweep_results/btc_usd_indicator_sweep_results.json
      QQQ     -> Indicator_sweep_results/qqq_indicator_sweep_results.json

    The results file itself is created/overwritten when results are written.
    """
    if not isinstance(file_format, str) or not file_format.strip():
        raise ValueError(f"file_format must be 'json' or 'parquet'; got {file_format!r}")
    fmt = file_format.strip().lower().lstrip(".")
    if fmt not in {"json", "parquet"}:
        raise ValueError(f"file_format must be 'json' or 'parquet'; got {file_format!r}")

    slug = _ticker_results_slug(ticker)
    filename = f"{slug}_indicator_sweep_results.{fmt}"
    if os.path.isabs(filename) or os.path.dirname(filename) not in ("", "."):
        raise ValueError(
            f"Refusing results filename outside the results directory: {filename!r}"
        )

    results_dir = os.path.abspath(_indicator_sweep_results_dir())
    filepath = os.path.abspath(os.path.join(results_dir, filename))
    if os.path.dirname(filepath) != results_dir:
        raise ValueError(
            f"Refusing results filepath outside Indicator_sweep_results: {filepath!r}"
        )

    if os.path.isfile(filepath):
        print(f"Using existing results file for {ticker}: {filepath}")
    elif os.path.exists(filepath):
        raise ValueError(f"Results path exists but is not a file: {filepath!r}")
    else:
        print(f"Will create results file for {ticker}: {filepath}")
    return filepath



if "results_df" not in globals() or not isinstance(results_df, pd.DataFrame):
    raise ValueError("results_df is missing or not a DataFrame. Run the grid-search analysis cell first.")
if results_df.empty:
    raise ValueError("results_df is empty. Run the grid search cell first.")
if "oos_df_by_indicator" not in globals() or not isinstance(oos_df_by_indicator, dict):
    raise ValueError("oos_df_by_indicator is missing or not a dict. Run the OOS validation cell first.")
if not oos_df_by_indicator:
    raise ValueError("oos_df_by_indicator is empty. Run the OOS validation cell first.")
if "full_sample_results_by_indicator" not in globals() or not isinstance(full_sample_results_by_indicator, dict):
    raise ValueError("full_sample_results_by_indicator is missing or not a dict. Run the full-sample cell first.")
if not full_sample_results_by_indicator:
    raise ValueError("full_sample_results_by_indicator is empty. Run the full-sample cell first.")

if "TICKER" not in globals() or not str(TICKER).strip():
    raise ValueError("TICKER is missing. Run the download cell first.")

results_filepath = _results_filepath_for_ticker(TICKER)

saved_notebook_results = save_notebook_results_by_indicator(
    results_df,
    oos_df_by_indicator,
    full_sample_results_by_indicator,
    filepath=results_filepath,
    top_n=5,
    require_all_stages=True,
)


Using existing results file for TQQQ: tqqq_indicator_sweep_results.json


Saved notebook results by indicator to tqqq_indicator_sweep_results.json (143 rows; stages={'is': 65, 'oos': 65, 'full': 13})


In [15]:
# STORE BEST STRATEGY PER INDICATOR FOR VISUALIZATIONS

best_by_indicator = {}

# Union of indicators seen in IS results and/or OOS tables.
indicator_names = []
if "results_df" in globals() and not results_df.empty:
    indicator_names = list(results_df.groupby("indicator", sort=False).groups.keys())
if "oos_df_by_indicator" in globals() and oos_df_by_indicator:
    for name in oos_df_by_indicator.keys():
        if name not in indicator_names:
            indicator_names.append(name)

if not indicator_names:
    print("No OOS or in-sample results available to store.")
else:
    for indicator_name in indicator_names:
        stored = None

        oos_df_ind = None
        if "oos_df_by_indicator" in globals() and oos_df_by_indicator:
            oos_df_ind = oos_df_by_indicator.get(indicator_name)

        if oos_df_ind is not None and not oos_df_ind.empty:
            best_oos = best_oos_row(oos_df_ind)
            if best_oos is not None:
                try:
                    params = require_params_dict(best_oos["Params"], indicator_name)
                    stored = {
                        "source": "oos",
                        "params": params,
                        "param_display": best_oos["Strategy"],
                        "is_sharpe": safe_float(best_oos.get("IS_Sharpe")),
                        "oos_sharpe": safe_float(best_oos.get("OOS_Sharpe")),
                        "full_sample_sharpe": np.nan,
                    }
                except (TypeError, KeyError, ValueError) as exc:
                    print(f"Skipping OOS store for {indicator_name}: {exc}")

        if stored is None and "results_df" in globals() and not results_df.empty:
            indicator_results = results_df[results_df["indicator"] == indicator_name]
            best_is = best_is_row(indicator_results)
            if best_is is not None:
                try:
                    params = require_params_dict(best_is["params"], indicator_name)
                    stored = {
                        "source": "is",
                        "params": params,
                        "param_display": best_is["param_display"],
                        "is_sharpe": safe_float(best_is.get("sharpe_ratio")),
                        "oos_sharpe": np.nan,
                        "full_sample_sharpe": np.nan,
                    }
                except (TypeError, KeyError, ValueError) as exc:
                    print(f"Skipping IS store for {indicator_name}: {exc}")

        if stored is None:
            print(f"No storable strategy found for {indicator_name}.")
            continue

        fs_result = None
        if "full_sample_results_by_indicator" in globals() and full_sample_results_by_indicator:
            fs_result = full_sample_results_by_indicator.get(indicator_name)
        if fs_result is not None:
            stored["full_sample_sharpe"] = safe_float(fs_result.get("sharpe"))

        best_by_indicator[indicator_name] = stored

print("Best strategy stored per indicator:")
for indicator_name, info in best_by_indicator.items():
    print(
        f"  {indicator_name}: {info['param_display']} "
        f"(source={info['source']}, IS Sharpe={fmt_metric(info['is_sharpe'], '.3f')})"
    )

# Cross-indicator ranking (best stored strategy per indicator, ranked by OOS Sharpe).
indicator_ranking_rows = []
for indicator_name, info in best_by_indicator.items():
    indicator_ranking_rows.append({
        "Indicator": indicator_name,
        "Strategy": info["param_display"],
        "Source": info["source"],
        "IS Sharpe": info["is_sharpe"],
        "OOS Sharpe": info["oos_sharpe"],
        "Full Sample Sharpe": info.get("full_sample_sharpe", np.nan),
    })

indicator_ranking_df = pd.DataFrame(indicator_ranking_rows)

if indicator_ranking_df.empty:
    print("\nNo indicators available for cross-indicator ranking.")
else:
    has_finite_oos = indicator_ranking_df["OOS Sharpe"].apply(
        lambda x: isinstance(x, (int, float)) and np.isfinite(x)
    ).any()

    indicator_ranking_df = (
        indicator_ranking_df
        .sort_values(["OOS Sharpe", "IS Sharpe"], ascending=[False, False], na_position="last")
        .reset_index(drop=True)
    )
    indicator_ranking_df.insert(0, "Rank", range(1, len(indicator_ranking_df) + 1))

    print("\n" + "=" * 90)
    print("INDICATOR RANKING (BEST STORED STRATEGY PER INDICATOR)")
    print("=" * 90)
    print("Ranking basis: OOS Sharpe (higher is better).")
    print("Each row uses that indicator's best stored strategy (OOS-best when available, else IS-best).")
    if not has_finite_oos:
        print("Warning: no finite OOS Sharpe values; table order is informational only.")
    if "full_sample_results_by_indicator" not in globals() or not full_sample_results_by_indicator:
        print("Warning: full-sample results missing; Full Sample Sharpe will show as n/a.")
    print("=" * 90)

    ranking_display_df = indicator_ranking_df.assign(
        **{
            "IS Sharpe": indicator_ranking_df["IS Sharpe"].map(lambda x: fmt_metric(x, ".3f")),
            "OOS Sharpe": indicator_ranking_df["OOS Sharpe"].map(lambda x: fmt_metric(x, ".3f")),
            "Full Sample Sharpe": indicator_ranking_df["Full Sample Sharpe"].map(lambda x: fmt_metric(x, ".3f")),
        }
    )[["Rank", "Indicator", "Strategy", "IS Sharpe", "OOS Sharpe", "Full Sample Sharpe"]]
    print(ranking_display_df.to_string(index=False))

    if has_finite_oos:
        best_ranked = indicator_ranking_df.iloc[0]
        print("\n" + "-" * 90)
        print(f"TOP RANKED INDICATOR: {best_ranked['Indicator']} ({best_ranked['Strategy']})")
        print(f"OOS Sharpe: {fmt_metric(best_ranked['OOS Sharpe'], '.3f')}")
        print(f"IS Sharpe: {fmt_metric(best_ranked['IS Sharpe'], '.3f')}")
        print(f"Full Sample Sharpe: {fmt_metric(best_ranked['Full Sample Sharpe'], '.3f')}")
    else:
        print("\nNo top-ranked indicator declared because OOS Sharpe is unavailable for all rows.")

# Compatibility for Triple-EMA-specific downstream cells.
if "Triple EMA" in best_by_indicator:
    BEST_INDICATOR = "Triple EMA"
    BEST_STRATEGY_LABEL = best_by_indicator["Triple EMA"]["param_display"]
    BEST_PARAMS = best_by_indicator["Triple EMA"]["params"]
    BEST_IS_SHARPE = best_by_indicator["Triple EMA"]["is_sharpe"]
    BEST_OOS_SHARPE = best_by_indicator["Triple EMA"]["oos_sharpe"]
    missing = [k for k in ("ema1_period", "ema2_period", "ema3_period") if k not in BEST_PARAMS]
    if missing:
        print(f"Triple EMA best params missing keys {missing}; downstream EMA defaults not set.")
        for name in ("BEST_EMA1", "BEST_EMA2", "BEST_EMA3"):
            if name in globals():
                del globals()[name]
    else:
        BEST_EMA1 = int(BEST_PARAMS["ema1_period"])
        BEST_EMA2 = int(BEST_PARAMS["ema2_period"])
        BEST_EMA3 = int(BEST_PARAMS["ema3_period"])
        print(f"\nDownstream Triple EMA defaults set to: EMA({BEST_EMA1}, {BEST_EMA2}, {BEST_EMA3})")
elif "BEST_EMA1" in globals():
    del BEST_EMA1, BEST_EMA2, BEST_EMA3


Best strategy stored per indicator:
  Triple EMA: EMA(4,117,125) (source=oos, IS Sharpe=0.928)
  MACD: MACD(12,69,78) (source=oos, IS Sharpe=0.724)
  AROON: AROON(21) (source=oos, IS Sharpe=0.722)
  STC: STC(38,99,23) (source=oos, IS Sharpe=0.964)
  KAMA: KAMA(10,7,99) (source=oos, IS Sharpe=0.807)
  SUPERTREND: SUPERTREND(ATR=89,Factor=3) (source=oos, IS Sharpe=0.808)
  KALMAN: KALMAN(Q=1,R=159) (source=oos, IS Sharpe=0.851)
  RSI: RSI(73) (source=oos, IS Sharpe=0.779)
  ADX: ADX(DI=21,Smooth=31) (source=oos, IS Sharpe=1.170)
  DONCHIAN: DONCHIAN(24) (source=oos, IS Sharpe=0.488)
  TRIX: TRIX(7) (source=oos, IS Sharpe=0.711)
  VORTEX: VORTEX(115) (source=oos, IS Sharpe=0.586)
  ALMA: ALMA(2,99,0.10,3) (source=oos, IS Sharpe=0.786)

INDICATOR RANKING (BEST STORED STRATEGY PER INDICATOR)
Ranking basis: OOS Sharpe (higher is better).
Each row uses that indicator's best stored strategy (OOS-best when available, else IS-best).
 Rank  Indicator                    Strategy IS Sharpe OOS Shar

In [16]:
# FULL-SAMPLE ROLLING SHARPE + DRAWDOWNS - for all strategies

# Prefer stored per-indicator best strategies (OOS-best when available, else best IS).
strategies_to_plot = []
if "best_by_indicator" in globals() and best_by_indicator:
    for indicator_name, info in best_by_indicator.items():
        try:
            params = require_params_dict(info["params"], indicator_name)
        except (TypeError, KeyError, ValueError) as exc:
            print(f"{indicator_name}: skipping (invalid stored params: {exc})")
            continue
        strategies_to_plot.append((indicator_name, params, info["param_display"]))
elif "results_df" in globals() and not results_df.empty:
    for indicator_name, indicator_results in results_df.groupby("indicator", sort=False):
        best = best_is_row(indicator_results)
        if best is None:
            continue
        try:
            params = require_params_dict(best["params"], indicator_name)
        except (TypeError, KeyError, ValueError) as exc:
            print(f"{indicator_name}: skipping (invalid IS params: {exc})")
            continue
        strategies_to_plot.append((indicator_name, params, best["param_display"]))

if not strategies_to_plot:
    print("No results to visualize.")
elif "close" not in globals() or close is None or len(close) < 2:
    print("Close series missing or too short for rolling Sharpe / drawdown plots.")
else:
    # Reuse full-sample portfolios when available to avoid re-running identical backtests.
    fs_by_indicator = (
        full_sample_results_by_indicator
        if "full_sample_results_by_indicator" in globals() and full_sample_results_by_indicator
        else {}
    )
    # Window depends only on series length / annualization constants, so compute once.
    rolling_window = max(20, min(TRADING_DAYS_PER_YEAR, max(1, len(close) // 4)))

    for best_indicator, best_params, param_display in strategies_to_plot:
        best_label = f"{best_indicator} {param_display}"
        try:
            fs_result = fs_by_indicator.get(best_indicator)
            if fs_result is not None and fs_result.get("portfolio") is not None:
                pf_full = fs_result["portfolio"]
            else:
                best_spec = get_active_indicator_spec(best_indicator)
                entries_series, exits_series = best_spec["build_signals"](
                    close, best_params, shift_signals=True
                )
                pf_full = run_signal_backtest(close, entries_series, exits_series)

            ret = pf_full.returns()

            # 1) Rolling Sharpe (dynamic annualization window)
            if len(ret) > rolling_window:
                rolling_sharpe = ret.rolling(window=rolling_window).apply(
                    lambda x: (x.mean() * TRADING_DAYS_PER_YEAR) / (x.std() * np.sqrt(TRADING_DAYS_PER_YEAR)) if x.std() and x.std() != 0 else np.nan,
                    raw=False
                )

                plt.figure(figsize=(12, 5))
                plt.plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=2, color='blue', alpha=0.85)
                plt.axhline(y=1.0, color='green', linestyle='--', alpha=0.6, label='Sharpe = 1.0')
                plt.axhline(y=0.5, color='orange', linestyle='--', alpha=0.6, label='Sharpe = 0.5')
                plt.axhline(y=0.0, color='red', linestyle='--', alpha=0.6, label='Sharpe = 0.0')
                plt.title(f'Rolling Sharpe (window={rolling_window}) - {best_label} Full Sample', fontsize=14, fontweight='bold')
                plt.xlabel('Date', fontsize=12)
                plt.ylabel('Rolling Sharpe Ratio', fontsize=12)
                plt.grid(True, alpha=0.3)
                plt.legend()
                plt.tight_layout()
                plt.show()
            else:
                print(f"Not enough data for rolling Sharpe calculation: {best_label}")

            # 2) Drawdowns (underwater)
            eq = (1 + ret).cumprod()
            peak = eq.cummax()
            dd = (eq - peak) / peak.replace(0, np.nan)

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

            ax1.plot(eq.index, (eq - 1) * 100, color='black', linewidth=1.5, label='Cumulative Return %')
            ax1.plot(peak.index, (peak - 1) * 100, color='green', linewidth=1.0, alpha=0.7, label='Peak %')
            ax1.fill_between(eq.index, (eq - 1) * 100, (peak - 1) * 100, color='red', alpha=0.25, label='Drawdown Area')
            ax1.set_title(f'Cumulative Returns & Drawdowns - {best_label} Full Sample', fontsize=14, fontweight='bold')
            ax1.set_ylabel('Return (%)', fontsize=12)
            ax1.grid(True, alpha=0.3)
            ax1.legend()

            ax2.fill_between(dd.index, dd * 100, 0, color='red', alpha=0.7, label='Drawdown %')
            ax2.plot(dd.index, dd * 100, color='darkred', linewidth=1)
            ax2.set_title('Underwater Chart (Drawdown from Peak)', fontsize=13)
            ax2.set_xlabel('Date', fontsize=12)
            ax2.set_ylabel('Drawdown (%)', fontsize=12)
            ax2.grid(True, alpha=0.3)
            ax2.legend()

            plt.tight_layout()
            plt.show()

            print(f"Rolling Sharpe and Drawdown analysis complete for: {best_label}")
        except Exception as exc:
            print(f"{best_label}: rolling Sharpe / drawdown failed ({type(exc).__name__}: {exc})")


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\4057478222.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\4057478222.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Rolling Sharpe and Drawdown analysis complete for: Triple EMA EMA(4,117,125)


Rolling Sharpe and Drawdown analysis complete for: MACD MACD(12,69,78)


Rolling Sharpe and Drawdown analysis complete for: AROON AROON(21)


Rolling Sharpe and Drawdown analysis complete for: STC STC(38,99,23)


...

Rolling Sharpe and Drawdown analysis complete for: DONCHIAN DONCHIAN(24)


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\4057478222.py:61: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(12, 5))


Rolling Sharpe and Drawdown analysis complete for: TRIX TRIX(7)


Rolling Sharpe and Drawdown analysis complete for: VORTEX VORTEX(115)


Rolling Sharpe and Drawdown analysis complete for: ALMA ALMA(2,99,0.10,3)


In [17]:
# STEP 5C: VISUALS ONLY (Signals + Equity Curves) - for all strategies

def _params_match(stored_params, current_params):
    return isinstance(stored_params, dict) and stored_params == current_params


# Prefer stored per-indicator best strategies (OOS-best when available, else best IS).
strategies_to_plot = []
if "best_by_indicator" in globals() and best_by_indicator:
    for indicator_name, info in best_by_indicator.items():
        if "active_indicator_grids" in globals() and indicator_name not in active_indicator_grids:
            print(f"{indicator_name}: skipping (not in active_indicator_grids)")
            continue
        try:
            params = require_params_dict(info["params"], indicator_name)
            param_display = info.get("param_display")
            if not param_display:
                param_display = get_active_indicator_spec(indicator_name)["format_params"](params)
        except (TypeError, KeyError, ValueError) as exc:
            print(f"{indicator_name}: skipping (invalid stored params: {exc})")
            continue
        strategies_to_plot.append((indicator_name, params, param_display))
elif "results_df" in globals() and not results_df.empty:
    if "best_is_row" not in globals():
        print("best_is_row is not defined; run the grid-search setup cells first.")
    else:
        for indicator_name, indicator_results in results_df.groupby("indicator", sort=False):
            if "active_indicator_grids" in globals() and indicator_name not in active_indicator_grids:
                print(f"{indicator_name}: skipping (not in active_indicator_grids)")
                continue
            best = best_is_row(indicator_results)
            if best is None:
                continue
            try:
                params = require_params_dict(best["params"], indicator_name)
                param_display = best.get("param_display") or get_active_indicator_spec(indicator_name)["format_params"](params)
            except (TypeError, KeyError, ValueError) as exc:
                print(f"{indicator_name}: skipping (invalid IS params: {exc})")
                continue
            strategies_to_plot.append((indicator_name, params, param_display))

required_series = ("close", "train_close", "val_close")
missing_series = [name for name in required_series if name not in globals() or globals()[name] is None]

if not strategies_to_plot:
    print("No results to visualize.")
elif missing_series:
    print(f"Missing required price series: {', '.join(missing_series)}")
elif "split_idx" not in globals():
    print("split_idx is not defined; cannot build train/val equity curves.")
elif len(close) < 2:
    print("Close series too short for visualizations.")
elif len(train_close) < 1 or len(val_close) < 1:
    print(f"Train/val segments are empty: train={len(train_close)}, val={len(val_close)}")
elif split_idx <= 0 or split_idx >= len(close):
    print(f"Invalid split_idx={split_idx} for close length {len(close)}.")
elif len(train_close) != split_idx or len(val_close) != len(close) - split_idx:
    print(
        f"Train/val lengths do not match split_idx: "
        f"train={len(train_close)}, val={len(val_close)}, split_idx={split_idx}, close={len(close)}"
    )
else:
    fs_by_indicator = (
        full_sample_results_by_indicator
        if "full_sample_results_by_indicator" in globals() and full_sample_results_by_indicator
        else {}
    )

    for indicator_name, params, param_display in strategies_to_plot:
        best_label = f"{indicator_name} {param_display}"
        try:
            spec = get_active_indicator_spec(indicator_name)
            entries_series, exits_series = spec["build_signals"](close, params, shift_signals=True)

            if len(entries_series) != len(close) or len(exits_series) != len(close):
                raise ValueError(
                    f"Signal length mismatch: entries={len(entries_series)}, "
                    f"exits={len(exits_series)}, close={len(close)}"
                )

            entries_full = entries_series.to_numpy(dtype=bool)
            exits_full = exits_series.to_numpy(dtype=bool)
            entries_train = entries_full[:split_idx]
            exits_train = exits_full[:split_idx]
            entries_val = entries_full[split_idx:]
            exits_val = exits_full[split_idx:]

            if len(entries_train) != len(train_close) or len(entries_val) != len(val_close):
                raise ValueError("Train/val signal slices do not align with train_close/val_close.")

            # Reuse cached full-sample portfolio when params match; always run train/val backtests.
            fs_result = fs_by_indicator.get(indicator_name)
            if (
                fs_result is not None
                and fs_result.get("portfolio") is not None
                and _params_match(fs_result.get("params"), params)
            ):
                pf_full = fs_result["portfolio"]
            else:
                pf_full = run_signal_backtest(close, entries_full, exits_full)

            pf_train = run_signal_backtest(train_close, entries_train, exits_train)
            pf_val = run_signal_backtest(val_close, entries_val, exits_val)

            # 1) Price + indicator overlay + signals (full sample)
            fig, ax = plt.subplots(figsize=(14, 7))
            ax.plot(close.index, close.values, label='Close', color='black', linewidth=1.5, alpha=0.7)

            if indicator_name == "Triple EMA":
                ema1 = int(params["ema1_period"])
                ema2 = int(params["ema2_period"])
                ema3 = int(params["ema3_period"])
                ema1_full = get_ema_series(close, ema1)
                ema2_full = get_ema_series(close, ema2)
                ema3_full = get_ema_series(close, ema3)
                ax.plot(close.index, ema1_full.values, label=f'Fast EMA ({ema1})', color='blue', alpha=0.8, linewidth=1.2)
                ax.plot(close.index, ema2_full.values, label=f'Medium EMA ({ema2})', color='orange', alpha=0.8, linewidth=1.2)
                ax.plot(close.index, ema3_full.values, label=f'Slow EMA ({ema3})', color='purple', alpha=0.8, linewidth=1.2)

            buy_idx = close.index[entries_full]
            sell_idx = close.index[exits_full]
            ax.scatter(buy_idx, close.reindex(buy_idx).values, marker='^', color='green', s=80, label='Buy', zorder=5)
            ax.scatter(sell_idx, close.reindex(sell_idx).values, marker='v', color='red', s=80, label='Sell', zorder=5)

            ax.set_title(f'Best {param_display} - Full Sample Signals', fontsize=14, fontweight='bold')
            ax.set_xlabel('Date', fontsize=12)
            ax.set_ylabel('Price', fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()

            # 2) Equity curves: train / val / full
            eq_full = (1 + pf_full.returns()).cumprod()
            eq_train = (1 + pf_train.returns()).cumprod()
            eq_val = (1 + pf_val.returns()).cumprod()

            fig, ax = plt.subplots(figsize=(14, 7))
            ax.plot(close.index, eq_full.values, label='Full Sample', color='black', linewidth=2)
            ax.plot(train_close.index, eq_train.values, label='Train (In-Sample)', color='blue', linewidth=1.5, alpha=0.8)
            ax.plot(val_close.index, eq_val.values, label='Validation (Out-of-Sample)', color='orange', linewidth=1.5, alpha=0.8)
            ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')

            ax.set_title(f'Best {param_display} - Equity Curves', fontsize=14, fontweight='bold')
            ax.set_xlabel('Date', fontsize=12)
            ax.set_ylabel('Cumulative Returns (normalized to 1)', fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()

            print(f"Visualizations complete for: {best_label}")
        except Exception as exc:
            print(f"{best_label}: visualization failed ({type(exc).__name__}: {exc})")

Visualizations complete for: Triple EMA EMA(4,117,125)


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Visualizations complete for: MACD MACD(12,69,78)


Visualizations complete for: AROON AROON(21)
Visualizations complete for: STC STC(38,99,23)


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


...

C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Visualizations complete for: DONCHIAN DONCHIAN(24)
Visualizations complete for: TRIX TRIX(7)


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Visualizations complete for: VORTEX VORTEX(115)
Visualizations complete for: ALMA ALMA(2,99,0.10,3)


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\2331449690.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# FULL-SAMPLE TRADE-BY-TRADE RETURNS (BAR) + CUMULATIVE PER-TRADE EQUITY — ALL TRADESbt

# Reuse the same full Close series as full-sample evaluation (avoid re-parsing columns).
plot_close = None
if "close" in globals() and close is not None and len(close) >= 2:
    plot_close = close.astype(float).copy()
    plot_close.name = TICKER
elif "stock_data" in globals() and stock_data is not None and "TICKER" in globals():
    try:
        plot_close = select_close_series(stock_data, TICKER).astype(float)
        plot_close.name = TICKER
    except (KeyError, ValueError, TypeError) as exc:
        print(f"Could not resolve Close series from stock_data ({exc})")

# Prefer stored per-indicator best strategies (OOS-best when available, else best IS).
strategies_to_plot = []
if "best_by_indicator" in globals() and best_by_indicator:
    for indicator_name, info in best_by_indicator.items():
        try:
            params = require_params_dict(info.get("params"), indicator_name)
            param_display = info.get("param_display") or str(params)
        except (TypeError, KeyError, ValueError) as exc:
            print(f"{indicator_name}: skipping (invalid stored params: {exc})")
            continue
        strategies_to_plot.append((indicator_name, params, param_display))
elif "results_df" in globals() and not results_df.empty:
    if "best_is_row" not in globals():
        print("best_is_row helper missing. Run the grid-search cells first.")
    else:
        for indicator_name, indicator_results in results_df.groupby("indicator", sort=False):
            best = best_is_row(indicator_results)
            if best is None:
                continue
            try:
                params = require_params_dict(best["params"], indicator_name)
            except (TypeError, KeyError, ValueError) as exc:
                print(f"{indicator_name}: skipping (invalid IS params: {exc})")
                continue
            strategies_to_plot.append((indicator_name, params, best.get("param_display") or str(params)))

if not strategies_to_plot:
    print("No results to visualize.")
elif plot_close is None or len(plot_close) < 2:
    print("Close series missing or too short for trade-by-trade plots.")
else:
    fs_by_indicator = (
        full_sample_results_by_indicator
        if "full_sample_results_by_indicator" in globals() and full_sample_results_by_indicator
        else {}
    )

    if "_params_match" not in globals():
        def _params_match(stored_params, current_params):
            return isinstance(stored_params, dict) and stored_params == current_params

    for indicator_name, params, param_display in strategies_to_plot:
        best_label = f"{indicator_name} {param_display}"
        print("\n" + "=" * 90)
        print(f"FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — {best_label}")
        print("=" * 90)

        try:
            fs_result = fs_by_indicator.get(indicator_name)
            if (
                fs_result is not None
                and fs_result.get("portfolio") is not None
                and _params_match(fs_result.get("params"), params)
            ):
                pf_full = fs_result["portfolio"]
            else:
                if fs_result is not None and not _params_match(fs_result.get("params"), params):
                    print(f"{best_label}: cached portfolio params differ; rebuilding backtest.")
                spec = get_active_indicator_spec(indicator_name)
                entries_series, exits_series = spec["build_signals"](
                    plot_close, params, shift_signals=True
                )
                if len(entries_series) != len(plot_close) or len(exits_series) != len(plot_close):
                    raise ValueError(
                        f"Signal length mismatch: entries={len(entries_series)}, "
                        f"exits={len(exits_series)}, close={len(plot_close)}"
                    )
                pf_full = run_signal_backtest(plot_close, entries_series, exits_series)

            trades = pf_full.trades
            trade_returns = (
                trades.returns.values
                if hasattr(trades.returns, "values")
                else np.asarray(trades.returns)
            )
            trade_returns = np.asarray(trade_returns, dtype=float).ravel()
            bad_mask = ~np.isfinite(trade_returns)
            if np.any(bad_mask):
                print(
                    f"{best_label}: Dropping {int(bad_mask.sum())} trade(s) with non-finite returns."
                )
                trade_returns = trade_returns[~bad_mask]

            if trade_returns.size == 0:
                print(f"{best_label}: No trades to plot.")
                continue

            winning_trades = trade_returns[trade_returns > 0]
            losing_trades = trade_returns[trade_returns < 0]
            total_trades = int(trade_returns.size)
            win_count = int(winning_trades.size)
            loss_count = int(losing_trades.size)
            win_rate = (win_count / total_trades * 100.0) if total_trades > 0 else 0.0

            avg_win_pct = (winning_trades.mean() * 100.0) if win_count > 0 else 0.0
            avg_loss_pct = (losing_trades.mean() * 100.0) if loss_count > 0 else 0.0
            max_win_pct = (winning_trades.max() * 100.0) if win_count > 0 else 0.0
            max_loss_pct = (losing_trades.min() * 100.0) if loss_count > 0 else 0.0

            print(f"Total trades plotted: {total_trades}")
            print(f"Win Rate: {win_rate:.1f}% ({win_count}W / {loss_count}L)")
            print(f"Avg Win: {avg_win_pct:.2f}% | Avg Loss: {avg_loss_pct:.2f}%")
            print(f"Max Win: {max_win_pct:.2f}% | Max Loss: {max_loss_pct:.2f}%")

            equity_per_trade = np.cumprod(1.0 + trade_returns)

            x = np.arange(1, trade_returns.size + 1)
            fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

            colors = np.where(trade_returns >= 0, "green", "red")
            axes[0].bar(x, trade_returns * 100.0, color=colors, alpha=0.85, width=0.8)
            axes[0].axhline(0, color="black", linewidth=1, alpha=0.6)

            stats_text = (
                f"Win Rate: {win_rate:.1f}% ({win_count}W/{loss_count}L)\n"
                f"Avg Win: {avg_win_pct:.2f}% | Avg Loss: {avg_loss_pct:.2f}%\n"
                f"Max Win: {max_win_pct:.2f}% | Max Loss: {max_loss_pct:.2f}%"
            )
            axes[0].text(
                0.02,
                0.98,
                stats_text,
                transform=axes[0].transAxes,
                fontsize=10,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
            )
            axes[0].set_title(
                f"Per-Trade Returns (%) - {best_label} Full Sample",
                fontsize=13,
                fontweight="bold",
            )
            axes[0].set_ylabel("Return (%)")
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(x, equity_per_trade, color="black", linewidth=2)
            axes[1].set_title("Cumulative Equity per Trade (Trade Domain)", fontsize=13)
            axes[1].set_xlabel("Trade #")
            axes[1].set_ylabel("Equity (x)")
            axes[1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            print(f"Trade-by-trade analysis complete for: {best_label}")
        except KeyError as exc:
            print(f"{best_label}: unknown indicator ({exc})")
        except ValueError as exc:
            print(f"{best_label}: invalid data/signals ({exc})")
        except Exception as exc:
            print(f"{best_label}: trade-by-trade plot failed ({type(exc).__name__}: {exc})")



FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — Triple EMA EMA(4,117,125)
Triple EMA EMA(4,117,125): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 25
Win Rate: 60.0% (15W / 10L)
Avg Win: 25.84% | Avg Loss: -6.18%
Max Win: 123.02% | Max Loss: -12.52%


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\124681817.py:157: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Trade-by-trade analysis complete for: Triple EMA EMA(4,117,125)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — MACD MACD(12,69,78)
Total trades plotted: 21
Win Rate: 52.4% (11W / 10L)
Avg Win: 45.20% | Avg Loss: -7.26%
Max Win: 123.79% | Max Loss: -15.90%
Trade-by-trade analysis complete for: MACD MACD(12,69,78)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — AROON AROON(21)
AROON AROON(21): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 40
Win Rate: 52.5% (21W / 19L)
Avg Win: 23.01% | Avg Loss: -8.64%
Max Win: 102.66% | Max Loss: -22.71%
Trade-by-trade analysis complete for: AROON AROON(21)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — STC STC(38,99,23)
Total trades plotted: 27
Win Rate: 59.3% (16W / 11L)
Avg Win: 36.69% | Avg Loss: -8.79%
Max Win: 102.52% | Max Loss: -24.79%


Trade-by-trade analysis complete for: STC STC(38,99,23)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — KAMA KAMA(10,7,99)
KAMA KAMA(10,7,99): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 48
Win Rate: 41.7% (20W / 28L)
Avg Win: 21.56% | Avg Loss: -4.04%
Max Win: 117.74% | Max Loss: -10.25%


Trade-by-trade analysis complete for: KAMA KAMA(10,7,99)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — SUPERTREND SUPERTREND(ATR=89,Factor=3)
SUPERTREND SUPERTREND(ATR=89,Factor=3): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 35
Win Rate: 45.7% (16W / 19L)
Avg Win: 39.22% | Avg Loss: -9.38%
Max Win: 92.19% | Max Loss: -15.17%
Trade-by-trade analysis complete for: SUPERTREND SUPERTREND(ATR=89,Factor=3)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — KALMAN KALMAN(Q=1,R=159)
KALMAN KALMAN(Q=1,R=159): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 106
Win Rate: 49.1% (52W / 54L)
Avg Win: 12.30% | Avg Loss: -4.89%
Max Win: 80.03% | Max Loss: -14.82%


Trade-by-trade analysis complete for: KALMAN KALMAN(Q=1,R=159)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — RSI RSI(73)
RSI RSI(73): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 43
Win Rate: 53.5% (23W / 20L)
Avg Win: 17.29% | Avg Loss: -3.97%
Max Win: 129.43% | Max Loss: -8.39%
Trade-by-trade analysis complete for: RSI RSI(73)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — ADX ADX(DI=21,Smooth=31)
ADX ADX(DI=21,Smooth=31): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 20
Win Rate: 75.0% (15W / 5L)
Avg Win: 33.37% | Avg Loss: -4.56%
Max Win: 138.33% | Max Loss: -6.78%
Trade-by-trade analysis complete for: ADX ADX(DI=21,Smooth=31)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — DONCHIAN DONCHIAN(24)
Total trades plotted: 36
Win Rate: 50.0% (18W / 18L)
Avg Win: 20.25% | Avg Loss: -9.13%
Max Win: 81.93% | Max Loss: -18.26%
Trade-by-trade analysis complete for: DONCHIAN DONCHIAN(24)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — TRIX TRIX(7)
TRIX TRIX(7): Dropping 1 trade(s) wi

Trade-by-trade analysis complete for: TRIX TRIX(7)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — VORTEX VORTEX(115)
VORTEX VORTEX(115): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 23
Win Rate: 60.9% (14W / 9L)
Avg Win: 20.95% | Avg Loss: -6.31%
Max Win: 82.00% | Max Loss: -15.86%
Trade-by-trade analysis complete for: VORTEX VORTEX(115)

FULL-SAMPLE TRADE-BY-TRADE ANALYSIS — ALMA ALMA(2,99,0.10,3)
ALMA ALMA(2,99,0.10,3): Dropping 1 trade(s) with non-finite returns.
Total trades plotted: 22
Win Rate: 68.2% (15W / 7L)
Avg Win: 21.49% | Avg Loss: -5.84%
Max Win: 112.01% | Max Loss: -12.58%
Trade-by-trade analysis complete for: ALMA ALMA(2,99,0.10,3)


In [19]:
# COMPACT PARAMETER SENSITIVITY ANALYSIS - HEATMAPS ONLY

from IPython.display import HTML, display
from matplotlib.ticker import PercentFormatter

# Keep the combined sensitivity charts and summaries expanded in notebook output.
display(HTML("""
<style>
.jp-OutputArea-output.jp-OutputArea-executeResult,
.jp-OutputArea-output.jp-OutputArea-displayData,
.jp-OutputArea-output.jp-OutputArea-stdout,
.output_scroll,
.output_subarea {
    max-height: none !important;
    height: auto !important;
    overflow-y: visible !important;
}
</style>
"""))

# ALMA has 4 grid params; sensitivity varies fast/slow/offset and holds sigma fixed at the base.
_ALMA_SENSITIVITY_FIXED = {}


def validate_alma_sensitivity_params(fast_length, slow_length, offset):
    sigma = _ALMA_SENSITIVITY_FIXED.get("alma_sigma")
    if sigma is None:
        raise ValueError(
            "ALMA sensitivity sigma is not set. Resolve the base strategy first."
        )
    fast_length, slow_length, offset, _sigma = validate_alma_crossover_params(
        fast_length, slow_length, offset, sigma
    )
    return fast_length, slow_length, offset


def build_alma_sensitivity_signals(close_series, params, shift_signals=True):
    full_params = dict(params)
    if "alma_sigma" not in full_params:
        sigma = _ALMA_SENSITIVITY_FIXED.get("alma_sigma")
        if sigma is None:
            raise ValueError(
                "ALMA sensitivity sigma is not set. Resolve the base strategy first."
            )
        full_params["alma_sigma"] = sigma
    return build_alma_signals(close_series, full_params, shift_signals=shift_signals)


INDICATOR_SENSITIVITY_SPECS = {
    "Triple EMA": {
        "param_keys": ("ema1_period", "ema2_period", "ema3_period"),
        "param_grid_sources": ("ema1_periods", "ema2_periods", "ema3_periods"),
        "range_mins": (2, 3, 4),
        "summary_title": "EMA SENSITIVITY SUMMARY",
        "base_label_fmt": "EMA({},{},{})",
        "base_missing_msg": "Base EMA combination missing from sensitivity results.",
        "validate_combo": validate_triple_ema_params,
        "build_signals": build_triple_ema_signals,
        "param_specs": [
            {"name": "EMA1", "col": "ema1", "xlabel": "EMA1 Period", "title": "EMA1 (Fast)"},
            {"name": "EMA2", "col": "ema2", "xlabel": "EMA2 Period", "title": "EMA2 (Medium)"},
            {"name": "EMA3", "col": "ema3", "xlabel": "EMA3 Period", "title": "EMA3 (Slow)"},
        ],
    },
    "MACD": {
        "param_keys": ("macd_fast_period", "macd_slow_period", "macd_signal_period"),
        "param_grid_sources": ("macd_fast_periods", "macd_slow_periods", "macd_signal_periods"),
        "range_mins": (2, 3, 2),
        "summary_title": "MACD SENSITIVITY SUMMARY",
        "base_label_fmt": "MACD({},{},{})",
        "base_missing_msg": "Base MACD combination missing from sensitivity results.",
        "validate_combo": validate_macd_params,
        "build_signals": build_macd_signals,
        "param_specs": [
            {"name": "MACD Fast", "col": "ema1", "xlabel": "MACD Fast Period", "title": "MACD Fast"},
            {"name": "MACD Slow", "col": "ema2", "xlabel": "MACD Slow Period", "title": "MACD Slow"},
            {"name": "MACD Signal", "col": "ema3", "xlabel": "MACD Signal Period", "title": "MACD Signal"},
        ],
    },
    "AROON": {
        "param_keys": ("aroon_timeperiod", "aroon_timeperiod", "aroon_timeperiod"),
        "param_grid_sources": ("aroon_timeperiods", "aroon_timeperiods", "aroon_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "AROON SENSITIVITY SUMMARY",
        "base_label_fmt": "AROON({})",
        "base_missing_msg": "Base AROON combination missing from sensitivity results.",
        "validate_combo": validate_aroon_sensitivity_params,
        "build_signals": build_aroon_signals,
        "format_base_label": lambda timeperiod, _a, _b: f"AROON({timeperiod})",
        "param_specs": [
            {"name": "AROON Timeperiod", "col": "ema1", "xlabel": "AROON Timeperiod", "title": "AROON Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {1: "value", 2: "value"}},
        ],
    },
    "STC": {
        "param_keys": ("stc_ema_short_period", "stc_ema_long_period", "stc_cycle_period"),
        "param_grid_sources": ("stc_ema_short_periods", "stc_ema_long_periods", "stc_cycle_periods"),
        "range_mins": (2, 3, 2),
        "summary_title": "STC SENSITIVITY SUMMARY",
        "base_label_fmt": "STC({},{},{})",
        "base_missing_msg": "Base STC combination missing from sensitivity results.",
        "validate_combo": validate_stc_params,
        "build_signals": build_stc_signals,
        "param_specs": [
            {"name": "STC EMA Short", "col": "ema1", "xlabel": "STC EMA Short Period", "title": "STC EMA Short"},
            {"name": "STC EMA Long", "col": "ema2", "xlabel": "STC EMA Long Period", "title": "STC EMA Long"},
            {"name": "STC Cycle", "col": "ema3", "xlabel": "STC Cycle Period", "title": "STC Cycle Period"},
        ],
    },
    "KAMA": {
        "param_keys": ("kama_period", "kama_fast_ema_constant", "kama_slow_ema_constant"),
        "param_grid_sources": ("kama_periods", "kama_fast_ema_constants", "kama_slow_ema_constants"),
        "range_mins": (2, 2, 2),
        "summary_title": "KAMA SENSITIVITY SUMMARY",
        "base_label_fmt": "KAMA({},{},{})",
        "base_missing_msg": "Base KAMA combination missing from sensitivity results.",
        "validate_combo": validate_kama_params,
        "build_signals": build_kama_signals,
        "param_specs": [
            {"name": "KAMA Period", "col": "ema1", "xlabel": "KAMA Period", "title": "KAMA Period"},
            {"name": "KAMA Fast EMA", "col": "ema2", "xlabel": "KAMA Fastest EMA Constant", "title": "KAMA Fastest EMA Constant"},
            {"name": "KAMA Slow EMA", "col": "ema3", "xlabel": "KAMA Slowest EMA Constant", "title": "KAMA Slowest EMA Constant"},
        ],
    },
    "SUPERTREND": {
        "param_keys": ("supertrend_atr_length", "supertrend_factor", "supertrend_factor"),
        "param_grid_sources": ("supertrend_atr_lengths", "supertrend_factors", "supertrend_factors"),
        "range_mins": (2, 1, 1),
        "summary_title": "SUPERTREND SENSITIVITY SUMMARY",
        "base_label_fmt": "SUPERTREND(ATR={},Factor={})",
        "base_missing_msg": "Base SUPERTREND combination missing from sensitivity results.",
        "validate_combo": validate_supertrend_sensitivity_params,
        "build_signals": build_supertrend_signals,
        "format_base_label": lambda atr_length, factor, _anchor: f"SUPERTREND(ATR={atr_length},Factor={factor})",
        "param_specs": [
            {"name": "SUPERTREND ATR Length", "col": "ema1", "xlabel": "ATR Length", "title": "SUPERTREND ATR Length"},
            {"name": "SUPERTREND Factor", "col": "ema2", "xlabel": "Factor / Multiplier", "title": "SUPERTREND Factor"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {2: "dim1"}},
            {"dim": 1, "grid_index": 1, "sync": {2: "value"}},
        ],
    },
    "KALMAN": {
        "param_keys": ("kalman_process_noise", "kalman_measurement_noise", "kalman_measurement_noise"),
        "param_grid_sources": ("kalman_process_noise_values", "kalman_measurement_noise_values", "kalman_measurement_noise_values"),
        "range_mins": (1, 1, 1),
        "summary_title": "KALMAN SENSITIVITY SUMMARY",
        "base_label_fmt": "KALMAN(Q={},R={})",
        "base_missing_msg": "Base KALMAN combination missing from sensitivity results.",
        "validate_combo": validate_kalman_sensitivity_params,
        "build_signals": build_kalman_signals,
        "format_base_label": lambda process_noise, measurement_noise, _anchor: f"KALMAN(Q={process_noise},R={measurement_noise})",
        "param_specs": [
            {"name": "KALMAN Process Noise Q", "col": "ema1", "xlabel": "Process Noise (Q)", "title": "KALMAN Process Noise (Q)"},
            {"name": "KALMAN Measurement Noise R", "col": "ema2", "xlabel": "Measurement Noise (R)", "title": "KALMAN Measurement Noise (R)"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {2: "dim1"}},
            {"dim": 1, "grid_index": 1, "sync": {2: "value"}},
        ],
    },
    "RSI": {
        "param_keys": ("rsi_timeperiod", "rsi_timeperiod", "rsi_timeperiod"),
        "param_grid_sources": ("rsi_timeperiods", "rsi_timeperiods", "rsi_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "RSI SENSITIVITY SUMMARY",
        "base_label_fmt": "RSI({})",
        "base_missing_msg": "Base RSI combination missing from sensitivity results.",
        "validate_combo": validate_rsi_sensitivity_params,
        "build_signals": build_rsi_signals,
        "format_base_label": lambda timeperiod, _a, _b: f"RSI({timeperiod})",
        "param_specs": [
            {"name": "RSI Timeperiod", "col": "ema1", "xlabel": "RSI Timeperiod", "title": "RSI Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {1: "value", 2: "value"}},
        ],
    },
    "ADX": {
        "param_keys": ("adx_di_length", "adx_smoothing_timeperiod", "adx_smoothing_timeperiod"),
        "param_grid_sources": ("adx_di_lengths", "adx_smoothing_timeperiods", "adx_smoothing_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "ADX SENSITIVITY SUMMARY",
        "base_label_fmt": "ADX(DI={},Smooth={})",
        "base_missing_msg": "Base ADX combination missing from sensitivity results.",
        "validate_combo": validate_adx_sensitivity_params,
        "build_signals": build_adx_signals,
        "format_base_label": lambda di_length, adx_smoothing, _anchor: f"ADX(DI={di_length},Smooth={adx_smoothing})",
        "param_specs": [
            {"name": "ADX DI Length", "col": "ema1", "xlabel": "DI Length", "title": "ADX DI Length"},
            {"name": "ADX Smoothing", "col": "ema2", "xlabel": "ADX Smoothing Timeperiod", "title": "ADX Smoothing Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {2: "dim1"}},
            {"dim": 1, "grid_index": 1, "sync": {2: "value"}},
        ],
    },
    "DONCHIAN": {
        "param_keys": ("donchian_timeperiod", "donchian_timeperiod", "donchian_timeperiod"),
        "param_grid_sources": ("donchian_timeperiods", "donchian_timeperiods", "donchian_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "DONCHIAN SENSITIVITY SUMMARY",
        "base_label_fmt": "DONCHIAN({})",
        "base_missing_msg": "Base DONCHIAN combination missing from sensitivity results.",
        "validate_combo": validate_donchian_sensitivity_params,
        "build_signals": build_donchian_signals,
        "format_base_label": lambda timeperiod, _a, _b: f"DONCHIAN({timeperiod})",
        "param_specs": [
            {"name": "DONCHIAN Timeperiod", "col": "ema1", "xlabel": "DONCHIAN Timeperiod", "title": "DONCHIAN Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {1: "value", 2: "value"}},
        ],
    },
    "TRIX": {
        "param_keys": ("trix_timeperiod", "trix_timeperiod", "trix_timeperiod"),
        "param_grid_sources": ("trix_timeperiods", "trix_timeperiods", "trix_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "TRIX SENSITIVITY SUMMARY",
        "base_label_fmt": "TRIX({})",
        "base_missing_msg": "Base TRIX combination missing from sensitivity results.",
        "validate_combo": validate_trix_sensitivity_params,
        "build_signals": build_trix_signals,
        "format_base_label": lambda timeperiod, _a, _b: f"TRIX({timeperiod})",
        "param_specs": [
            {"name": "TRIX Timeperiod", "col": "ema1", "xlabel": "TRIX Timeperiod", "title": "TRIX Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {1: "value", 2: "value"}},
        ],
    },
    "VORTEX": {
        "param_keys": ("vortex_timeperiod", "vortex_timeperiod", "vortex_timeperiod"),
        "param_grid_sources": ("vortex_timeperiods", "vortex_timeperiods", "vortex_timeperiods"),
        "range_mins": (2, 2, 2),
        "summary_title": "VORTEX SENSITIVITY SUMMARY",
        "base_label_fmt": "VORTEX({})",
        "base_missing_msg": "Base VORTEX combination missing from sensitivity results.",
        "validate_combo": validate_vortex_sensitivity_params,
        "build_signals": build_vortex_signals,
        "format_base_label": lambda timeperiod, _a, _b: f"VORTEX({timeperiod})",
        "param_specs": [
            {"name": "VORTEX Timeperiod", "col": "ema1", "xlabel": "VORTEX Timeperiod", "title": "VORTEX Timeperiod"},
        ],
        "vary_axes": [
            {"dim": 0, "grid_index": 0, "sync": {1: "value", 2: "value"}},
        ],
    },
    "ALMA": {
        "param_keys": ("alma_fast_length", "alma_slow_length", "alma_offset"),
        "param_grid_sources": ("alma_fast_lengths", "alma_slow_lengths", "alma_offsets"),
        "range_mins": (2, 3, 0),
        "summary_title": "ALMA SENSITIVITY SUMMARY",
        "base_label_fmt": "ALMA(fast={},slow={},offset={})",
        "base_missing_msg": "Base ALMA combination missing from sensitivity results.",
        "validate_combo": validate_alma_sensitivity_params,
        "build_signals": build_alma_sensitivity_signals,
        "format_base_label": lambda fast_length, slow_length, offset: (
            f"ALMA(fast={fast_length},slow={slow_length},"
            f"offset={offset / 100:.2f},sigma={_ALMA_SENSITIVITY_FIXED.get('alma_sigma', '?')})"
        ),
        "fixed_from_base_keys": ("alma_sigma",),
        "fixed_param_store": _ALMA_SENSITIVITY_FIXED,
        "param_specs": [
            {"name": "ALMA Fast Length", "col": "ema1", "xlabel": "ALMA Fast Length", "title": "ALMA Fast Length"},
            {"name": "ALMA Slow Length", "col": "ema2", "xlabel": "ALMA Slow Length", "title": "ALMA Slow Length"},
            {"name": "ALMA Offset", "col": "ema3", "xlabel": "ALMA Offset (hundredths)", "title": "ALMA Offset"},
        ],
    },
}


_SENSITIVITY_GRID_CACHE = {}


def resolve_param_grid(source_name):
    if source_name in _SENSITIVITY_GRID_CACHE:
        return _SENSITIVITY_GRID_CACHE[source_name]

    grid_values = globals().get(source_name)
    if grid_values is None:
        raise KeyError(f"Missing grid definition: {source_name}")
    if not hasattr(grid_values, "__iter__") or isinstance(grid_values, (str, bytes)):
        raise TypeError(f"Grid definition {source_name} must be an iterable of numbers")
    try:
        grid = [int(value) for value in grid_values]
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Grid definition {source_name} contains non-integer values") from exc
    if not grid:
        raise ValueError(f"Grid definition {source_name} is empty")

    _SENSITIVITY_GRID_CACHE[source_name] = grid
    return grid


def resolve_sensitivity_param_grids(spec):
    grids = [resolve_param_grid(source_name) for source_name in spec["param_grid_sources"]]
    return grids[: len(spec["param_specs"])]


def grid_index_for_base(grid_values, base_value):
    if not grid_values:
        raise ValueError("Cannot locate base value on an empty grid")
    base_value = int(base_value)
    try:
        return grid_values.index(base_value)
    except ValueError:
        return min(range(len(grid_values)), key=lambda idx: abs(grid_values[idx] - base_value))


def grid_window_values(grid_values, base_value, window_steps=15):
    """Return grid-tested values within ±window_steps of base on the search grid."""
    idx = grid_index_for_base(grid_values, base_value)
    lo = max(0, idx - window_steps)
    hi = min(len(grid_values), idx + window_steps + 1)
    return grid_values[lo:hi]


def grid_step_size(grid_values):
    if len(grid_values) < 2:
        return 1
    positive_steps = [b - a for a, b in zip(grid_values, grid_values[1:]) if b > a]
    return min(positive_steps) if positive_steps else 1


def set_grid_aligned_xticks(ax, param_values, grid_values=None):
    ticks = sorted({int(value) for value in param_values if pd.notna(value)})
    if not ticks:
        return
    if len(ticks) > 20:
        step = grid_step_size(grid_values) if grid_values else grid_step_size(ticks)
        tick_stride = max(1, len(ticks) // 15)
        tick_step = step * tick_stride
        ticks = [tick for tick in ticks if (tick - ticks[0]) % tick_step == 0]
    ax.set_xticks(ticks)
    if len(ticks) > 12:
        ax.tick_params(axis="x", rotation=45)


def make_sensitivity_combo(bases, vary_dim, value, sync_rules=None):
    combo = list(bases)
    combo[vary_dim] = int(value)
    for target_dim, rule in (sync_rules or {}).items():
        if rule == "value":
            combo[target_dim] = int(value)
        elif rule == "dim1":
            combo[target_dim] = combo[1]
        elif rule == "base":
            combo[target_dim] = bases[target_dim]
    return tuple(combo)


def default_vary_axes(spec):
    return [
        {"dim": idx, "grid_index": idx, "sync": {}}
        for idx in range(len(spec["param_specs"]))
    ]


def variation_mask_for_axis(sens_df, axis, bases):
    mask = pd.Series(True, index=sens_df.index)
    vary_dim = axis["dim"]
    param_cols = ("ema1", "ema2", "ema3")

    for dim, col in enumerate(param_cols):
        if dim == vary_dim or dim in axis.get("sync", {}):
            continue
        mask &= sens_df[col] == bases[dim]

    for target_dim, rule in axis.get("sync", {}).items():
        col = param_cols[target_dim]
        if rule == "value":
            mask &= sens_df[col] == sens_df[param_cols[vary_dim]]
        elif rule == "dim1":
            mask &= sens_df[col] == sens_df["ema2"]
        elif rule == "base":
            mask &= sens_df[col] == bases[target_dim]

    return mask


def build_variation_sets(sens_df, bases, spec):
    vary_axes = spec.get("vary_axes") or default_vary_axes(spec)
    param_cols = ("ema1", "ema2", "ema3")
    variation_sets = []
    for axis in vary_axes:
        vary_col = param_cols[axis["dim"]]
        mask = variation_mask_for_axis(sens_df, axis, bases)
        variation_sets.append(sens_df[mask].copy().sort_values(vary_col))
    return variation_sets


def pct_delta(values, base_value):
    """Return percentage delta from a finite, non-zero base value."""
    if pd.isna(base_value) or not np.isfinite(base_value) or base_value == 0:
        return np.nan
    return ((values - base_value) / abs(base_value) * 100)


def drawdown_pct_delta(values, base_value):
    """Return percentage delta in drawdown magnitude, where positive means deeper drawdown."""
    if pd.isna(base_value) or not np.isfinite(base_value) or base_value == 0:
        return np.nan
    return ((values.abs() - abs(base_value)) / abs(base_value) * 100)


def max_abs_finite(values):
    finite_values = pd.to_numeric(values, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    return finite_values.abs().max() if not finite_values.empty else np.nan


def count_delta(values, base_value):
    """Return absolute count delta from a finite base value."""
    if pd.isna(base_value) or not np.isfinite(base_value):
        return np.nan
    return values - base_value


def neighbor_variations(variations, param_col, base_param, grid_values):
    """Keep only the base value and its immediate neighbors on the search grid."""
    if variations.empty:
        return variations
    idx = grid_index_for_base(grid_values, base_param)
    neighbor_values = {
        grid_values[j]
        for j in (idx - 1, idx, idx + 1)
        if 0 <= j < len(grid_values)
    }
    param_values = pd.to_numeric(variations[param_col], errors="coerce").dropna().astype(int)
    return variations.loc[param_values.index[param_values.isin(neighbor_values)]]


def is_finite_number(value):
    return pd.notna(value) and np.isfinite(value)


def directional_delta_colors(delta_values):
    colors = []
    for x in pd.to_numeric(delta_values, errors='coerce'):
        if not is_finite_number(x):
            colors.append('gray')
        elif x < -10:
            colors.append('red')
        elif x < 0:
            colors.append('orange')
        elif x < 10:
            colors.append('lightgreen')
        else:
            colors.append('darkgreen')
    return colors


def absolute_delta_colors(delta_values):
    colors = []
    for x in pd.to_numeric(delta_values, errors='coerce'):
        if not is_finite_number(x):
            colors.append('gray')
        elif abs(x) > 40:
            colors.append('red')
        elif abs(x) > 20:
            colors.append('orange')
        elif abs(x) > 10:
            colors.append('lightgreen')
        else:
            colors.append('darkgreen')
    return colors


def add_base_line(ax, base_value, label):
    if is_finite_number(base_value):
        ax.axhline(base_value, color='black', linewidth=1.5, alpha=0.7, label=label)


def prefer_finite(evaluated, stored):
    """Prefer sensitivity-evaluated metric; fall back to stored when non-finite."""
    if is_finite_number(evaluated):
        return evaluated
    if is_finite_number(stored):
        return stored
    return np.nan


def note_if_reference_differs(kind, evaluated, stored):
    if (
        is_finite_number(evaluated)
        and is_finite_number(stored)
        and abs(evaluated - stored) > 1e-6
    ):
        print(
            f"Note: using sensitivity-evaluated {kind} Sharpe {evaluated:.3f} "
            f"for reference line (stored/grid Sharpe was {stored:.3f})."
        )
    elif is_finite_number(stored) and not is_finite_number(evaluated):
        print(
            f"Note: sensitivity-evaluated {kind} Sharpe was non-finite; "
            f"falling back to stored/grid Sharpe {stored:.3f}."
        )


def extract_base_metrics(base_row):
    """Pull base reference metrics safely from the evaluated sensitivity row."""
    required = (
        'is_sharpe', 'oos_sharpe',
        'is_trades', 'oos_trades',
        'is_volatility', 'oos_volatility',
        'is_maxdd', 'oos_maxdd',
    )
    missing = [col for col in required if col not in base_row.index]
    if missing:
        raise KeyError(f"sensitivity base row missing columns: {', '.join(missing)}")

    def _trades(col):
        value = safe_float(base_row[col])
        return int(value) if is_finite_number(value) else np.nan

    return {
        'is_sharpe': safe_float(base_row['is_sharpe']),
        'oos_sharpe': safe_float(base_row['oos_sharpe']),
        'is_trades': _trades('is_trades'),
        'oos_trades': _trades('oos_trades'),
        'is_volatility': safe_float(base_row['is_volatility']),
        'oos_volatility': safe_float(base_row['oos_volatility']),
        'is_maxdd': safe_float(base_row['is_maxdd']),
        'oos_maxdd': safe_float(base_row['oos_maxdd']),
    }


def portfolio_metrics_with_prefix(pf, prefix):
    """Shared IS/OOS metric extraction to avoid duplicated Portfolio calls."""
    return {
        f"{prefix}_sharpe": safe_float(pf.sharpe_ratio(freq=FREQ, year_freq=YEAR_FREQ)),
        f"{prefix}_return": safe_float(pf.total_return()),
        f"{prefix}_maxdd": safe_float(pf.max_drawdown()),
        f"{prefix}_volatility": safe_float(pf.annualized_volatility(freq=FREQ, year_freq=YEAR_FREQ)),
        f"{prefix}_trades": len(pf.trades),
    }


def eval_indicator_combo_both(e1: int, e2: int, e3: int, spec) -> dict:
    """Evaluate one already-validated parameter combo on IS and OOS."""
    params = dict(zip(spec['param_keys'], (e1, e2, e3)))

    e_is, x_is = spec['build_signals'](train_close, params, shift_signals=True)
    pf_is = run_signal_backtest(train_close, e_is, x_is)

    e_oos, x_oos = spec['build_signals'](val_close, params, shift_signals=True)
    pf_oos = run_signal_backtest(val_close, e_oos, x_oos)

    return {
        'ema1': e1,
        'ema2': e2,
        'ema3': e3,
        **portfolio_metrics_with_prefix(pf_is, 'is'),
        **portfolio_metrics_with_prefix(pf_oos, 'oos'),
    }


def plot_trade_values(ax, variations, param_spec, base_params, title, value_col, delta_col, base_value, grid_values=None):
    colors = absolute_delta_colors(variations[delta_col])
    ax.bar(variations[param_spec['col']], variations[value_col], color=colors, edgecolor='black', linewidth=0.5)
    add_base_line(ax, base_value, f'Base trades={base_value:.0f}')
    ax.axvline(
        base_params[param_spec['col']], color='blue', linestyle='--', linewidth=2.5, alpha=0.7,
        label=f'Base={base_params[param_spec["col"]]}'
    )
    ax.set_xlabel(param_spec['xlabel'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Total Trades', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=10, loc='upper right')
    set_grid_aligned_xticks(ax, variations[param_spec['col']], grid_values)


def plot_volatility_values(ax, variations, param_spec, base_params, title, value_col, delta_col, base_value, grid_values=None):
    colors = absolute_delta_colors(variations[delta_col])
    ax.bar(variations[param_spec['col']], variations[value_col], color=colors, edgecolor='black', linewidth=0.5)
    add_base_line(ax, base_value, f'Base vol={base_value:.2%}')
    ax.axvline(
        base_params[param_spec['col']], color='blue', linestyle='--', linewidth=2.5, alpha=0.7,
        label=f'Base={base_params[param_spec["col"]]}'
    )
    ax.set_xlabel(param_spec['xlabel'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Volatility', fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=10, loc='upper right')
    set_grid_aligned_xticks(ax, variations[param_spec['col']], grid_values)


def plot_maxdd_values(ax, variations, param_spec, base_params, title, value_col, delta_col, base_value, grid_values=None):
    colors = absolute_delta_colors(variations[delta_col])
    ax.bar(variations[param_spec['col']], variations[value_col], color=colors, edgecolor='black', linewidth=0.5)
    add_base_line(ax, base_value, f'Base max DD={base_value:.2%}')
    ax.axvline(
        base_params[param_spec['col']], color='blue', linestyle='--', linewidth=2.5, alpha=0.7,
        label=f'Base={base_params[param_spec["col"]]}'
    )
    ax.set_xlabel(param_spec['xlabel'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Max Drawdown', fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=10, loc='upper right')
    set_grid_aligned_xticks(ax, variations[param_spec['col']], grid_values)


def sensitivity_prerequisites_ok():
    required = (
        'train_close',
        'val_close',
        'FREQ',
        'YEAR_FREQ',
        'run_signal_backtest',
        'require_params_dict',
        'best_is_row',
        'safe_float',
    )
    missing = [name for name in required if name not in globals()]
    if missing:
        print(f"Missing prerequisites for sensitivity analysis: {', '.join(missing)}")
        return False
    if getattr(train_close, 'empty', len(train_close) == 0):
        print("train_close is empty; cannot run sensitivity analysis.")
        return False
    if getattr(val_close, 'empty', len(val_close) == 0):
        print("val_close is empty; cannot run sensitivity analysis.")
        return False
    return True


def resolve_sensitivity_indicators():
    candidates = []
    if 'active_indicator_names' in globals() and active_indicator_names:
        candidates = list(active_indicator_names)
    elif 'best_by_indicator' in globals() and best_by_indicator:
        candidates = list(best_by_indicator.keys())
    elif 'results_df' in globals() and not results_df.empty:
        candidates = list(results_df.groupby('indicator', sort=False).groups.keys())

    return [name for name in candidates if name in INDICATOR_SENSITIVITY_SPECS]


def resolve_indicator_base(indicator_name, spec):
    try:
        fixed_store = spec.get("fixed_param_store")
        fixed_keys = tuple(spec.get("fixed_from_base_keys") or ())
        # Prevent stale fixed values from a prior indicator/run leaking forward.
        if fixed_store is not None:
            fixed_store.clear()

        if 'best_by_indicator' in globals() and indicator_name in best_by_indicator:
            info = best_by_indicator[indicator_name]
            params = require_params_dict(info['params'], indicator_name)
            if fixed_store is not None:
                for key in fixed_keys:
                    if key not in params:
                        raise KeyError(f"{indicator_name} base params missing fixed key: {key}")
                    fixed_store[key] = int(params[key])
            p1, p2, p3 = (int(params[key]) for key in spec['param_keys'])
            return p1, p2, p3, safe_float(info['is_sharpe']), safe_float(info['oos_sharpe'])

        if 'results_df' not in globals() or results_df.empty:
            return None

        indicator_results = results_df[results_df['indicator'] == indicator_name]
        best = best_is_row(indicator_results)
        if best is None:
            return None

        if fixed_store is not None:
            for key in fixed_keys:
                if key not in best:
                    raise KeyError(f"{indicator_name} base results missing fixed key: {key}")
                fixed_store[key] = int(best[key])
        p1, p2, p3 = (int(best[key]) for key in spec['param_keys'])
        return p1, p2, p3, safe_float(best['sharpe_ratio']), np.nan
    except (KeyError, TypeError, ValueError) as exc:
        print(f"Could not resolve base parameters for {indicator_name}: {exc}")
        return None



def build_sensitivity_combos(p1_base, p2_base, p3_base, spec, param_grids):
    window_steps = spec.get("neighbor_window", 15)
    bases = (int(p1_base), int(p2_base), int(p3_base))
    vary_axes = spec.get("vary_axes") or default_vary_axes(spec)

    raw_combos = []
    for axis in vary_axes:
        dim = axis["dim"]
        grid_idx = axis.get("grid_index", dim)
        if grid_idx >= len(param_grids):
            continue
        grid = param_grids[grid_idx]
        for value in grid_window_values(grid, bases[dim], window_steps):
            raw_combos.append(make_sensitivity_combo(bases, dim, value, axis.get("sync")))

    raw_combos = list(dict.fromkeys(raw_combos))
    all_combos = []
    for combo in raw_combos:
        try:
            all_combos.append(spec['validate_combo'](*combo))
        except ValueError:
            pass
    return all_combos


def run_indicator_sensitivity_analysis(indicator_name):
    spec = INDICATOR_SENSITIVITY_SPECS.get(indicator_name)
    if spec is None:
        print(f"Skipping {indicator_name}: no sensitivity spec defined for this indicator.")
        return
    base_info = resolve_indicator_base(indicator_name, spec)
    if base_info is None:
        print(f"Skipping {indicator_name}: no results available for parameter sensitivity analysis.")
        return

    p1_base, p2_base, p3_base, base_is_sharpe, base_oos_sharpe = base_info
    bases = [int(p1_base), int(p2_base), int(p3_base)]

    try:
        # Snap all 3 sensitivity slots onto their full grid sources so the exact
        # base combo is always among evaluated rows (avoids wasted runs then skip).
        full_grids = [resolve_param_grid(name) for name in spec["param_grid_sources"]]
        param_grids = full_grids[: len(spec["param_specs"])]
    except (KeyError, TypeError, ValueError) as exc:
        print(f"Skipping {indicator_name}: {exc}")
        return

    for dim_idx, grid_values in enumerate(full_grids):
        if bases[dim_idx] not in grid_values:
            snapped = grid_values[grid_index_for_base(grid_values, bases[dim_idx])]
            print(
                f"Note: {indicator_name} base parameter dim {dim_idx}={bases[dim_idx]} "
                f"is not on the search grid; snapping to nearest grid value {snapped}."
            )
            bases[dim_idx] = int(snapped)

    try:
        bases = list(spec["validate_combo"](*bases))
    except ValueError as exc:
        print(f"Skipping {indicator_name}: snapped base params are invalid ({bases}): {exc}")
        return

    p1_base, p2_base, p3_base = (int(bases[0]), int(bases[1]), int(bases[2]))
    bases = (p1_base, p2_base, p3_base)

    format_base_label = spec.get('format_base_label')
    if callable(format_base_label):
        base_label = format_base_label(p1_base, p2_base, p3_base)
    else:
        base_label = spec['base_label_fmt'].format(p1_base, p2_base, p3_base)

    print("\n" + "=" * 90)
    print(f"🔬 PARAMETER SENSITIVITY ANALYSIS - {indicator_name} - Base: {base_label}")

    all_combos = build_sensitivity_combos(p1_base, p2_base, p3_base, spec, param_grids)
    if not all_combos:
        print(f"Skipping {indicator_name}: no valid sensitivity combinations were generated.")
        return

    print(f"\n🔄 Testing {len(all_combos)} parameter variations...")

    rows = []
    failed_sensitivity_combos = []
    for combo in all_combos:
        try:
            rows.append(eval_indicator_combo_both(*combo, spec))
        except Exception as exc:
            failed_sensitivity_combos.append((combo, str(exc)))

    if failed_sensitivity_combos:
        first_failed_combo, first_error = failed_sensitivity_combos[0]
        print(
            f"⚠️ Skipped {len(failed_sensitivity_combos)} failed sensitivity variation(s). "
            f"First failure: {first_failed_combo} -> {first_error[:120]}"
        )

    if not rows:
        print(f"❌ No sensitivity results computed for {indicator_name}.")
        return

    sens_df = pd.DataFrame(rows)
    variation_sets = build_variation_sets(sens_df, bases, spec)
    base_params = {'ema1': p1_base, 'ema2': p2_base, 'ema3': p3_base}

    base_trade_row = sens_df[(sens_df['ema1'] == p1_base) & (sens_df['ema2'] == p2_base) & (sens_df['ema3'] == p3_base)]
    if base_trade_row.empty:
        print(f"Skipping {indicator_name}: {spec['base_missing_msg']}")
        return
    if len(base_trade_row) > 1:
        print(
            f"Note: {indicator_name} matched {len(base_trade_row)} base rows; "
            "using the first evaluated match for reference metrics."
        )

    # Reference line + deltas must use the same metrics as the bars (evaluated),
    # not the stored grid/CSV Sharpes which can differ after re-evaluation.
    stored_is_sharpe, stored_oos_sharpe = base_is_sharpe, base_oos_sharpe
    try:
        base_metrics = extract_base_metrics(base_trade_row.iloc[0])
    except (KeyError, TypeError, ValueError) as exc:
        print(f"Skipping {indicator_name}: could not read base metrics ({exc})")
        return

    evaluated_is_sharpe = base_metrics['is_sharpe']
    evaluated_oos_sharpe = base_metrics['oos_sharpe']
    note_if_reference_differs('IS', evaluated_is_sharpe, stored_is_sharpe)
    note_if_reference_differs('OOS', evaluated_oos_sharpe, stored_oos_sharpe)
    base_is_sharpe = prefer_finite(evaluated_is_sharpe, stored_is_sharpe)
    base_oos_sharpe = prefer_finite(evaluated_oos_sharpe, stored_oos_sharpe)

    if is_finite_number(base_is_sharpe) and is_finite_number(base_oos_sharpe):
        print(f"IS Sharpe: {base_is_sharpe:.3f} | OOS Sharpe: {base_oos_sharpe:.3f}")
    elif is_finite_number(base_is_sharpe):
        print(f"IS Sharpe: {base_is_sharpe:.3f}")
    else:
        print("IS Sharpe: n/a")

    has_valid_oos_sharpe = is_finite_number(base_oos_sharpe)
    base_is_trades = base_metrics['is_trades']
    base_oos_trades = base_metrics['oos_trades']
    base_is_volatility = base_metrics['is_volatility']
    base_oos_volatility = base_metrics['oos_volatility']
    base_is_maxdd = base_metrics['is_maxdd']
    base_oos_maxdd = base_metrics['oos_maxdd']

    for variations in variation_sets:
        variations['is_sharpe_delta'] = pct_delta(variations['is_sharpe'], base_is_sharpe)
        variations['oos_sharpe_delta'] = (
            pct_delta(variations['oos_sharpe'], base_oos_sharpe)
            if has_valid_oos_sharpe else np.nan
        )
        variations['is_trades_delta'] = pct_delta(variations['is_trades'], base_is_trades)
        variations['oos_trades_delta'] = pct_delta(variations['oos_trades'], base_oos_trades)
        variations['is_trades_count_delta'] = count_delta(variations['is_trades'], base_is_trades)
        variations['oos_trades_count_delta'] = count_delta(variations['oos_trades'], base_oos_trades)
        variations['is_volatility_delta'] = pct_delta(variations['is_volatility'], base_is_volatility)
        variations['oos_volatility_delta'] = pct_delta(variations['oos_volatility'], base_oos_volatility)
        variations['is_maxdd_delta'] = drawdown_pct_delta(variations['is_maxdd'], base_is_maxdd)
        variations['oos_maxdd_delta'] = drawdown_pct_delta(variations['oos_maxdd'], base_oos_maxdd)

    n_params = len(spec['param_specs'])
    n_cols = max(n_params, 1)
    fig_w = max(6.5 * n_cols, 8)

    def _new_sensitivity_figure(title):
        fig, axes = plt.subplots(2, n_cols, figsize=(fig_w, 10), squeeze=False)
        fig.suptitle(title, fontsize=16, fontweight='bold')
        if not has_valid_oos_sharpe:
            for col_idx in range(n_cols):
                axes[1, col_idx].set_visible(False)
        return fig, axes

    fig, axes = _new_sensitivity_figure(
        f'Parameter Sensitivity Analysis - {indicator_name} - Base: {base_label}'
    )

    for col_idx, (param_spec, variations, grid_values) in enumerate(zip(spec['param_specs'], variation_sets, param_grids)):
        if variations.empty:
            axes[0, col_idx].set_title(f"{param_spec['title']} - In-Sample (no data)", fontsize=12, fontweight='bold')
            continue
        colors_is = directional_delta_colors(variations['is_sharpe_delta'])
        axes[0, col_idx].bar(
            variations[param_spec['col']], variations['is_sharpe'],
            color=colors_is, edgecolor='black', linewidth=0.5
        )
        add_base_line(axes[0, col_idx], base_is_sharpe, f'Base Sharpe={base_is_sharpe:.3f}')
        axes[0, col_idx].axvline(
            base_params[param_spec['col']], color='blue', linestyle='--', linewidth=2.5, alpha=0.7,
            label=f'Base={base_params[param_spec["col"]]}'
        )
        axes[0, col_idx].set_xlabel(param_spec['xlabel'], fontsize=11, fontweight='bold')
        axes[0, col_idx].set_ylabel('Sharpe Ratio', fontsize=11, fontweight='bold')
        axes[0, col_idx].set_title(f"{param_spec['title']} - In-Sample", fontsize=12, fontweight='bold')
        axes[0, col_idx].grid(axis='y', alpha=0.3)
        axes[0, col_idx].legend(fontsize=10, loc='upper right')
        set_grid_aligned_xticks(axes[0, col_idx], variations[param_spec['col']], grid_values)

        if has_valid_oos_sharpe:
            colors_oos = directional_delta_colors(variations['oos_sharpe_delta'])
            axes[1, col_idx].bar(
                variations[param_spec['col']], variations['oos_sharpe'],
                color=colors_oos, edgecolor='black', linewidth=0.5
            )
            add_base_line(axes[1, col_idx], base_oos_sharpe, f'Base Sharpe={base_oos_sharpe:.3f}')
            axes[1, col_idx].axvline(
                base_params[param_spec['col']], color='blue', linestyle='--', linewidth=2.5, alpha=0.7,
                label=f'Base={base_params[param_spec["col"]]}'
            )
            axes[1, col_idx].set_xlabel(param_spec['xlabel'], fontsize=11, fontweight='bold')
            axes[1, col_idx].set_ylabel('Sharpe Ratio', fontsize=11, fontweight='bold')
            axes[1, col_idx].set_title(f"{param_spec['title']} - Out-of-Sample", fontsize=12, fontweight='bold')
            axes[1, col_idx].grid(axis='y', alpha=0.3)
            axes[1, col_idx].legend(fontsize=10, loc='upper right')
            set_grid_aligned_xticks(axes[1, col_idx], variations[param_spec['col']], grid_values)

    plt.tight_layout()
    plt.show()

    fig_trades, trade_axes = _new_sensitivity_figure(
        f'Total Trades Sensitivity Analysis - {indicator_name} - Base: {base_label}'
    )

    for col_idx, (param_spec, variations, grid_values) in enumerate(zip(spec['param_specs'], variation_sets, param_grids)):
        if variations.empty:
            trade_axes[0, col_idx].set_title(f"{param_spec['title']} - In-Sample Trades (no data)", fontsize=12, fontweight='bold')
            continue
        plot_trade_values(
            trade_axes[0, col_idx], variations, param_spec, base_params,
            f"{param_spec['title']} - In-Sample Trades", 'is_trades', 'is_trades_delta', base_is_trades, grid_values
        )
        if has_valid_oos_sharpe:
            plot_trade_values(
                trade_axes[1, col_idx], variations, param_spec, base_params,
                f"{param_spec['title']} - Out-of-Sample Trades", 'oos_trades', 'oos_trades_delta', base_oos_trades, grid_values
            )

    plt.tight_layout()
    plt.show()

    fig_vol, vol_axes = _new_sensitivity_figure(
        f'Volatility Sensitivity Analysis - {indicator_name} - Base: {base_label}'
    )

    for col_idx, (param_spec, variations, grid_values) in enumerate(zip(spec['param_specs'], variation_sets, param_grids)):
        if variations.empty:
            vol_axes[0, col_idx].set_title(f"{param_spec['title']} - In-Sample Volatility (no data)", fontsize=12, fontweight='bold')
            continue
        plot_volatility_values(
            vol_axes[0, col_idx], variations, param_spec, base_params,
            f"{param_spec['title']} - In-Sample Volatility", 'is_volatility', 'is_volatility_delta', base_is_volatility, grid_values
        )
        if has_valid_oos_sharpe:
            plot_volatility_values(
                vol_axes[1, col_idx], variations, param_spec, base_params,
                f"{param_spec['title']} - Out-of-Sample Volatility", 'oos_volatility', 'oos_volatility_delta', base_oos_volatility, grid_values
            )

    plt.tight_layout()
    plt.show()

    fig_maxdd, maxdd_axes = _new_sensitivity_figure(
        f'Max Drawdown Sensitivity Analysis - {indicator_name} - Base: {base_label}'
    )

    for col_idx, (param_spec, variations, grid_values) in enumerate(zip(spec['param_specs'], variation_sets, param_grids)):
        if variations.empty:
            maxdd_axes[0, col_idx].set_title(f"{param_spec['title']} - In-Sample Max Drawdown (no data)", fontsize=12, fontweight='bold')
            continue
        plot_maxdd_values(
            maxdd_axes[0, col_idx], variations, param_spec, base_params,
            f"{param_spec['title']} - In-Sample Max Drawdown", 'is_maxdd', 'is_maxdd_delta', base_is_maxdd, grid_values
        )
        if has_valid_oos_sharpe:
            plot_maxdd_values(
                maxdd_axes[1, col_idx], variations, param_spec, base_params,
                f"{param_spec['title']} - Out-of-Sample Max Drawdown", 'oos_maxdd', 'oos_maxdd_delta', base_oos_maxdd, grid_values
            )

    plt.tight_layout()
    plt.show()

    print(f"\n📋 {spec['summary_title']}")
    print("=" * 80)

    ema_summary_data = []
    trades_summary_data = []
    volatility_summary_data = []
    maxdd_summary_data = []

    for param_spec, variations, grid_values in zip(spec['param_specs'], variation_sets, param_grids):
        if variations.empty:
            ema_summary_data.append({
                'Parameter': param_spec['name'],
                'IS Range': 'N/A',
                'IS Max Δ%': 'N/A',
                'OOS Range': 'N/A',
                'OOS Max Δ%': 'N/A',
                'Sensitivity': 'N/A',
            })
            trades_summary_data.append({
                'Parameter': param_spec['name'],
                'IS Trades': 'N/A',
                'IS Trades Max Δ%': 'N/A',
                'IS Trades Max Δ': 'N/A',
                'OOS Trades': 'N/A',
                'OOS Trades Max Δ%': 'N/A',
                'OOS Trades Max Δ': 'N/A',
                'Neighbor Trade Robustness': 'N/A',
            })
            volatility_summary_data.append({
                'Parameter': param_spec['name'],
                'IS Volatility': 'N/A',
                'IS Volatility Max Δ%': 'N/A',
                'OOS Volatility': 'N/A',
                'OOS Volatility Max Δ%': 'N/A',
                'Neighbor Volatility Robustness': 'N/A',
            })
            maxdd_summary_data.append({
                'Parameter': param_spec['name'],
                'IS Max Drawdown': 'N/A',
                'IS Max Drawdown Max Δ%': 'N/A',
                'OOS Max Drawdown': 'N/A',
                'OOS Max Drawdown Max Δ%': 'N/A',
                'Neighbor Max Drawdown Robustness': 'N/A',
            })
            continue

        param_col = param_spec['col']
        base_param = base_params[param_col]
        neighbor_rows = neighbor_variations(variations, param_col, base_param, grid_values)

        is_trades_max_delta = max_abs_finite(variations['is_trades_delta'])
        oos_trades_max_delta = max_abs_finite(variations['oos_trades_delta'])
        is_trades_max_count_delta = max_abs_finite(variations['is_trades_count_delta'])
        oos_trades_max_count_delta = max_abs_finite(variations['oos_trades_count_delta'])
        is_trades_neighbor_delta = max_abs_finite(neighbor_rows['is_trades_delta'])
        oos_trades_neighbor_delta = max_abs_finite(neighbor_rows['oos_trades_delta'])
        trade_neighbor_delta_values = [x for x in [is_trades_neighbor_delta, oos_trades_neighbor_delta] if not pd.isna(x)]
        max_trade_neighbor_delta = max(trade_neighbor_delta_values) if trade_neighbor_delta_values else np.nan
        is_volatility_max_delta = max_abs_finite(variations['is_volatility_delta'])
        oos_volatility_max_delta = max_abs_finite(variations['oos_volatility_delta'])
        is_volatility_neighbor_delta = max_abs_finite(neighbor_rows['is_volatility_delta'])
        oos_volatility_neighbor_delta = max_abs_finite(neighbor_rows['oos_volatility_delta'])
        volatility_neighbor_delta_values = [x for x in [is_volatility_neighbor_delta, oos_volatility_neighbor_delta] if not pd.isna(x)]
        max_volatility_neighbor_delta = max(volatility_neighbor_delta_values) if volatility_neighbor_delta_values else np.nan
        is_maxdd_max_delta = max_abs_finite(variations['is_maxdd_delta'])
        oos_maxdd_max_delta = max_abs_finite(variations['oos_maxdd_delta'])
        is_maxdd_neighbor_delta = max_abs_finite(neighbor_rows['is_maxdd_delta'])
        oos_maxdd_neighbor_delta = max_abs_finite(neighbor_rows['oos_maxdd_delta'])
        maxdd_neighbor_delta_values = [x for x in [is_maxdd_neighbor_delta, oos_maxdd_neighbor_delta] if not pd.isna(x)]
        max_maxdd_neighbor_delta = max(maxdd_neighbor_delta_values) if maxdd_neighbor_delta_values else np.nan

        ema_summary_data.append({
            'Parameter': param_spec['name'],
            'IS Range': f"{variations['is_sharpe'].min():.3f} - {variations['is_sharpe'].max():.3f}",
            'IS Max Δ%': f"{variations['is_sharpe_delta'].min():.1f}%",
            'OOS Range': f"{variations['oos_sharpe'].min():.3f} - {variations['oos_sharpe'].max():.3f}" if has_valid_oos_sharpe else 'N/A',
            'OOS Max Δ%': f"{variations['oos_sharpe_delta'].min():.1f}%" if has_valid_oos_sharpe and 'oos_sharpe_delta' in variations else 'N/A',
            'Sensitivity': '⚠️ HIGH' if abs(variations['is_sharpe_delta'].min()) > 40 else '✅ LOW'
        })
        trades_summary_data.append({
            'Parameter': param_spec['name'],
            'IS Trades': f"{variations['is_trades'].min():.0f} - {variations['is_trades'].max():.0f}",
            'IS Trades Max Δ%': f"{is_trades_max_delta:.1f}%" if not pd.isna(is_trades_max_delta) else 'N/A',
            'IS Trades Max Δ': f"{is_trades_max_count_delta:.0f}" if not pd.isna(is_trades_max_count_delta) else 'N/A',
            'OOS Trades': f"{variations['oos_trades'].min():.0f} - {variations['oos_trades'].max():.0f}",
            'OOS Trades Max Δ%': f"{oos_trades_max_delta:.1f}%" if not pd.isna(oos_trades_max_delta) else 'N/A',
            'OOS Trades Max Δ': f"{oos_trades_max_count_delta:.0f}" if not pd.isna(oos_trades_max_count_delta) else 'N/A',
            'Neighbor Trade Robustness': 'N/A' if pd.isna(max_trade_neighbor_delta) else ('⚠️ HIGH' if max_trade_neighbor_delta > 40 else '✅ LOW')
        })
        volatility_summary_data.append({
            'Parameter': param_spec['name'],
            'IS Volatility': f"{variations['is_volatility'].min():.2%} - {variations['is_volatility'].max():.2%}",
            'IS Volatility Max Δ%': f"{is_volatility_max_delta:.1f}%" if not pd.isna(is_volatility_max_delta) else 'N/A',
            'OOS Volatility': f"{variations['oos_volatility'].min():.2%} - {variations['oos_volatility'].max():.2%}",
            'OOS Volatility Max Δ%': f"{oos_volatility_max_delta:.1f}%" if not pd.isna(oos_volatility_max_delta) else 'N/A',
            'Neighbor Volatility Robustness': 'N/A' if pd.isna(max_volatility_neighbor_delta) else ('⚠️ HIGH' if max_volatility_neighbor_delta > 40 else '✅ LOW')
        })
        maxdd_summary_data.append({
            'Parameter': param_spec['name'],
            'IS Max Drawdown': f"{variations['is_maxdd'].min():.2%} - {variations['is_maxdd'].max():.2%}",
            'IS Max Drawdown Max Δ%': f"{is_maxdd_max_delta:.1f}%" if not pd.isna(is_maxdd_max_delta) else 'N/A',
            'OOS Max Drawdown': f"{variations['oos_maxdd'].min():.2%} - {variations['oos_maxdd'].max():.2%}",
            'OOS Max Drawdown Max Δ%': f"{oos_maxdd_max_delta:.1f}%" if not pd.isna(oos_maxdd_max_delta) else 'N/A',
            'Neighbor Max Drawdown Robustness': 'N/A' if pd.isna(max_maxdd_neighbor_delta) else ('⚠️ HIGH' if max_maxdd_neighbor_delta > 40 else '✅ LOW')
        })

    ema_summary_df = pd.DataFrame(ema_summary_data)
    print(ema_summary_df.to_string(index=False))

    print("\n\n")
    print("📋 TOTAL TRADES SENSITIVITY SUMMARY")
    print("=" * 80)
    trades_summary_df = pd.DataFrame(trades_summary_data)
    print(trades_summary_df.to_string(index=False))

    print("\n\n")
    print("📋 VOLATILITY SENSITIVITY SUMMARY")
    print("=" * 80)
    volatility_summary_df = pd.DataFrame(volatility_summary_data)
    print(volatility_summary_df.to_string(index=False))

    print("\n\n\n")
    print("📋 MAX DRAWDOWN SENSITIVITY SUMMARY")
    print("=" * 80)
    maxdd_summary_df = pd.DataFrame(maxdd_summary_data)
    print(maxdd_summary_df.to_string(index=False))

    print(f"\n✅ {indicator_name} sensitivity analysis complete! Green = Robust, Red = Fragile")


if 'results_df' not in globals() or results_df.empty:
    print("No results available for sensitivity analysis.")
elif not sensitivity_prerequisites_ok():
    pass
else:
    sensitivity_indicators = resolve_sensitivity_indicators()
    if not sensitivity_indicators:
        print("No indicators available for sensitivity analysis.")
    else:
        for indicator_name in sensitivity_indicators:
            try:
                run_indicator_sensitivity_analysis(indicator_name)
            except Exception as exc:
                print(f"❌ Sensitivity analysis failed for {indicator_name}: {exc}")



🔬 PARAMETER SENSITIVITY ANALYSIS - Triple EMA - Base: EMA(4,117,125)

🔄 Testing 49 parameter variations...


IS Sharpe: 0.928 | OOS Sharpe: 0.653


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:903: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:924: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:945: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


...


📋 VORTEX SENSITIVITY SUMMARY
        Parameter      IS Range IS Max Δ%     OOS Range OOS Max Δ% Sensitivity
VORTEX Timeperiod 0.328 - 0.654    -44.0% 0.136 - 0.909     -77.1%     ⚠️ HIGH



📋 TOTAL TRADES SENSITIVITY SUMMARY
        Parameter IS Trades IS Trades Max Δ% IS Trades Max Δ OOS Trades OOS Trades Max Δ% OOS Trades Max Δ Neighbor Trade Robustness
VORTEX Timeperiod    6 - 23            91.7%              11     7 - 15             36.4%                4                     ✅ LOW



📋 VOLATILITY SENSITIVITY SUMMARY
        Parameter   IS Volatility IS Volatility Max Δ%  OOS Volatility OOS Volatility Max Δ% Neighbor Volatility Robustness
VORTEX Timeperiod 45.39% - 48.97%                 3.9% 35.44% - 43.30%                 17.6%                          ✅ LOW




📋 MAX DRAWDOWN SENSITIVITY SUMMARY
        Parameter   IS Max Drawdown IS Max Drawdown Max Δ%  OOS Max Drawdown OOS Max Drawdown Max Δ% Neighbor Max Drawdown Robustness
VORTEX Timeperiod -66.83% - -62.09%                

IS Sharpe: 0.786 | OOS Sharpe: 0.665


C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:903: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:924: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



📋 ALMA SENSITIVITY SUMMARY
       Parameter      IS Range IS Max Δ%     OOS Range OOS Max Δ% Sensitivity
ALMA Fast Length 0.192 - 0.786    -75.6% 0.607 - 0.771      -8.8%     ⚠️ HIGH
ALMA Slow Length 0.162 - 0.786    -79.4% 0.381 - 0.665     -42.8%     ⚠️ HIGH
     ALMA Offset 0.618 - 0.786    -21.4% 0.601 - 0.687      -9.6%       ✅ LOW



📋 TOTAL TRADES SENSITIVITY SUMMARY
       Parameter IS Trades IS Trades Max Δ% IS Trades Max Δ OOS Trades OOS Trades Max Δ% OOS Trades Max Δ Neighbor Trade Robustness
ALMA Fast Length    8 - 13            38.5%               5     6 - 10             40.0%                4                     ✅ LOW
ALMA Slow Length   12 - 23            76.9%              10    10 - 14             40.0%                4                     ✅ LOW
     ALMA Offset   13 - 17            30.8%               4    10 - 11             10.0%                1                     ✅ LOW



📋 VOLATILITY SENSITIVITY SUMMARY
       Parameter   IS Volatility IS Volatility Max Δ%  OOS

C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:945: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\j_jon\AppData\Local\Temp\ipykernel_11484\3376849503.py:966: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
